# TrustXAI-Derm: When Should Clinicians NOT Trust an AI Explanation?
## A Trust-Aware Framework for Dermoscopic Skin Cancer Diagnosis
### FIXED EDITION — checkpoint/resume, time-budget guards, novelty additions

**What's new in this edition (accuracy fix + GAP-9/GAP-10):**
- **Accuracy fix (CELL 2 + CELL 6 + CELL 7):** the previous run stopped at `epochs=60` while
  val macro-F1 was still climbing monotonically (0.08→0.34) and `patience=10` never triggered —
  it hit an arbitrary cap, not convergence. `CFG['epochs']` is raised to 150 (patience to 15), and
  the `train_model()` resume gate — which previously skipped training unconditionally the moment
  `best.pth`/`history.csv` existed, silently ignoring any higher `epochs` value — now persists
  *why* a run stopped and correctly continues training when more epochs are requested. A combined
  Focal + 0.1×label-smoothed-CE loss (`CombinedFocalSmoothedCE`) restores the auxiliary term GAP1's
  own training spec calls for but the code never implemented.
- **CELL 17c — GAP-9 (Synthetic Dark-Skin Diffusion Augmentation):** ControlNet-conditioned
  Stable Diffusion generates structure-preserving dark-skin variants of existing lesions, gated by
  a Lesion Structure Preservation Index + ITA-shift check. Opt-in via
  `CFG['use_synthetic_dark_skin']`; see the cell for the full before/after workflow.
- **CELL 17d — GAP-10 (Explanation Contrastive Analysis):** DINOv2-retrieved visually-similar
  lesion pairs are checked for Grad-CAM++ consistency across 4 contrastive categories (consistent
  success/error, contradictory prediction, representation diversity).

Both new cells require internet access (to download Stable Diffusion/ControlNet/DINOv2 weights) and
print a clear skip message — without crashing the rest of the notebook — if that's unavailable.

**Two-phase DenseNet121 training (`CFG['training_phase']`):** the primary DenseNet121 run is
now split across two Kaggle commits so a single session never has to carry the full 150-epoch
budget:
1. **Commit 1:** leave `CFG['training_phase'] = 1` (default). CELL 7 trains to epoch 60 and
   stops there deliberately, then run CELL 20 and create a Kaggle Dataset from its
   `session_checkpoint_export/` output.
2. **Commit 2:** attach that dataset, set `CFG['ckpt_input_dir']` to its root (see CELL 20's
   printed instructions — this must be the dataset root, not a nested `checkpoints/` folder),
   set `CFG['training_phase'] = 2`, and re-run top to bottom. `train_model()`'s existing
   resume/continuation logic picks up from epoch 61 and trains on to the full
   `CFG['epochs']` (150) — it does **not** retrain from scratch.

This also fixes a real bug in the cross-session checkpoint hydration (CELL 2b): it previously
copied everything into the nested `checkpoints/` folder only, so the actual model weights
(`DenseNet121_best.pth`) and training history never made it back into `/kaggle/working/` in a
fresh session — a phase-2 run would have silently retrained from scratch instead of resuming.

---

## WARNING: READ FIRST — Kaggle Session Strategy (fixes "session end / stop / spec / time" issues)

**1. Use "Save & Run All (Commit)", not the interactive editor.**
Interactive Kaggle sessions die when your browser tab sleeps or your laptop locks — that's the #1 cause of
silent "session end" loss. Commit runs execute in the background on Kaggle's servers and have their own
(longer) GPU time budget, independent of your browser.

**2. Every expensive cell in this notebook is now resumable.**
Cells 7, 8, 10, 10b, 12, 13, 14, 15, 16, 17 all check a checkpoint file first. If the kernel dies, times out,
or you manually stop it, re-running the notebook (even a fresh commit) picks up from the last saved batch
instead of starting over.

**3. Checkpoints must survive between Kaggle sessions.**
`/kaggle/working` is wiped between *separate* commits unless you do one of:
- Output the previous commit's `/kaggle/working` as a Kaggle Dataset, then attach that dataset as input to
  the next run (`CFG['ckpt_input_dir']` below points at it). **This is the recommended pattern for a
  multi-session run.**
- Or just keep re-running the *same* interactive session / commit, where `/kaggle/working` persists for the
  life of that session.

**4. A global time budget stops heavy loops before Kaggle kills the kernel.**
`CFG['time_budget_seconds']` is checked inside every resumable loop. When you're within `margin` seconds of
the budget, the loop saves what it has and exits cleanly — instead of getting killed mid-write and losing the
whole batch. Set this a little under your actual GPU quota (e.g. 8h if you have a 9h T4 budget).

**5. GPU spec issues (OOM, DataParallel mismatches).**
`CFG['batch_size']` auto-halves on a caught CUDA OOM in the training loop (Cell 7/8) instead of crashing the
whole run. SHAP (Cell 12) is switched from `DeepExplainer` to `GradientExplainer`, which is faster and far
less memory-hungry for CNNs — this was the main blocker to running XDI on the *full* test set instead of a
200-image sample.

---

## CELL 1 — Install Dependencies (FIXED: shap, albumentations, scikit-image were used but never installed)


In [ ]:
# ============================================================
# CELL 1 — Install all required packages
# Runtime: ~60-90 seconds
# FIX: original notebook imported shap, albumentations, and skimage
# in later cells without ever pip-installing them. On a Kaggle base
# image these are sometimes preinstalled and sometimes not -- pinning
# them here removes a silent "works today, ImportError tomorrow" risk.
# ============================================================
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# FIX (third revision -- see history below): earlier attempts got this
# wrong in two different ways.
#   v1: installed each package one-at-a-time, letting pip resolve deps
#       freely. On a local test env this silently upgraded numpy past
#       the ceiling shap's numba dependency tolerates.
#   v2: "fixed" v1 by re-pinning numpy back down as a separate step
#       afterward. This broke Kaggle for real: Kaggle's own pandas is
#       pre-compiled against Kaggle's shipped numpy, so downgrading numpy
#       via pip after pandas is already on disk breaks pandas' compiled
#       C-extension ABI ("numpy.dtype size changed... Expected 96 from
#       C header, got 88 from PyObject" the moment `import pandas` ran).
#   v2b: tried --no-deps for every package to avoid touching numpy at
#       all. This went too far the other way: albumentations==1.4.15
#       needs a SPECIFIC compatible version of its own albucore
#       dependency, and --no-deps prevented that from being resolved,
#       so `import albumentations` failed with
#       "ImportError: cannot import name 'preserve_channel_dim' from
#       albucore.utils" against whatever albucore Kaggle happened to
#       already have.
#
# Correct fix: let pip's dependency resolver see every package's real
# constraints TOGETHER in a single call (not one separate subprocess per
# package, which is what caused v1's silent numpy drift in the first
# place), while EXPLICITLY pinning numpy to whatever Kaggle already has
# installed. This lets pip find a consistent set of versions for
# timm/grad-cam/torchmetrics/shap/albumentations/scikit-image (including
# a correct, compatible albucore for albumentations) that all agree with
# the numpy Kaggle's own pandas/scipy were built against, instead of
# either letting numpy drift (v1) or blocking legitimate sub-dependency
# resolution outright (v2b).
import numpy as _np_probe
_pinned_numpy = f'numpy=={_np_probe.__version__}'
print(f'Pinning numpy to the version Kaggle already has installed: {_pinned_numpy}')

packages = [
    'timm==1.0.3', # Model zoo: ConvNeXt-Tiny
    'grad-cam==1.5.5', # GradCAMPlusPlus hook library
    'torchmetrics==1.4.2', # Accuracy/F1/AUC/ECE
    'shap==0.46.0', # FIX: was imported (Cell 2/12/14) but never installed
    'albumentations>=1.4.24', # FIX (v4): exact pin 1.4.24 hard-requires albucore==0.0.23, which is missing preserve_channel_dim -> ImportError on import albumentations. Unpinning the upper bound lets pip resolve a self-consistent, currently-shipping albumentations/albucore pair instead of that broken snapshot.
    'scikit-image==0.24.0', # FIX: was imported (Cell 17/19) but never installed
    _pinned_numpy, # keep numpy fixed at Kaggle's own version through the joint resolve
]
try:
    pip_install(*packages)
except subprocess.CalledProcessError as e:
    print(f'WARNING: joint install failed: {e}. Falling back to installing each package '
          f'separately (numpy pin included in every call to keep it fixed).')
    for pkg in packages[:-1]:
        try:
            pip_install(pkg, _pinned_numpy)
        except subprocess.CalledProcessError as e2:
            print(f'WARNING: Failed to install {pkg}: {e2}. Continuing -- it may already be present.')

print("All packages installed (or already present) with numpy held at Kaggle's original version.")



## CELL 2 — Imports + Config + Checkpoint/Resume Infrastructure (MODIFIED: added resume system)


In [ ]:
# ============================================================
# CELL 2 — Imports + Global Config + Checkpoint/Resume Infrastructure
# ============================================================
import os, sys, gc, copy, json, warnings, time, random, signal, pickle, atexit
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
cv2.setNumThreads(0)
from PIL import Image
from tqdm import tqdm  # NEW: plain text-mode tqdm, not tqdm.auto/ipywidgets --
                       # ipywidgets progress bars register a live comm object per
                       # bar; across 150+ epochs x multiple loaders in one long
                       # Kaggle session, that's hundreds of widget objects held in
                       # the kernel's widget-state registry. Plain tqdm just prints
                       # text and holds nothing after each bar completes.
from scipy import stats
from scipy.stats import mannwhitneyu, kruskal, spearmanr, pearsonr, bootstrap, wilcoxon
from skimage import measure
from skimage.measure import regionprops

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import timm
from torchmetrics import (
    Accuracy, F1Score, AUROC, MatthewsCorrCoef, CalibrationError
)
from sklearn.metrics import (classification_report, confusion_matrix,
                              cohen_kappa_score, brier_score_loss,
                              precision_recall_curve, auc,
                              average_precision_score, roc_auc_score,
                              balanced_accuracy_score)
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold, cross_val_predict
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import shap

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# NEW (reproducibility, additive): deterministic DataLoader worker seeding.
# Currently a no-op because every loader uses num_workers=0. If you raise
# num_workers for throughput, pass worker_init_fn=seed_worker and generator=g_loader
# to each DataLoader so shuffling/augmentation stay reproducible across workers.
def seed_worker(worker_id):
    worker_seed = (SEED + worker_id) % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g_loader = torch.Generator()
g_loader.manual_seed(SEED)

# ── Device ───────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPUs available: {N_GPUS}')
    for gi in range(N_GPUS):
        print(f'[{gi}] {torch.cuda.get_device_name(gi)} | '
              f'{torch.cuda.get_device_properties(gi).total_memory / 1e9:.1f} GB')
    if N_GPUS > 1:
        print(f'-> train_model() below will wrap the model in nn.DataParallel '
              f'across all {N_GPUS} GPUs during training only.')

# NEW (reproducibility, additive): log the exact runtime stack so every reported
# number is traceable to a software/hardware environment. Printed once at startup;
# the output is captured in the notebook and can be quoted in the paper's repro section.
def _pkg_ver(_name):
    try:
        import importlib.metadata as _md
        return _md.version(_name)
    except Exception:
        return 'n/a'

ENV_INFO = {
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'cuda': torch.version.cuda or 'cpu',
    'cudnn': torch.backends.cudnn.version() if torch.cuda.is_available() else 'n/a',
    'gpu': (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'),
    'n_gpus': N_GPUS,
    'numpy': np.__version__,
    'timm': _pkg_ver('timm'),
    'shap': _pkg_ver('shap'),
    'albumentations': _pkg_ver('albumentations'),
    'grad-cam': _pkg_ver('grad-cam'),
    'scikit-learn': _pkg_ver('scikit-learn'),
    'seed': SEED,
}
print('\nEnvironment (for reproducibility):')
for _k, _v in ENV_INFO.items():
    print(f'  {_k:14s}: {_v}')

# ── Global hyperparameters ───────────────────────────────────
CFG = dict(
    img_size = 224,
    batch_size = 32, # FIX (2026-07-05): 32 not 64 -- lower peak GPU memory when
                     # DenseNet121 + ConvNeXt-Tiny run in one session (OOM auto-halve
                     # still backs this up). Trades a little speed for headroom.
    min_batch_size = 8, # NEW: OOM auto-halve floor
    epochs = 150,  # FIX (accuracy): the epoch-60 run (DenseNet121_history.csv)
                   # was still improving when it hit this cap -- val_f1 climbed
                   # 0.08->0.34 MONOTONICALLY, train_loss was still falling, and
                   # patience never triggered. It had also used only ~2.5h of the
                   # 8h time budget (60 epochs cost ~93s/epoch at steady state).
                   # Raised the cap so training can keep improving. This ONLY works
                   # together with the resume/continuation fix in CELL 7 below --
                   # without it, bumping this number alone silently does nothing.
    patience = 15, # FIX: val_f1 is noisy epoch-to-epoch near the end of the run
                   # (epoch 54: 0.3315, epoch 58: 0.3308, epoch 59: 0.3207, epoch 60:
                   # 0.3395 -- a new best right after two down-epochs). patience=10
                   # risked stopping on a noise dip; 15 gives more room.
    lr = 3e-4,
    weight_decay = 1e-4,
    focal_gamma = 2.0,
    dropout = 0.3,
    mc_dropout_T = 20,
    n_classes = 7,
    tixai_epsilon = 1e-7,
    xdi_threshold = 0.5,
    deferral_alpha = 0.1,
    allow_synthetic_masks_in_results = False,
    use_synthetic_dark_skin = False,  # NEW (GAP-9): set True + re-run CELL 5 and CELL 7
                                       # (force_retrain=True) to train on HAM10000 +
                                       # validated synthetic dark-skin images once CELL 17c
                                       # has produced train_df_augmented.csv.
    training_phase = 1,  # NEW: two-phase DenseNet121 training across two Kaggle commits.
                         # Phase 1 (this default): CELL 7 trains to epoch PHASE1_EPOCHS (60)
                         # and stops there -- a deliberate session boundary, not a crash/budget
                         # cutoff. Then run CELL 20, create a Kaggle Dataset from its output,
                         # and in your SECOND commit: attach that dataset, set
                         # CFG['ckpt_input_dir'] accordingly (see CELL 20's printed
                         # instructions), set training_phase=2 here, and re-run top to bottom.
                         # CELL 7's train_model() resume/continuation logic (already built)
                         # will pick up from epoch 61 and continue to the full CFG['epochs']
                         # (150) automatically -- phase 2 does NOT retrain from scratch.
    out_dir = Path('/kaggle/working'),

    # ── NEW: resume / checkpoint / time-budget controls ──────
    resume = True, # set False to force full recompute everywhere
    # FIX (scheduler bug, audit 2026-07-05): train_epoch() previously called
    # scheduler.step() once per epoch instead of once per batch, so every
    # checkpoint saved before this fix (including the existing epoch-60
    # DenseNet121_best.pth) was trained under a LR schedule that never left
    # its warmup phase. Resuming from that checkpoint would just keep
    # annealing a broken schedule for longer. force_retrain=True forces one
    # clean from-scratch run under the corrected per-batch scheduler.
    # IMPORTANT: set this back to False after this one retrain completes,
    # or every future run (including a real Phase 2) will retrain from
    # scratch instead of resuming.
    force_retrain = True, # set True to retrain even if best_*.pth exists
    force_recompute_all = False, # NEW (clean-slate switch for a re-run): when True,
                                 # load_ckpt()/load_progress_csv() IGNORE every cached
                                 # .pkl/.csv checkpoint and recompute all GAP cells from
                                 # scratch -- use this for an INTERACTIVE re-run where
                                 # stale pre-fix checkpoints may still sit in /kaggle/working.
                                 # LEAVE False for a fresh 'Save & Run All' Commit (where
                                 # /kaggle/working starts empty anyway) so that within-session
                                 # resume still works if the kernel times out mid-run. Do NOT
                                 # attach an old session_checkpoint_export dataset when doing
                                 # a clean re-run -- that is the other way stale results sneak in.
    time_budget_seconds = 8 * 3600, # set under your actual Kaggle GPU quota
    time_margin_seconds = 600, # stop starting new heavy work this close to budget
    save_every_n = 50, # rows between incremental checkpoint saves
    # If you're resuming from a PREVIOUS commit's checkpoints (attached as a
    # Kaggle Dataset input), point this at that dataset's checkpoints folder
    # and CELL 2b below will copy them into /kaggle/working/checkpoints.
    ckpt_input_dir = None, # e.g. Path('/kaggle/input/trustxai-checkpoints-v1/checkpoints')
)
CFG['out_dir'].mkdir(parents=True, exist_ok=True)
(CFG['out_dir'] / 'checkpoints').mkdir(parents=True, exist_ok=True)

# ── HAM10000 classes ─────────────────────────────────────────
CLASSES = ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
CLASS_COLORS = {
    'NV': '#4CAF50', 'MEL': '#F44336', 'BKL': '#FF9800',
    'BCC': '#9C27B0', 'AKIEC':'#2196F3', 'DF': '#795548',
    'VASC': '#00BCD4'
}
CLASS_COUNTS = {'NV':5364,'MEL':890,'BKL':879,'BCC':411,'AKIEC':261,'VASC':113,'DF':92}

print(f'\nConfig loaded. Classes: {CLASSES}')
print(f'Output dir: {CFG["out_dir"]}')



## CELL 2b — Checkpoint/Resume Utilities (NEW — used by every heavy cell below)


In [ ]:
# ============================================================
# CELL 2b — Generic checkpoint + resumable-loop infrastructure
#
# Fixes three classes of problems that plague long Kaggle runs:
# 1. "Session end" -- kernel killed mid-loop loses all progress.
# 2. "Stop system" -- you manually stop and lose the run.
# 3. "Time issue" -- 60-epoch training / full-test-set SHAP runs
# longer than a single Kaggle session allows.
#
# Pattern: every expensive per-image loop writes its partial results
# to a CSV every `save_every_n` rows. On re-run, already-processed
# image_ids are skipped. A global wall-clock budget makes loops exit
# cleanly (and save) before Kaggle kills the kernel outright.
# ============================================================
CKPT_DIR = CFG['out_dir'] / 'checkpoints'
SESSION_START = time.time()

# ── Optionally hydrate checkpoints from a PREVIOUS commit's output,
# attached as a Kaggle Dataset input. This is what makes a true
# multi-session run possible (since /kaggle/working itself does
# not persist across separate commits). ─────────────────────────
#
# FIX (2-phase training was silently NOT resuming): this used to copy
# EVERYTHING from ckpt_input_dir into CKPT_DIR (the nested checkpoints/
# subfolder). But CELL 20 exports two different kinds of files: the
# nested checkpoints/ subfolder (resumable-loop progress CSVs/pickles,
# which DO belong in CKPT_DIR) and TOP-LEVEL files -- DenseNet121_best.pth,
# DenseNet121_history.csv, DenseNet121_status.json, train/val/test_split.csv,
# etc. -- which train_model() and CELL 4/5 look for directly under
# CFG['out_dir'], NOT under checkpoints/. Dumping everything into CKPT_DIR
# meant DenseNet121_best.pth landed at .../checkpoints/DenseNet121_best.pth
# instead of .../DenseNet121_best.pth, so train_model() in a fresh
# session would never find it, silently retrain from scratch, and burn
# an entire phase's GPU time for nothing. Point CFG['ckpt_input_dir'] at
# the DATASET ROOT (see CELL 20's instructions) and this now restores
# both halves to their correct locations.
if CFG.get('ckpt_input_dir') is not None and Path(CFG['ckpt_input_dir']).exists():
    import shutil
    _ckpt_src_root = Path(CFG['ckpt_input_dir'])
    n_copied = 0
    for f in _ckpt_src_root.iterdir():
        if f.is_file():
            shutil.copy(f, CFG['out_dir'] / f.name)
            n_copied += 1
    _nested_ckpt_src = _ckpt_src_root / 'checkpoints'
    if _nested_ckpt_src.exists():
        for f in _nested_ckpt_src.iterdir():
            if f.is_file():
                shutil.copy(f, CKPT_DIR / f.name)
                n_copied += 1
    print(f'Hydrated {n_copied} checkpoint/output files from previous-session dataset: '
          f'{CFG["ckpt_input_dir"]} (top-level model weights/history/splits + nested '
          f'checkpoints/ progress files).')


def time_left():
    return CFG['time_budget_seconds'] - (time.time() - SESSION_START)


def time_critical(margin=None):
    """True once we're within `margin` seconds of the time budget."""
    margin = CFG['time_margin_seconds'] if margin is None else margin
    return time_left() < margin


def save_ckpt(name, obj):
    with open(CKPT_DIR / f'{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)
    print(f'checkpoint saved: {name}.pkl')


def load_ckpt(name):
    p = CKPT_DIR / f'{name}.pkl'
    if p.exists() and CFG['resume'] and not CFG.get('force_recompute_all', False):
        with open(p, 'rb') as f:
            obj = pickle.load(f)
        print(f'checkpoint loaded: {name}.pkl (skipping recompute)')
        return obj
    return None


def load_progress_csv(path, id_col='image_id'):
    path = Path(path)
    if path.exists() and CFG['resume'] and not CFG.get('force_recompute_all', False):
        return pd.read_csv(path)
    return pd.DataFrame()


def run_resumable_loop(items_df, ckpt_csv_path, process_fn, id_col='image_id',
                        desc='', save_every=None):
    """
    Iterate `items_df`, calling process_fn(row) -> dict|None for each row,
    skipping rows whose id_col already appears in ckpt_csv_path. Saves
    progress every `save_every` rows, on KeyboardInterrupt, and when the
    global time budget is nearly exhausted -- so a killed/stopped session
    never loses more than `save_every` rows of work.

    Returns the full accumulated results DataFrame (done + newly computed).
    """
    save_every = save_every or CFG['save_every_n']
    ckpt_csv_path = Path(ckpt_csv_path)

    done_df = load_progress_csv(ckpt_csv_path, id_col)
    done_ids = set(done_df[id_col].astype(str)) if (len(done_df) and id_col in done_df.columns) else set()
    results = done_df.to_dict('records')

    if id_col in items_df.columns:
        pending = items_df[~items_df[id_col].astype(str).isin(done_ids)]
    else:
        pending = items_df

    print(f'{desc}: {len(done_ids)} already done, {len(pending)} remaining '
          f'(time left ≈ {time_left()/60:.0f} min)')

    n_new = 0
    n_failed = 0
    last_error = None
    try:
        for i, (idx, row) in enumerate(tqdm(pending.iterrows(), total=len(pending), desc=desc)):
            if time_critical():
                print(f'Time budget nearly exhausted -- stopping "{desc}" early. '
                      f'{len(pending) - i} rows remain for the next run (re-run this '
                      f'notebook / commit and it will resume from here).')
                break
            try:
                r = process_fn(row)
            except Exception as e:
                # Previously this silently discarded every exception with no
                # trace -- if process_fn has a systematic bug, ALL rows would
                # fail and this loop would "succeed" with an empty checkpoint
                # CSV and zero diagnostic signal. Now we count failures and
                # keep the most recent exception so a 100%-failure run is
                # loud instead of silent.
                r = None
                n_failed += 1
                last_error = e
            if r is not None:
                results.append(r)
                n_new += 1
            if (i + 1) % save_every == 0:
                pd.DataFrame(results).to_csv(ckpt_csv_path, index=False)
    except KeyboardInterrupt:
        print(f'WARNING: Interrupted by user/system -- saving {n_new} newly-computed rows before exiting.')
    finally:
        out_df = pd.DataFrame(results)
        out_df.to_csv(ckpt_csv_path, index=False)
        print(f'{desc}: {len(out_df)} total rows saved to {ckpt_csv_path.name}')
        if n_failed > 0:
            print(f'WARNING: {desc}: {n_failed} row(s) raised an exception and were skipped '
                  f'(most recent: {type(last_error).__name__}: {last_error})')
        if len(pending) > 0 and n_new == 0 and n_failed == len(pending):
            print(f'{desc}: EVERY pending row failed with an exception -- this almost '
                  f'certainly indicates a bug in process_fn, not bad individual samples. '
                  f'Do not trust an empty/near-empty result from this checkpoint.')
    return out_df


def oom_safe_call(fn, *args, on_oom=None, **kwargs):
    """Run fn(*args, **kwargs); on CUDA OOM, clear cache and retry once via on_oom()."""
    try:
        return fn(*args, **kwargs)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print('WARNING: CUDA OOM caught -- clearing cache and retrying with reduced settings.')
        if on_oom is not None:
            return on_oom(*args, **kwargs)
        raise


# ── Save-on-exit safety net: if the kernel is killed by Kaggle's
# timeout/OOM-killer (not a clean Python exception), at least the
# most recent explicit save_ckpt/run_resumable_loop checkpoints
# already on disk survive, since they're flushed every save_every
# rows rather than only at the very end. ──────────────────────
def _on_exit():
    print(f'\nSession ending. Elapsed: {(time.time()-SESSION_START)/60:.1f} min. '
          f'Checkpoints in: {CKPT_DIR}')
atexit.register(_on_exit)

print('Checkpoint/resume infrastructure ready.')
print(f'Resume mode: {CFG["resume"]} | Time budget: {CFG["time_budget_seconds"]/3600:.1f}h')



## CELL 3 — Load HAM10000 Dataset from Kaggle (unchanged)


In [ ]:
# ============================================================
# CELL 3 — Dataset Discovery + Build DataFrame
# ============================================================
import os
from pathlib import Path
import pandas as pd

BASE = Path('/kaggle/input')
print("Searching for datasets in Kaggle input...")

meta_paths = list(BASE.rglob('*metadata*.csv')) + list(BASE.rglob('*HAM*.csv'))
if not meta_paths:
    raise FileNotFoundError("WARNING: Could not find the metadata CSV! Please make sure the HAM10000 dataset is attached.")

meta_path = meta_paths[0]
HAM_DIR = meta_path.parent
print(f'Found Metadata: {meta_path}')
print(f'Base Directory: {HAM_DIR}')

df = pd.read_csv(meta_path)
print(f'Raw metadata shape: {df.shape}')

CLASS_MAP_HAM = {
    'mel':'MEL','nv':'NV','bcc':'BCC',
    'akiec':'AKIEC','bkl':'BKL','df':'DF','vasc':'VASC'
}
# NOTE: do NOT redefine CLASS_TO_IDX here -- it was already set in CELL 2
# from CLASSES (alphabetical order); redefining it here would desync
# label_idx from CLASSES[idx] and silently corrupt every downstream
# class-name decode.

df['label'] = df['dx'].map(CLASS_MAP_HAM)
df['label_idx'] = df['label'].map(CLASS_TO_IDX)

img_dict = {}
print("Scanning for images...")
for img_file in HAM_DIR.rglob('*.jpg'):
    img_dict[img_file.stem] = str(img_file)
for img_file in HAM_DIR.rglob('*.png'):
    img_dict[img_file.stem] = str(img_file)

print(f'Total images found: {len(img_dict)}')

df['image_path'] = df['image_id'].map(img_dict)
df = df.dropna(subset=['image_path', 'label'])
df = df[df['image_path'].apply(lambda x: Path(x).exists())]

print(f'\nREADY! Valid images for training: {len(df)}')
print(f'\nClass distribution:')
print(df['label'].value_counts().to_string())



## CELL 4 — Deduplication + Stratified Train/Val/Test Split (unchanged, + skip-if-exists)


In [ ]:
# ============================================================
# CELL 4 — Patient-Level Stratified Split (prevent data leakage)
# ============================================================
from sklearn.model_selection import train_test_split

split_files = [CFG['out_dir']/'train_split.csv', CFG['out_dir']/'val_split.csv', CFG['out_dir']/'test_split.csv']
if CFG['resume'] and all(f.exists() for f in split_files):
    train_df = pd.read_csv(split_files[0])
    val_df = pd.read_csv(split_files[1])
    test_df = pd.read_csv(split_files[2])
    print('Loaded existing splits from checkpoint -- skipping resplit.')
else:
    if 'lesion_id' in df.columns:
        unique_lesions = df.drop_duplicates('lesion_id')[['lesion_id','label']]
        print(f'Unique lesions: {len(unique_lesions)} | Total images: {len(df)}')
        split_col = 'lesion_id'
    else:
        unique_lesions = df[['image_id','label']].drop_duplicates()
        split_col = 'image_id'

    train_ids, temp_ids = train_test_split(
        unique_lesions, test_size=0.20, random_state=SEED, stratify=unique_lesions['label']
    )
    val_ids, test_ids = train_test_split(
        temp_ids, test_size=0.50, random_state=SEED, stratify=temp_ids['label']
    )

    train_set = set(train_ids[split_col])
    val_set = set(val_ids[split_col])
    test_set = set(test_ids[split_col])

    train_df = df[df[split_col].isin(train_set)].reset_index(drop=True)
    val_df = df[df[split_col].isin(val_set)].reset_index(drop=True)
    test_df = df[df[split_col].isin(test_set)].reset_index(drop=True)

    train_df.to_csv(split_files[0], index=False)
    val_df.to_csv(split_files[1], index=False)
    test_df.to_csv(split_files[2], index=False)
    print('\nSplits saved.')

print(f'\nSplit Summary:')
print(f'Train: {len(train_df):,} images')
print(f'Val: {len(val_df):,} images')
print(f'Test: {len(test_df):,} images')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('HAM10000 Dataset Analysis', fontsize=16, fontweight='bold')
class_counts = train_df['label'].value_counts()
colors = [CLASS_COLORS[c] for c in class_counts.index]
bars = axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_title('Class Distribution (Training Set)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Lesion Class'); axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=30)
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 30,
                f'{count}', ha='center', va='bottom', fontsize=9, fontweight='bold')

max_count = class_counts.max()
ir = max_count / class_counts
ir_colors = ['#F44336' if v > 20 else '#FF9800' if v > 5 else '#4CAF50' for v in ir.values]
axes[1].bar(ir.index, ir.values, color=ir_colors, edgecolor='white', linewidth=1.2)
axes[1].set_title('Class Imbalance Ratio (NV=1x baseline)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Lesion Class'); axes[1].set_ylabel('Imbalance Ratio (×)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].axhline(y=20, color='red', linestyle='--', alpha=0.5, label='Severe imbalance (20×)')
axes[1].legend()

plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'Figure1_dataset_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise
print('Figure 1 saved.')



## CELL 5 — Hair Removal (DullRazor) + Preprocessing Pipeline (unchanged — already disk-cached)


In [ ]:
# ============================================================
# CELL 5 — DullRazor Hair Removal + Data Augmentation Pipeline
# ============================================================
import albumentations as A
from albumentations.pytorch import ToTensorV2

def dull_razor_hair_removal(image_np):
    # FIX: the previous version thresholded the blackhat response directly
    # (kernel=9x9 solid rectangle, threshold=10) and inpainted everything
    # above it. A blob-shaped kernel's blackhat response fires on ANY dark
    # region smaller than 9x9 -- not just hair, but also the lesion's own
    # pigment network, globules, and dots, which are visually similar in
    # scale. Visual inspection of the "before/after" sample-image figure
    # confirmed this: MEL/BCC/BKL/DF/AKIEC all lost their actual lesion
    # texture, replaced by smooth polygonal inpainting artifacts, because
    # cv2.inpaint was told to erase large chunks of real diagnostic
    # structure, not just thin hair strands.
    #
    # Hairs are THIN everywhere along their length; pigment structures are
    # comparatively wide/blobby. An earlier attempt at this fix filtered
    # candidate regions by whole-connected-component elongation (bounding
    # box aspect ratio), but that breaks down for the very common case of
    # multiple hairs crossing each other or touching the lesion boundary --
    # they merge into one blobby connected component and get (wrongly)
    # rejected as "not hair-shaped". Morphological OPENING is robust to
    # this because it tests LOCAL thickness at every pixel, not whole-shape
    # geometry: a disk kernel slightly wider than typical hair thickness
    # removes genuinely thin structures under opening (erosion narrower
    # than the kernel never recovers under the following dilation) while
    # wider blob-like pigment structures survive largely intact. Verified
    # against a synthetic test image (a large uniform disk + three crossing
    # diagonal lines): this version correctly masks the crossing "hairs"
    # while leaving the "lesion" interior untouched, where the earlier
    # elongation-based version masked 0 pixels at all (the merged crossing
    # lines failed its aspect-ratio test) and the original version masked
    # a large fraction of the lesion interior.
    #
    # Known residual limitation: a thin ring right at the lesion boundary
    # can still pick up a few pixels of blackhat response (the kernel spans
    # both the darker interior and lighter surrounding skin there), but this
    # is a boundary-proximity effect on the order of a few pixels, not the
    # near-total interior destruction the original version produced.
    gray = cv2.cvtColor(image_np, cv2.COLOR_BGR2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, candidate_mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)

    disk = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    opened = cv2.morphologyEx(candidate_mask, cv2.MORPH_OPEN, disk)
    hair_mask = cv2.subtract(candidate_mask, opened)

    # Discard tiny specks (threshold noise), not genuine hair segments.
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(hair_mask, connectivity=8)
    cleaned = np.zeros_like(hair_mask)
    for label in range(1, n_labels):
        if stats[label, cv2.CC_STAT_AREA] >= 8:
            cleaned[labels == label] = 255
    hair_mask = cleaned

    if hair_mask.sum() == 0:
        return image_np  # nothing hair-shaped detected -- leave the image untouched

    hair_mask = cv2.dilate(hair_mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)), iterations=1)
    result = cv2.inpaint(image_np, hair_mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA)
    return result

HAIR_REMOVED_CACHE_DIR = CFG['out_dir'] / 'hair_removed_cache'
HAIR_REMOVED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

class SkinDataset(Dataset):
    def __init__(self, df, transform=None, use_hair_removal=True, cache_dir=HAIR_REMOVED_CACHE_DIR):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.use_hair_removal = use_hair_removal
        self.cache_dir = Path(cache_dir) if cache_dir is not None else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = str(row.get('image_id', idx))
        cache_path = None
        if self.use_hair_removal and self.cache_dir is not None:
            cache_path = self.cache_dir / (img_id + '.jpg')

        if cache_path is not None and cache_path.exists():
            img_bgr = cv2.imread(str(cache_path))
        else:
            img_bgr = cv2.imread(str(row['image_path']))
            if img_bgr is None:
                img_bgr = np.array(Image.open(row['image_path']).convert('RGB'))
                img_bgr = cv2.cvtColor(img_bgr, cv2.COLOR_RGB2BGR)
            if self.use_hair_removal:
                img_bgr = dull_razor_hair_removal(img_bgr)
                if cache_path is not None:
                    cv2.imwrite(str(cache_path), img_bgr)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        if self.transform:
            augmented = self.transform(image=img_rgb)
            img_tensor = augmented['image']
        else:
            img_tensor = torch.from_numpy(img_rgb.transpose(2,0,1)).float() / 255.0
        label = int(row['label_idx'])
        return img_tensor, label, img_id

train_transform = A.Compose([
    A.Resize(CFG['img_size'], CFG['img_size']),
    A.RandomResizedCrop(size=(CFG['img_size'], CFG['img_size']), scale=(0.7, 1.0), p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=180, p=0.7),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05, p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(1, 20), hole_width_range=(1, 20),
                    fill=0, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(CFG['img_size'], CFG['img_size']),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# NEW (GAP-9): if CELL 17c has produced a validated dark-skin-augmented manifest
# and the opt-in flag is set, train on that instead of the plain split. This is
# what makes the GAP-9 before/after TIxAI-fairness comparison possible -- the
# flag defaults to False so nothing changes unless you deliberately opt in.
_aug_manifest_path = CFG['out_dir'] / 'train_df_augmented.csv'
if CFG.get('use_synthetic_dark_skin') and _aug_manifest_path.exists():
    print(f'CFG["use_synthetic_dark_skin"]=True -- loading augmented manifest '
          f'({_aug_manifest_path.name}) in place of the plain train split.')
    train_df = pd.read_csv(_aug_manifest_path)
elif CFG.get('use_synthetic_dark_skin'):
    print(f'WARNING: CFG["use_synthetic_dark_skin"]=True but {_aug_manifest_path.name} '
          f'does not exist yet -- run CELL 17c first. Falling back to the plain split.')

train_ds = SkinDataset(train_df, train_transform, use_hair_removal=True)
val_ds = SkinDataset(val_df, val_transform, use_hair_removal=True)
test_ds = SkinDataset(test_df, val_transform, use_hair_removal=True)

class_counts_train = train_df['label_idx'].value_counts().sort_index()
sample_weights = [1.0 / class_counts_train[label_idx]
                  for label_idx in train_df['label_idx']]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

def worker_init_fn(worker_id):
    np.random.seed(SEED + worker_id)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                          sampler=sampler, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'],
                          shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=16,
                          shuffle=False, num_workers=0, pin_memory=True)

print(f'Datasets created:')
print(f'Train: {len(train_ds):,} samples | {len(train_loader)} batches')
print(f'Val: {len(val_ds):,} samples | {len(val_loader)} batches')
print(f'Test: {len(test_ds):,} samples | {len(test_loader)} batches')

fig, axes = plt.subplots(2, 7, figsize=(21, 6))
fig.suptitle('Sample Images per Class (Before & After Hair Removal)', fontsize=14, fontweight='bold')
for ci, cls in enumerate(CLASSES):
    cls_df = train_df[train_df['label'] == cls]
    if len(cls_df) == 0:
        continue
    img_path = cls_df.sample(1).iloc[0]['image_path']
    img_orig = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    img_orig_r = cv2.resize(img_orig, (224, 224))
    img_hr = dull_razor_hair_removal(cv2.imread(img_path))
    img_hr = cv2.cvtColor(cv2.resize(img_hr, (224, 224)), cv2.COLOR_BGR2RGB)

    axes[0, ci].imshow(img_orig_r); axes[0, ci].set_title(f'{cls}\n(Original)', fontsize=9); axes[0, ci].axis('off')
    axes[1, ci].imshow(img_hr); axes[1, ci].set_title(f'{cls}\n(Hair Removed)', fontsize=9); axes[1, ci].axis('off')

plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'Figure2_sample_images.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise
print('Figure 2 (sample images) saved.')



## CELL 6 — DenseNet121 Architecture + Focal Loss (unchanged)


In [ ]:
# ============================================================
# CELL 6 — DenseNet121 Architecture (Primary Backbone)
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha.to(DEVICE) if alpha is not None else None
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        log_p = F.log_softmax(inputs, dim=1)
        p_t = torch.exp(
            log_p.gather(1, targets.unsqueeze(1))
        ).squeeze(1).clamp(min=1e-8)
        focal_weight = (1.0 - p_t).pow(self.gamma)
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
        else:
            alpha_t = torch.ones_like(p_t)
        loss = -alpha_t * focal_weight * torch.log(p_t)
        return loss.mean() if self.reduction == 'mean' else loss


class DenseNet121_TrustXAI(nn.Module):
    def __init__(self, n_classes=7, dropout_rate=0.3, use_mc_dropout=False):
        super().__init__()
        self.use_mc_dropout = use_mc_dropout
        self.backbone = models.densenet121(weights='DEFAULT')
        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity()
        self.dropout1 = nn.Dropout(p=dropout_rate)
        self.fc1 = nn.Linear(in_features, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.dropout2 = nn.Dropout(p=0.2)
        self.classifier = nn.Linear(512, n_classes)

    def forward(self, x):
        features = self.backbone(x)
        if self.use_mc_dropout:
            features = self.dropout1(features)
        x = F.relu(self.bn1(self.fc1(features)))
        x = self.dropout2(x)
        return self.classifier(x)

    def get_cam_target_layers(self):
        return [self.backbone.features.denseblock4.denselayer16.conv2]

    def get_param_groups(self):
        f = self.backbone.features
        stem_and_transitions = (
            list(f.conv0.parameters()) + list(f.norm0.parameters()) +
            list(f.transition1.parameters()) + list(f.transition2.parameters()) +
            list(f.transition3.parameters()) + list(f.norm5.parameters())
        )
        return [
            {'params': stem_and_transitions, 'lr': 1e-5},
            {'params': f.denseblock1.parameters(), 'lr': 1e-5},
            {'params': f.denseblock2.parameters(), 'lr': 1e-5},
            {'params': f.denseblock3.parameters(), 'lr': 5e-5},
            {'params': f.denseblock4.parameters(), 'lr': 5e-5},
            {'params': list(self.fc1.parameters()) +
                       list(self.bn1.parameters()) +
                       list(self.classifier.parameters()), 'lr': CFG['lr']},
        ]


class ConvNeXtTiny_Baseline(nn.Module):
    def __init__(self, n_classes=7):
        super().__init__()
        self.backbone = timm.create_model(
            'convnext_tiny', pretrained=True, num_classes=n_classes
        )

    def forward(self, x):
        return self.backbone(x)

    def get_cam_target_layers(self):
        # FIX (broken Grad-CAM hook, audit 2026-07-05): .conv_dw is only the
        # DEPTHWISE 7x7 conv inside the last ConvNeXt block -- depthwise means
        # each output channel is a function of exactly ONE input channel (no
        # cross-channel mixing), and it runs BEFORE that block's LayerNorm +
        # pointwise-MLP channel-mixing + residual add. Grad-CAM computed there
        # sees a spatial map that hasn't yet been shaped into class-
        # discriminative channels, which is consistent with the near-zero
        # ConvNeXt TIxAI (~0.0006) observed downstream -- a degenerate hook,
        # not a real localization score. Hooking the full last block instead
        # captures its output AFTER norm+MLP+residual, the ConvNeXt analogue
        # of DenseNet121's denselayer16.conv2 hook point (last spatial
        # feature map before global pooling).
        return [self.backbone.stages[-1].blocks[-1]]


n_train = len(train_df)
class_weights = torch.tensor(
    [n_train / (CFG['n_classes'] * train_df['label_idx'].value_counts().sort_index()[i])
     for i in range(CFG['n_classes'])],
    dtype=torch.float32
)
manual_alpha = {'DF': 3.0, 'VASC': 2.5, 'AKIEC': 2.0, 'BCC': 1.5,
                'MEL': 1.2, 'BKL': 1.2, 'NV': 1.0}
alpha = torch.tensor([manual_alpha[c] * class_weights[i].item()
                      for i, c in enumerate(CLASSES)], dtype=torch.float32)
alpha = alpha / alpha.sum() * CFG['n_classes']


class CombinedFocalSmoothedCE(nn.Module):
    """
    FIX (accuracy/calibration): GAP1's own training spec (Section 4.5,
    GAP1_Model_Development_Report.md) calls for
        L_total = L_focal + 0.1 * L_CE-smoothed   (label smoothing eps=0.1)
    to combine focal loss's rare-class focus with label smoothing's
    calibration/overconfidence benefit. The notebook previously used bare
    FocalLoss only -- this restores the auxiliary smoothed-CE term the
    design doc specifies (it also directly helps the temperature-scaling
    calibration step in CELL 9, which assumes a reasonably-calibrated model
    to begin with).
    """
    def __init__(self, alpha, gamma, smoothing=0.1, aux_weight=0.1):
        super().__init__()
        self.focal = FocalLoss(alpha=alpha, gamma=gamma)
        # FIX (RuntimeError: "weight is on cpu, different from other tensors on
        # cuda:0"): FocalLoss moves its own copy of alpha to DEVICE internally
        # (see FocalLoss.__init__ above), but nn.CrossEntropyLoss's `weight`
        # buffer does not get moved automatically just by constructing this
        # module -- it only moves if .to(device) is called on the module
        # afterward, which nothing here does. Move it explicitly, same as
        # FocalLoss does for its own alpha.
        alpha_device = alpha.to(DEVICE) if alpha is not None else None
        self.ce_smooth = nn.CrossEntropyLoss(label_smoothing=smoothing, weight=alpha_device)
        self.aux_weight = aux_weight

    def forward(self, inputs, targets):
        return self.focal(inputs, targets) + self.aux_weight * self.ce_smooth(inputs, targets)


assert torch.isfinite(alpha).all(), 'FocalLoss alpha contains non-finite values -- check class_weights/manual_alpha'
criterion = CombinedFocalSmoothedCE(alpha=alpha, gamma=CFG['focal_gamma'], smoothing=0.1, aux_weight=0.1)
criterion = criterion.to(DEVICE)  # FIX: move whole loss module to DEVICE so ce_smooth's weight buffer (and any other submodule buffers) land on cuda, not just alpha

model = DenseNet121_TrustXAI(
    n_classes=CFG['n_classes'],
    dropout_rate=CFG['dropout'],
    use_mc_dropout=False
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'DenseNet121-TrustXAI instantiated')
print(f'Total params: {total_params/1e6:.2f}M')
print(f'Trainable params: {trainable/1e6:.2f}M')
print(f'Grad-CAM++ hook: denseblock4.denselayer16.conv2')
print(f'Focal Loss alpha: {alpha.tolist()}')



## CELL 7 — Training DenseNet121 (MODIFIED: resume + OOM auto-retry + epoch checkpoint)


In [ ]:
# ============================================================
# CELL 7 — Training Loop with Early Stopping
# FIXES applied:
# - Skips training entirely if a finished best_*.pth + history.csv
# already exist and CFG['force_retrain'] is False (resume support).
# - On CUDA OOM, halves batch_size, rebuilds the loaders, and retries
# the current epoch instead of crashing the whole run.
# - Saves history.csv after EVERY epoch (not just at the end), so a
# killed session still has a usable training curve + best weights.
# - Respects the global time budget: stops cleanly (keeping best
# weights so far) instead of getting killed mid-epoch.
# - FIX (scheduler bug, audit 2026-07-05): OneCycleLR below is built with
# steps_per_epoch=len(train_loader), i.e. it expects one .step() call per
# BATCH. The previous version called scheduler.step() once per EPOCH,
# outside the batch loop, so across ~90 batches/epoch the LR schedule
# advanced at roughly 1% of its intended rate and never meaningfully
# warmed up or annealed. This is a very plausible root cause of the
# unusually slow val-F1 climb (0.08->0.34 over 60 epochs) that originally
# motivated raising CFG['epochs']/patience. scheduler.step() now fires
# once per batch, immediately after the optimizer step, matching
# OneCycleLR's steps_per_epoch contract. Any checkpoint trained before
# this fix was trained under the broken schedule -- see CFG['force_retrain']
# in CELL 2, which is set to retrain DenseNet121 from scratch once.
# ============================================================

def train_epoch(model, loader, optimizer, scheduler, criterion, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    n_skipped = 0
    for imgs, labels, _ in tqdm(loader, desc='Train', leave=False):
        if not torch.isfinite(imgs).all():
            continue
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)
        if not torch.isfinite(loss):
            n_skipped += 1
            continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    if n_skipped > 0:
        print('(skipped ' + str(n_skipped) + ' non-finite-loss batches this epoch)')
    if total == 0:
        return float('nan'), 0.0
    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    for imgs, labels, _ in tqdm(loader, desc='Val', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        if not torch.isfinite(imgs).all():
            continue
        logits = model(imgs)
        total_loss += criterion(logits, labels).item() * imgs.size(0)
        all_preds.append(logits.argmax(1).cpu())
        all_labels.append(labels.cpu())
    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    f1 = F1Score(task='multiclass', num_classes=CFG['n_classes'], average='macro')(preds, labels)
    bal_acc= Accuracy(task='multiclass', num_classes=CFG['n_classes'], average='macro')(preds, labels)
    return total_loss / len(loader.dataset), f1.item(), bal_acc.item()


def build_loaders(batch_size, pin_memory=True):
    """Rebuild loaders at a given batch size -- used by the OOM auto-halve path
    and by CELL 8 to give ConvNeXt-Tiny its own smaller, lower-memory loaders."""
    tl = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                     num_workers=0, pin_memory=pin_memory)
    vl = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                     num_workers=0, pin_memory=pin_memory)
    return tl, vl


def train_model(model, name='DenseNet121', epochs=None, patience=None,
                train_loader_override=None, val_loader_override=None,
                use_data_parallel=None):
    epochs = epochs or CFG['epochs']
    patience = patience or CFG['patience']

    best_path = CFG['out_dir'] / f'{name}_best.pth'
    hist_path = CFG['out_dir'] / f'{name}_history.csv'
    status_path = CFG['out_dir'] / f'{name}_status.json'

    # FIX (accuracy-blocking bug): the old gate here was simply
    #   `if best_path.exists() and hist_path.exists(): skip`.
    # Both files are written starting at epoch 1 (history.csv every epoch,
    # best.pth on the first improving epoch), so this ALWAYS short-circuited
    # once ANY training had happened. It had no way to tell "ran out of the
    # epochs=60 budget while still improving" (this run's actual situation --
    # val_f1 climbed 0.08->0.34 monotonically and patience never triggered)
    # apart from "genuinely converged." Practical effect: raising CFG['epochs']
    # and re-running silently did nothing -- it reloaded the same epoch-60
    # weights and printed a message that looked exactly like normal
    # resume-skip behaviour. Fixed by persisting *why* a run stopped
    # (status_path) and only truly skipping when that reason means more
    # epochs can not help (early-stopped on patience, or already reached >=
    # the newly-requested epochs); otherwise we fall through and continue
    # training from the saved weights for the remaining epochs.
    prev_status = None
    if status_path.exists():
        with open(status_path) as f:
            prev_status = json.load(f)

    if (CFG['resume'] and not CFG['force_retrain']
            and best_path.exists() and hist_path.exists() and prev_status is not None):
        already_ran = prev_status.get('last_epoch', 0)
        if prev_status.get('reason') == 'early_stopped' or already_ran >= epochs:
            print(f'{name}: previous run already {prev_status.get("reason", "finished")} '
                  f'at epoch {already_ran} (requested epochs={epochs}) -- skipping training. '
                  f'(set CFG["force_retrain"]=True to retrain from scratch anyway)')
            model.load_state_dict(torch.load(best_path, map_location=DEVICE))
            hist_df = pd.read_csv(hist_path)
            return model, hist_df
        else:
            print(f'{name}: previous run stopped at epoch {already_ran} due to '
                  f'"{prev_status.get("reason")}", but epochs={epochs} was requested '
                  f'(more than {already_ran}) -- continuing training instead of skipping.')
    elif (CFG['resume'] and not CFG['force_retrain']
            and best_path.exists() and hist_path.exists() and prev_status is None):
        # Checkpoint predates status_path (e.g. the original epoch-60 run).
        # Preserve the old skip-on-file-existence behaviour for it once -- but
        # write the marker now so the NEXT call with a higher epochs value can
        # correctly continue instead of skip forever.
        print(f'{name}: found existing best weights + history but no status marker '
              f'(pre-fix checkpoint) -- skipping training once more. Re-run again after '
              f'this, or delete {hist_path.name}, to actually continue training with the '
              f'new epochs={epochs} budget.')
        model.load_state_dict(torch.load(best_path, map_location=DEVICE))
        hist_df = pd.read_csv(hist_path)
        with open(status_path, 'w') as f:
            json.dump({'last_epoch': int(hist_df['epoch'].max()) if len(hist_df) else 0,
                       'reason': 'target_reached', 'target_epochs': epochs}, f)
        return model, hist_df

    # NEW: allow a caller (e.g. CELL 8's ConvNeXt-Tiny run) to supply its own
    # loaders at a smaller batch size than the global train_loader/val_loader,
    # instead of always reusing DenseNet121's CFG['batch_size']=64 loaders --
    # running two full models' worth of activations/gradients back-to-back in
    # one Kaggle session (with nn.DataParallel replicating each across both
    # T4s) is a real, avoidable source of extra peak memory for a model that
    # doesn't need to match the primary model's batch size.
    cur_train_loader = train_loader_override if train_loader_override is not None else train_loader
    cur_val_loader = val_loader_override if val_loader_override is not None else val_loader
    local_batch_size = cur_train_loader.batch_size

    # ── Resume mid-training if a partial history exists ────────────────────────
    start_epoch = 1
    history = []
    best_f1 = 0.0
    best_wts = copy.deepcopy(model.state_dict())
    no_improve = 0
    if CFG['resume'] and hist_path.exists() and not CFG['force_retrain']:
        prev_hist = pd.read_csv(hist_path)
        if len(prev_hist) > 0:
            history = prev_hist.to_dict('records')
            start_epoch = int(prev_hist['epoch'].max()) + 1
            best_f1 = float(prev_hist['val_f1'].max())
            if best_path.exists():
                best_wts = torch.load(best_path, map_location=DEVICE)
                model.load_state_dict(best_wts)
            print(f'Resuming {name} training from epoch {start_epoch} (best_f1 so far={best_f1:.4f})')

    optimizer = torch.optim.AdamW(
        model.get_param_groups() if hasattr(model, 'get_param_groups')
        else model.parameters(),
        lr=CFG['lr'], weight_decay=CFG['weight_decay']
    )
    # FIX: when continuing from start_epoch > 1, scope OneCycleLR to the
    # REMAINING epochs instead of the full original `epochs`. The previous
    # version always built OneCycleLR(epochs=epochs) regardless of start_epoch,
    # so on any multi-session resume the LR schedule silently restarted its
    # warmup+anneal from step 0 while the epoch counter kept incrementing --
    # i.e. every resume would re-warm the LR back up right when it should have
    # been annealing toward zero. Scoping to (epochs - start_epoch + 1) gives a
    # coherent warmup+anneal for exactly the remaining training (this is also
    # the standard "warm restart" pattern for extending an existing schedule).
    remaining_epochs = max(1, epochs - start_epoch + 1)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[pg['lr'] for pg in optimizer.param_groups],
        steps_per_epoch=len(cur_train_loader),
        epochs=remaining_epochs, pct_start=0.3
    )
    scaler = torch.cuda.amp.GradScaler()

    # NEW: allow a caller to opt OUT of nn.DataParallel even when N_GPUS > 1.
    # DataParallel's gather step (collecting per-GPU outputs back onto one
    # device every forward/backward call) is a recurring, not one-time, CPU
    # overhead -- this matters for a repeated kernel-restart OOM that keeps
    # happening mid-epoch (including during validation, which is gather-heavy)
    # rather than at a single fixed point. Defaults to the previous behavior
    # (use DataParallel whenever N_GPUS > 1) when not specified.
    _use_dp = use_data_parallel if use_data_parallel is not None else (N_GPUS > 1)
    forward_model = nn.DataParallel(model) if _use_dp else model
    if _use_dp:
        print(f'(training with nn.DataParallel across {N_GPUS} GPUs)')
    else:
        print(f'(training on a single GPU -- nn.DataParallel disabled for {name})')

    print(f'\nTraining {name} for up to {epochs} epochs (patience={patience}), starting at epoch {start_epoch}...')
    stop_reason = 'target_reached'
    for epoch in range(start_epoch, epochs + 1):
        if time_critical(margin=900):
            print(f'Time budget nearly exhausted -- stopping {name} training at epoch {epoch-1}. '
                  f'Re-run this notebook/commit to resume training from here.')
            stop_reason = 'time_budget'
            break

        t0 = time.time()
        try:
            tr_loss, tr_acc = train_epoch(forward_model, cur_train_loader, optimizer, scheduler, criterion, scaler)
            vl_loss, vl_f1, vl_bacc = validate(forward_model, cur_val_loader, criterion)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if local_batch_size <= CFG['min_batch_size']:
                print('WARNING: OOM at minimum batch size -- cannot reduce further. Stopping training.')
                break
            local_batch_size = max(CFG['min_batch_size'], local_batch_size // 2)
            print(f'WARNING: CUDA OOM -- halving batch size to {local_batch_size} and retrying this epoch.')
            cur_train_loader, cur_val_loader = build_loaders(local_batch_size)
            continue

        elapsed = time.time() - t0
        print(f'Epoch {epoch:03d}/{epochs} | '
              f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | '
              f'val_loss={vl_loss:.4f} val_f1={vl_f1:.4f} val_bacc={vl_bacc:.4f} | '
              f'{elapsed:.0f}s')

        history.append({
            'epoch': epoch, 'train_loss': tr_loss, 'train_acc': tr_acc,
            'val_loss': vl_loss, 'val_f1': vl_f1, 'val_bacc': vl_bacc
        })
        # Save history EVERY epoch -- this is what makes mid-training resume possible
        pd.DataFrame(history).to_csv(hist_path, index=False)

        # NEW: proactively release CUDA-cached (but no-longer-referenced)
        # memory every epoch. This does not fix a true per-batch leak (there
        # wasn't one in train_epoch/validate -- both already move results to
        # CPU/.item() before storing), but it does prevent the CUDA caching
        # allocator from holding onto stale blocks across a long 150-epoch
        # run, which matters more the longer a single Kaggle session goes.
        gc.collect()
        torch.cuda.empty_cache()

        if vl_f1 > best_f1:
            best_f1 = vl_f1
            best_wts = copy.deepcopy(model.state_dict())
            no_improve = 0
            torch.save(best_wts, best_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'[STOP] Early stop at epoch {epoch} (best val_f1={best_f1:.4f})')
                stop_reason = 'early_stopped'
                break

    model.load_state_dict(best_wts)
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(hist_path, index=False)
    last_epoch_reached = int(hist_df['epoch'].max()) if len(hist_df) else start_epoch - 1
    with open(status_path, 'w') as f:
        json.dump({'last_epoch': last_epoch_reached, 'reason': stop_reason,
                   'target_epochs': epochs}, f)
    print(f'\n{name} training complete/paused ({stop_reason}). Best val_F1 = {best_f1:.4f}')
    return model, hist_df


# ── Train DenseNet121 (PRIMARY model) ─────────────────────────
# NEW: two-phase schedule (CFG['training_phase'], set in CELL 2). Phase 1
# targets epoch 60 (a deliberate Kaggle-session boundary); phase 2 targets
# the full CFG['epochs'] (150) and relies on train_model()'s resume/
# continuation logic above to pick up from wherever phase 1 left off,
# NOT retrain from scratch.
PHASE1_EPOCHS = 60
# FIX (2026-07-05): a fresh full retrain must reach the FULL epoch budget in one
# session. Previously this targeted only PHASE1_EPOCHS (60) whenever training_phase==1
# (the default), so force_retrain=True would retrain from scratch but still STOP at 60
# -- an undertrained model, defeating the point of the re-run. force_retrain=True now
# forces the full CFG['epochs'] target regardless of phase; the deliberate 2-commit
# phase-1-stops-at-60 path still works when force_retrain is False.
_dense_epoch_target = (CFG['epochs'] if (CFG.get('force_retrain', False) or CFG['training_phase'] == 2)
                       else PHASE1_EPOCHS)
print(f'\nTraining phase {CFG["training_phase"]}/2 -- DenseNet121 target epoch: {_dense_epoch_target} '
      f'(full target: {CFG["epochs"]})')
model, history_dense = train_model(model, name='DenseNet121', epochs=_dense_epoch_target)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DenseNet121 Training Curves', fontsize=14, fontweight='bold')
metrics = [('train_loss', 'val_loss', 'Loss'), ('train_acc', None, 'Train Accuracy'), ('val_f1', 'val_bacc', 'Validation Metrics')]
for ax, (m1, m2, title) in zip(axes, metrics):
    ax.plot(history_dense['epoch'], history_dense[m1], 'b-', label=m1)
    if m2 and m2 in history_dense:
        ax.plot(history_dense['epoch'], history_dense[m2], 'r--', label=m2)
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'Figure3_training_curves.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise



## CELL 8 — Train ConvNeXt-Tiny (MODIFIED: same resume support as Cell 7)


In [ ]:
# ============================================================
# CELL 8 — ConvNeXt-Tiny Baseline Training
# Reuses the resumable train_model() from CELL 7 -- same resume,
# OOM-halving, and time-budget behavior applies here automatically.
# ============================================================

# Guard: Cell 8 needs Cells 1-7 to have already run in THIS kernel session
# (model class defs, data loaders, train_model, etc). After a kernel
# restart, re-run Cells 1-7 first -- thanks to the checkpoint/resume logic
# in CELL 2b and train_model(), DenseNet121 will reload its saved best
# weights instead of retraining from scratch, so that's fast, not another
# full training run.
for _dep in ('ConvNeXtTiny_Baseline', 'train_model', 'build_loaders', 'train_loader', 'val_loader', 'CFG', 'DEVICE'):
    if _dep not in globals():
        raise RuntimeError(
            f"'{_dep}' is not defined -- looks like the kernel was restarted "
            f"and Cells 1-7 haven't been re-run yet in this session. Run "
            f"Cells 1-7 again (Kernel > Restart & Run All is easiest), then "
            f"re-run this cell."
        )

# FIX (kernel restart: "notebook tried to allocate more memory than is
# available"): this crashed partway through ConvNeXt-Tiny's FIRST epoch on a
# Kaggle GPU T4 x2 session. Two contributing factors, addressed below:
#
# 1. Nothing released CELL 7's training-time memory before starting a SECOND
#    full model here. CELL 7's DenseNet121 `model` itself must stay alive --
#    every cell from CELL 9 onward (calibration, TIxAI, XDI, CSM, everything)
#    needs it -- so it is NOT deleted. But the optimizer state, gradient
#    buffers, OneCycleLR scheduler, and nn.DataParallel wrapper CELL 7 built
#    were all local to that train_model() call and are already
#    Python-garbage-collectible by now; explicitly running gc.collect() +
#    torch.cuda.empty_cache() here returns any CUDA-cached blocks from that
#    call back to the allocator instead of leaving them cached "just in case".
# 2. ConvNeXt-Tiny doesn't need to reuse DenseNet121's CFG['batch_size']=64
#    loaders -- it's a secondary accuracy baseline, not the primary XAI
#    instrument, so there's no reason to run it at the same batch size,
#    especially while a second full model + nn.DataParallel replication
#    across both T4s adds its own peak-memory cost on top of whatever CELL
#    7 already used. Build smaller-batch loaders for it specifically via the
#    same build_loaders() helper CELL 7's own OOM-auto-halving path uses.
gc.collect()
torch.cuda.empty_cache()

# FIX (kernel restart STILL happened after batch_size//4 + pin_memory=False):
# those two changes cut ConvNeXt-Tiny's OWN footprint, but CELL 7's
# DenseNet121 `model` is still sitting on BOTH GPUs the whole time (weights
# + its nn.DataParallel replica + whatever CUDA allocator blocks are still
# tagged to it), on top of ConvNeXt-Tiny's own weights + activations + a
# SECOND nn.DataParallel replication -- two full models resident on a
# T4 x2 session at once is the actual peak-memory driver, not just
# ConvNeXt-Tiny's batch size. `model` must stay ALIVE (CELL 9+ needs it)
# but it does not need to stay ON the GPU while ConvNeXt-Tiny trains --
# move it to CPU for the duration of this cell, then back to DEVICE
# afterward.
model = model.cpu()
gc.collect()
torch.cuda.empty_cache()

# NEW: train_loader/val_loader (batch_size=64, pin_memory=True, built in
# CELL 5) are confirmed unused anywhere after this point -- CELL 19's
# calibration builds its own val_cal_loader, not this one, and train_loader
# is never referenced again at all. They've been sitting in memory holding
# pinned host buffers for the whole session for no reason; free them.
for _stale_name in ('train_loader', 'val_loader'):
    if _stale_name in globals():
        del globals()[_stale_name]
gc.collect()

CONVNEXT_BATCH_SIZE = max(CFG['min_batch_size'], CFG['batch_size'] // 4)
convnext_train_loader, convnext_val_loader = build_loaders(CONVNEXT_BATCH_SIZE, pin_memory=False)
print(f'ConvNeXt-Tiny: using batch_size={CONVNEXT_BATCH_SIZE}, pin_memory=False '
      f'(DenseNet121 used batch_size={CFG["batch_size"]}, pin_memory=True).')

# NEW: this OOM-kill recurred even with DenseNet121 moved to CPU, batch
# size at a quarter of DenseNet's, pin_memory off, gc/cache clearing every
# epoch, all matplotlib figures closed, and ipywidget tqdm replaced with
# plain text -- it happened DURING validation specifically. nn.DataParallel's
# gather step (collecting per-GPU outputs back onto one device) is a
# recurring per-batch CPU overhead, not a one-time cost, and validation is
# exactly where that gather runs most densely. ConvNeXt-Tiny (~28M params)
# does not need 2 GPUs for reasonable speed -- run it on a single GPU
# instead of guessing at yet another shrink to DataParallel's footprint.
USE_DATA_PARALLEL_CONVNEXT = False

model_convnext = ConvNeXtTiny_Baseline(n_classes=CFG['n_classes']).to(DEVICE)
model_convnext.get_param_groups = lambda: model_convnext.parameters()

model_convnext, history_convnext = train_model(
    model_convnext, name='ConvNeXt-Tiny', epochs=30, patience=8,
    train_loader_override=convnext_train_loader, val_loader_override=convnext_val_loader,
    use_data_parallel=USE_DATA_PARALLEL_CONVNEXT
)

print('\nConvNeXt-Tiny training complete/paused.')

# Restore DenseNet121 to GPU now that ConvNeXt-Tiny is done -- CELL 9
# onward (calibration, TIxAI, XDI, CSM, etc.) needs `model` back on DEVICE.
model = model.to(DEVICE)



## CELL 9 — Temperature Scaling Calibration + Test Evaluation (MODIFIED: checkpointed)


In [ ]:
# ============================================================
# CELL 9 — Temperature Scaling Post-Calibration
# FIX: results + T values are pickled, so a re-run doesn't redo the
# full-test-set forward pass (the slowest part) unless you ask it to.
# ============================================================
from sklearn.model_selection import train_test_split

val_sel_df, val_cal_df = train_test_split(val_df, test_size=0.5, random_state=SEED, stratify=val_df['label'])
val_cal_ds = SkinDataset(val_cal_df, val_transform, use_hair_removal=True)
val_cal_loader = DataLoader(val_cal_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=0, pin_memory=True)

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature


@torch.no_grad()
def collect_logits_labels(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for imgs, labels, _ in tqdm(loader, desc='Collecting', leave=False):
        imgs = imgs.to(DEVICE)
        all_logits.append(model(imgs).cpu())
        all_labels.append(labels)
    return torch.cat(all_logits), torch.cat(all_labels)


def temperature_scale(model, val_loader):
    ts = TemperatureScaler().to(DEVICE)
    logits, labels = collect_logits_labels(model, val_loader)
    logits = logits.to(DEVICE)
    labels = labels.to(DEVICE)
    optimizer = torch.optim.LBFGS([ts.temperature], lr=0.01, max_iter=50)
    criterion_nll = nn.CrossEntropyLoss()

    def eval_closure():
        optimizer.zero_grad()
        loss = criterion_nll(ts(logits), labels)
        loss.backward()
        return loss

    optimizer.step(eval_closure)
    T_opt = ts.temperature.item()
    print(f'Optimal temperature T* = {T_opt:.4f}')
    return T_opt


@torch.no_grad()
def full_evaluate(model, loader, T_scale=1.0, model_name='Model'):
    model.eval()
    all_preds, all_probs, all_labels, all_ids = [], [], [], []
    for imgs, labels, ids in tqdm(loader, desc=f'Eval {model_name}', leave=False):
        imgs = imgs.to(DEVICE)
        logits = model(imgs) / T_scale
        probs = torch.softmax(logits, dim=1).cpu()
        all_preds.append(logits.argmax(1).cpu())
        all_probs.append(probs)
        all_labels.append(labels)
        all_ids.extend(ids)

    preds = torch.cat(all_preds)
    probs = torch.cat(all_probs)
    labels = torch.cat(all_labels)

    acc = Accuracy(task='multiclass', num_classes=CFG['n_classes'])(preds, labels).item()
    bal_acc = Accuracy(task='multiclass', num_classes=CFG['n_classes'], average='macro')(preds, labels).item()
    macro_f1= F1Score(task='multiclass', num_classes=CFG['n_classes'], average='macro')(preds, labels).item()
    macro_auc=AUROC(task='multiclass', num_classes=CFG['n_classes'])(probs, labels).item()
    mcc = MatthewsCorrCoef(task='multiclass', num_classes=CFG['n_classes'])(preds, labels).item()
    ece = CalibrationError(task='multiclass', num_classes=CFG['n_classes'], n_bins=15)(probs, labels).item()

    labels_np = labels.numpy(); preds_np = preds.numpy(); probs_np = probs.numpy()
    kappa = cohen_kappa_score(labels_np, preds_np)
    y_bin = label_binarize(labels_np, classes=list(range(CFG['n_classes'])))
    brier = np.mean([brier_score_loss(y_bin[:, c], probs_np[:, c]) for c in range(CFG['n_classes'])])
    pr_aucs = [average_precision_score((labels_np == c).astype(int), probs_np[:, c]) for c in range(CFG['n_classes'])]
    macro_prauc = np.mean(pr_aucs)

    def bootstrap_metric(y_t, y_p, metric_fn, n_boot=2000, seed=42):
        rng = np.random.RandomState(seed)
        scores = []
        for _ in range(n_boot):
            idx = rng.choice(len(y_t), len(y_t), replace=True)
            scores.append(metric_fn(y_t[idx], y_p[idx]))
        return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)

    ba_mean, ba_lo, ba_hi = bootstrap_metric(labels_np, preds_np, balanced_accuracy_score)

    print(f'\n{model_name} Results (T={T_scale:.3f}):')
    print(f'Accuracy: {acc:.4f}')
    print(f'Balanced Accuracy: {bal_acc:.4f} (95% CI: {ba_lo:.4f} – {ba_hi:.4f})')
    print(f'Macro F1: {macro_f1:.4f}')
    print(f'Macro AUC-ROC: {macro_auc:.4f}')
    print(f'Macro PR-AUC: {macro_prauc:.4f}')
    print(f'MCC: {mcc:.4f}')
    print(f'Cohen Kappa: {kappa:.4f}')
    print(f'Brier Score: {brier:.4f}')
    print(f'ECE: {ece:.4f} (target: <0.08)')
    # FIX: full_evaluate() is also called on external-validation loaders
    # (e.g. PAD-UFES-20, CELL 9c) whose diagnostic codes never include a
    # DF or VASC equivalent -- classification_report() then infers a
    # `labels` array shorter than the 7-entry `target_names=CLASSES`,
    # raising "Number of classes, 6, does not match size of target_names,
    # 7". Passing labels=range(n_classes) explicitly always reports the
    # full label space (missing classes just show 0 support) so this
    # works on any loader, not only ones containing all 7 classes.
    print('\n' + classification_report(labels.numpy(), preds.numpy(),
                                       labels=list(range(CFG['n_classes'])),
                                       target_names=CLASSES, zero_division=0))

    return {
        'accuracy': acc, 'balanced_acc': bal_acc, 'macro_f1': macro_f1,
        'macro_auc': macro_auc, 'macro_prauc': macro_prauc, 'mcc': mcc,
        'kappa': kappa, 'brier': brier, 'ece': ece,
        'preds': preds, 'probs': probs, 'labels': labels, 'ids': all_ids
    }


_ckpt9 = load_ckpt('cell9_eval')
if _ckpt9 is not None:
    T_dense, dense_results, convnext_T, convnext_results = (
        _ckpt9['T_dense'], _ckpt9['dense_results'], _ckpt9['convnext_T'], _ckpt9['convnext_results']
    )
else:
    print('Calibrating DenseNet121...')
    T_dense = temperature_scale(model, val_cal_loader)
    print('\n'+ '='*60)
    dense_results = full_evaluate(model, test_loader, T_dense, 'DenseNet121')
    print('='*60)
    convnext_T = temperature_scale(model_convnext, val_cal_loader)
    convnext_results = full_evaluate(model_convnext, test_loader, convnext_T, 'ConvNeXt-Tiny')
    save_ckpt('cell9_eval', {'T_dense': T_dense, 'dense_results': dense_results,
                              'convnext_T': convnext_T, 'convnext_results': convnext_results})

comp_df = pd.DataFrame({
    'Metric': ['Accuracy','Balanced Acc','Macro F1','Macro AUC','Macro PR-AUC','MCC','Cohen Kappa','Brier Score','ECE'],
    'DenseNet121': [dense_results[k] for k in ['accuracy','balanced_acc','macro_f1','macro_auc','macro_prauc','mcc','kappa','brier','ece']],
    'ConvNeXt-Tiny':[convnext_results[k] for k in ['accuracy','balanced_acc','macro_f1','macro_auc','macro_prauc','mcc','kappa','brier','ece']],
})
comp_df.to_csv(CFG['out_dir'] / 'Table2_model_comparison.csv', index=False)
print('\nModel Comparison (Table 2 for paper):')
print(comp_df.to_string(index=False))



## CELL 9b — Per-Class ROC-AUC Bar Chart (unchanged)


In [ ]:
# ============================================================
# CELL 9b -- Per-Class ROC-AUC Bar Chart
# ============================================================

def per_class_auc(probs, labels, n_classes):
    labels_np = labels.numpy()
    probs_np = probs.numpy()
    aucs = []
    for c in range(n_classes):
        y_true_c = (labels_np == c).astype(int)
        if y_true_c.sum() == 0 or y_true_c.sum() == len(y_true_c):
            aucs.append(np.nan)
            continue
        aucs.append(roc_auc_score(y_true_c, probs_np[:, c]))
    return aucs

dense_auc_per_class = per_class_auc(dense_results['probs'], dense_results['labels'], CFG['n_classes'])
convnext_auc_per_class = per_class_auc(convnext_results['probs'], convnext_results['labels'], CFG['n_classes'])

auc_table = pd.DataFrame({
    'Class': CLASSES,
    'DenseNet121_AUC': dense_auc_per_class,
    'ConvNeXt-Tiny_AUC': convnext_auc_per_class,
})
auc_table.to_csv(CFG['out_dir'] / 'Table_per_class_auc.csv', index=False)
print('\nPer-Class ROC-AUC (one-vs-rest):')
print(auc_table.to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(CLASSES))
width = 0.35
b1 = ax.bar(x - width/2, dense_auc_per_class, width, label='DenseNet121 (Primary)',
            color='#2196F3', edgecolor='white', linewidth=0.6)
b2 = ax.bar(x + width/2, convnext_auc_per_class, width, label='ConvNeXt-Tiny (Baseline)',
            color='#FF5722', edgecolor='white', linewidth=0.6)
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01, f'{h:.3f}',
                    ha='center', va='bottom', fontsize=8)
ax.axhline(dense_results['macro_auc'], color='#2196F3', linestyle='--', alpha=0.6,
           label=f"DenseNet121 macro AUC={dense_results['macro_auc']:.3f}")
ax.axhline(convnext_results['macro_auc'], color='#FF5722', linestyle='--', alpha=0.6,
           label=f"ConvNeXt-Tiny macro AUC={convnext_results['macro_auc']:.3f}")
ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=20)
ax.set_ylabel('ROC-AUC (one-vs-rest)'); ax.set_ylim(0, 1.08)
ax.set_title('Per-Class ROC-AUC: DenseNet121 vs ConvNeXt-Tiny', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, ncol=2); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'Figure3b_roc_auc_barchart.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise
print('\nFigure 3b (Per-Class ROC-AUC bar chart) saved.')



## CELL 9c — External Validation on PAD-UFES-20 (NEW — Novelty Add #4)


In [ ]:
# ============================================================
# CELL 9c — External Validation (PAD-UFES-20)
#
# NEW: Addresses the audit's "missing experiment": all headline numbers
# so far are single-dataset (HAM10000). This re-evaluates the trained
# DenseNet121 on PAD-UFES-20 (different camera/population), and saves
# an external-validation results object that CELL 10/13 can optionally
# reuse for a second TIxAI / Confusion-Saliency-Matrix pass.
#
# OPTIONAL: skips gracefully (does not crash the notebook) if the
# PAD-UFES-20 dataset isn't attached to this Kaggle session.
# Add it via: Add Data -> search "PAD-UFES-20".
# PAD diagnostic codes that don't exist in HAM10000 (e.g. SCC) are
# dropped rather than guessed into a wrong bucket.
# ============================================================
# FIX: same nested-mount bug as CELL 10's MASK_DIR -- iterdir() only
# saw immediate children of /kaggle/input, but datasets mount nested
# (/kaggle/input/datasets/<owner>/<slug>/...), so this always reported
# "not attached" even when PAD-UFES-20 was correctly attached. Search
# recursively for the metadata CSV itself instead, same pattern CELL 3
# already uses successfully for HAM10000.
pad_meta_paths = [p for p in Path('/kaggle/input').rglob('*metadata*.csv')
                   if 'pad' in str(p).lower() and 'ufes' in str(p).lower()]
PAD_DIR = pad_meta_paths[0].parent if pad_meta_paths else None

if PAD_DIR is None:
    print('ℹ PAD-UFES-20 not attached -- skipping external validation. '
          'Add the dataset via "Add Data" to enable this cell.')
    pad_df = None
    pad_results = None
else:
    if not pad_meta_paths:
        print('WARNING: PAD-UFES-20 attached but metadata CSV not found -- skipping.')
        pad_df = None
        pad_results = None
    else:
        pad_meta = pd.read_csv(pad_meta_paths[0])
        # PAD-UFES-20 diagnostic codes -> HAM10000-compatible labels.
        # SCC (squamous cell carcinoma) has no HAM10000 equivalent class -> dropped.
        PAD_CLASS_MAP = {'NEV':'NV','MEL':'MEL','BCC':'BCC','ACK':'AKIEC','SEK':'BKL'}
        diag_col = 'diagnostic' if 'diagnostic' in pad_meta.columns else pad_meta.columns[
            [c for c in pad_meta.columns if 'diag' in c.lower()][0:1][0]
        ]
        pad_meta['label'] = pad_meta[diag_col].map(PAD_CLASS_MAP)
        n_dropped = pad_meta['label'].isna().sum()
        pad_df = pad_meta.dropna(subset=['label']).copy()
        pad_df['label_idx'] = pad_df['label'].map(CLASS_TO_IDX)
        print(f'Dropped {n_dropped} PAD-UFES-20 rows with no HAM10000-equivalent class (e.g. SCC).')

        img_id_col = 'img_id' if 'img_id' in pad_df.columns else pad_df.columns[0]
        pad_img_dict = {f.stem: str(f) for f in PAD_DIR.rglob('*.png')}
        pad_img_dict.update({f.stem: str(f) for f in PAD_DIR.rglob('*.jpg')})
        pad_df['image_path'] = pad_df[img_id_col].apply(
            lambda x: pad_img_dict.get(Path(str(x)).stem, None)
        )
        pad_df = pad_df.dropna(subset=['image_path'])
        pad_df = pad_df[pad_df['image_path'].apply(lambda p: Path(p).exists())]
        # synthesize an image_id col matching the rest of the pipeline's convention
        if 'image_id' not in pad_df.columns:
            pad_df['image_id'] = pad_df[img_id_col].astype(str)
        pad_df = pad_df.reset_index(drop=True)
        print(f'PAD-UFES-20 ready: {len(pad_df)} valid images, '
              f'classes present: {sorted(pad_df["label"].unique().tolist())}')

        if len(pad_df) > 0:
            pad_ds = SkinDataset(pad_df, val_transform, use_hair_removal=True)
            pad_loader = DataLoader(pad_ds, batch_size=CFG['batch_size'], shuffle=False,
                                     num_workers=0, pin_memory=True)
            _ckpt9c = load_ckpt('cell9c_pad_eval')
            if _ckpt9c is not None:
                pad_results = _ckpt9c
            else:
                pad_results = full_evaluate(model, pad_loader, T_dense, 'DenseNet121-on-PAD-UFES-20')
                save_ckpt('cell9c_pad_eval', pad_results)

            print('\nGeneralization gap (HAM10000 test -> PAD-UFES-20):')
            for k in ['accuracy','balanced_acc','macro_f1','macro_auc']:
                gap = dense_results[k] - pad_results[k]
                print(f'{k:15s}: HAM10000={dense_results[k]:.4f} PAD={pad_results[k]:.4f} '
                      f'drop={gap:+.4f} ({100*gap/dense_results[k]:+.1f}%)')
            pd.DataFrame([{
                'dataset': 'PAD-UFES-20', **{k: pad_results[k] for k in
                    ['accuracy','balanced_acc','macro_f1','macro_auc','macro_prauc','mcc','kappa','brier','ece']}
            }]).to_csv(CFG['out_dir'] / 'Table_external_validation_PAD.csv', index=False)
        else:
            pad_results = None



## CELL 10 — GAP-1: Per-Class TIxAI (MODIFIED: resumable, chunked checkpointing)


In [ ]:
# ============================================================
# CELL 10 — GAP #1: Per-Class TIxAI Computation
# FIX: now uses run_resumable_loop -- a killed/stopped session resumes
# from the last saved batch instead of recomputing every image.
# ============================================================
# FIX: the previous version only checked immediate children of
# /kaggle/input via iterdir() -- but datasets are actually mounted
# nested (e.g. /kaggle/input/datasets/<owner>/<slug>/...), same as
# CELL 3 discovered for the HAM10000 metadata. iterdir() never saw
# past the first level, so it always reported "not found" even with
# a correctly-attached mask dataset. Search recursively instead.
masks_dict = {}
_mask_dirs = [d for d in Path('/kaggle/input').rglob('*') if d.is_dir() and d.name.lower() == 'masks']
if _mask_dirs:
    # Dataset ships a dedicated masks/ folder (e.g. surajghuwalewala/ham1000-
    # segmentation-and-classification) -- filenames there may or may not carry
    # the ISIC-specific _segmentation suffix, so match on stem either way.
    for _mdir in _mask_dirs:
        for f in _mdir.rglob('*'):
            if f.is_file() and f.suffix.lower() in ('.png', '.jpg', '.jpeg'):
                masks_dict[f.stem.replace('_segmentation', '')] = str(f)
else:
    # Fallback: official ISIC2018 Task 1 naming convention, wherever it lives.
    for f in Path('/kaggle/input').rglob('*_segmentation.png'):
        masks_dict[f.stem.replace('_segmentation', '')] = str(f)

if masks_dict:
    USE_SYNTHETIC_MASKS = False
    print(f'Found {len(masks_dict)} segmentation masks')
else:
    print('WARNING: ISIC 2018 masks not found! Add dataset: shonenkov/isic2018')
    print('Creating synthetic masks for demonstration...')
    USE_SYNTHETIC_MASKS = True


def compute_tixai(gradcam_map, mask_path_or_array):
    if isinstance(mask_path_or_array, str):
        mask = cv2.imread(mask_path_or_array, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (gradcam_map.shape[1], gradcam_map.shape[0]))
    else:
        mask = mask_path_or_array
    mask_bin = (mask > 127).astype(np.float32)
    total_sal = gradcam_map.sum() + CFG['tixai_epsilon']
    inside = (gradcam_map * mask_bin).sum()
    return float(inside / total_sal)


def make_synthetic_mask(img_size=224):
    mask = np.zeros((img_size, img_size), dtype=np.uint8)
    center = (img_size//2, img_size//2)
    rx, ry = int(img_size * 0.3), int(img_size * 0.35)
    cv2.ellipse(mask, center, (rx, ry), 0, 0, 360, 255, -1)
    noise = np.random.randint(0, 30, (img_size, img_size), dtype=np.uint8)
    mask = np.clip(mask.astype(np.int32) + noise - 15, 0, 255).astype(np.uint8)
    return mask


cam_dense = GradCAMPlusPlus(model=model, target_layers=model.get_cam_target_layers())
model.eval()
_mean_n = np.array([0.485,0.456,0.406]); _std_n = np.array([0.229,0.224,0.225])

def _tixai_process_row(row):
    img_id = str(row.get('image_id', row.name))
    true_cls = row['label']
    true_idx = int(row['label_idx'])
    img_path = row['image_path']

    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None
    img_bgr = dull_razor_hair_removal(img_bgr)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_rs = cv2.resize(img_rgb, (CFG['img_size'], CFG['img_size']))
    img_norm = img_rs.astype(np.float32) / 255.0
    img_t = torch.from_numpy(((img_norm - _mean_n) / _std_n).transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(img_t) / T_dense
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = int(probs.argmax())
        pred_cls = CLASSES[pred_idx]
        confidence= float(probs.max())

    if pred_idx != true_idx:
        return {'image_id': img_id, 'true_class': true_cls, 'pred_class': pred_cls,
                'confidence': confidence, 'correct': False, 'tixai': None, 'mask_source': None}

    targets = [ClassifierOutputTarget(pred_idx)]
    try:
        gradcam_map = cam_dense(input_tensor=img_t, targets=targets)[0]
    except Exception:
        return None

    if not USE_SYNTHETIC_MASKS and img_id in masks_dict:
        score = compute_tixai(gradcam_map, masks_dict[img_id])
        mask_source = 'GT'
    else:
        synth_mask = make_synthetic_mask(CFG['img_size'])
        score = compute_tixai(gradcam_map, synth_mask)
        mask_source = 'synthetic'

    return {'image_id': img_id, 'true_class': true_cls, 'pred_class': pred_cls,
            'confidence': confidence, 'correct': True, 'tixai': score, 'mask_source': mask_source}


print('\nComputing Per-Class TIxAI (GAP #1)...')
print('(Only correctly classified samples — TIxAI measures explanation quality\n'
      'conditional on correct prediction)\n')

tixai_df = run_resumable_loop(
    test_df, CFG['out_dir'] / 'tixai_results_dense.csv', _tixai_process_row,
    id_col='image_id', desc='TIxAI'
)

correct_df = tixai_df[tixai_df['correct'] == True].dropna(subset=['tixai'])
correct_df['tixai'] = correct_df['tixai'].astype(float)
print(f'\nTIxAI computed for {len(correct_df)} correctly classified samples')

n_gt = (correct_df['mask_source'] == 'GT').sum()
n_synth = (correct_df['mask_source'] == 'synthetic').sum()
TIXAI_GT_FRACTION = n_gt / max(len(correct_df), 1)
TIXAI_USES_SYNTHETIC_MASKS = n_synth > 0
if TIXAI_USES_SYNTHETIC_MASKS:
    print(f'\nWARNING: WARNING: {n_synth}/{len(correct_df)} TIxAI scores used SYNTHETIC masks, not real ISIC2018 ground truth.')
    pct_synth = 100 * (1 - TIXAI_GT_FRACTION)
    print(f'({round(pct_synth, 1)}% synthetic). Attach the ISIC2018 Task 1 dataset for publication-grade numbers.')
else:
    print(f'\nAll {n_gt} TIxAI scores use real ISIC2018 ground-truth masks.')

tixai_stats = []
for cls in CLASSES:
    cls_scores = correct_df[correct_df['true_class'] == cls]['tixai'].values
    if len(cls_scores) == 0:
        continue
    # FIX: scipy.stats.bootstrap requires >=2 observations per sample --
    # it previously only guarded against exactly 0, so a class with a
    # single correctly-classified TIxAI score (realistic for the rarest
    # classes, DF/VASC, which have only ~90 raw training images) raised
    # "each sample in data must contain two or more observations along
    # axis" and crashed the whole cell instead of just that class's CI.
    # Report the point estimate with a NaN CI in that case, consistent
    # with the n>=5 guards already used elsewhere in this notebook
    # (Kruskal-Wallis grouping, Confusion Saliency Matrix rendering).
    if len(cls_scores) >= 2:
        res = bootstrap((cls_scores,), np.median, n_resamples=10000,
                        confidence_level=0.95, method='BCa', random_state=SEED)
        ci_low, ci_high = res.confidence_interval.low, res.confidence_interval.high
    else:
        ci_low, ci_high = float('nan'), float('nan')
    tixai_stats.append({
        'class': cls, 'n_samples': len(cls_scores),
        'median_tixai': np.median(cls_scores), 'mean_tixai': np.mean(cls_scores),
        'std_tixai': np.std(cls_scores),
        'ci_low': ci_low, 'ci_high': ci_high,
        'iqr': np.percentile(cls_scores, 75) - np.percentile(cls_scores, 25),
        'reliability_tier': 'High' if np.median(cls_scores)>0.65 else
                           'Moderate' if np.median(cls_scores)>0.45 else 'Low'
    })

tixai_stats_df = pd.DataFrame(tixai_stats)
tixai_stats_df.to_csv(CFG['out_dir'] / 'Table3_tixai_per_class.csv', index=False)
print('\nPer-Class TIxAI Statistics (Table 3 for paper):')
print(tixai_stats_df[['class','n_samples','median_tixai','ci_low','ci_high','reliability_tier']].to_string(index=False))



## CELL 10b — GAP-7: Per-Fitzpatrick-Skin-Type (FST) Explanation Fairness Audit (NEW — Novelty Add #1)


In [ ]:
# ============================================================
# CELL 10b — Per-FST Explanation-Reliability Audit
#
# NEW: NOVELTY: nobody has audited whether explanation reliability
# (TIxAI) itself is equitable across skin tones -- existing fairness
# work (Bhattacharyya 2026, Role-of-Calibration 2025) only audits
# ACCURACY/CALIBRATION fairness, never explanation quality.
#
# HAM10000 has no FST labels, so we estimate skin tone via the
# Individual Typology Angle (ITA) on peri-lesional (non-lesion) skin
# pixels -- a standard dermatology proxy. This is an ESTIMATE, not a
# clinical FST label; report it with that caveat (same caveat your
# audit already raised about Fitzpatrick17k's own atlas-sourced FST
# labels being noisy).
# ============================================================

def estimate_ita(img_rgb, mask_bin):
    """Individual Typology Angle on peri-lesional (non-lesion) skin pixels."""
    skin_mask = (mask_bin == 0)
    if skin_mask.sum() < 50: # not enough peri-lesional skin pixels to estimate
        return None
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L = lab[...,0][skin_mask]
    b = lab[...,2][skin_mask]
    # standard LAB->ITA scaling (L in 0-100, b roughly -128..127 in OpenCV's 0-255 LAB it's shifted -- normalize)
    L_norm = L * (100.0/255.0)
    b_norm = b - 128.0
    ita = np.degrees(np.arctan2(L_norm.mean() - 50.0, b_norm.mean()))
    return float(ita)

def ita_to_fst_bucket(ita):
    if ita is None:
        return None
    if ita > 55: return 'I-II'# very light
    elif ita > 28: return 'III-IV'# medium
    else: return 'V-VI'# dark

cam_dense_fst = GradCAMPlusPlus(model=model, target_layers=model.get_cam_target_layers())

def _fst_process_row(row):
    img_id = str(row.get('image_id', row.name))
    true_idx = int(row['label_idx'])
    img_bgr = cv2.imread(row['image_path'])
    if img_bgr is None:
        return None
    img_bgr = dull_razor_hair_removal(img_bgr)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_rs = cv2.resize(img_rgb, (CFG['img_size'], CFG['img_size']))
    img_norm = img_rs.astype(np.float32) / 255.0
    img_t = torch.from_numpy(((img_norm - _mean_n) / _std_n).transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(img_t) / T_dense
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = int(probs.argmax())

    if pred_idx != true_idx:
        return None # TIxAI (and therefore this audit) is conditional on a correct prediction

    if not USE_SYNTHETIC_MASKS and img_id in masks_dict:
        mask = cv2.imread(masks_dict[img_id], cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (CFG['img_size'], CFG['img_size']))
        mask_bin = (mask > 127).astype(np.uint8)
        mask_source = 'GT'
    else:
        mask_bin = (make_synthetic_mask(CFG['img_size']) > 127).astype(np.uint8)
        mask_source = 'synthetic'

    ita = estimate_ita(img_rs, mask_bin)
    fst_bucket = ita_to_fst_bucket(ita)
    if fst_bucket is None:
        return None

    try:
        gcam = cam_dense_fst(input_tensor=img_t, targets=[ClassifierOutputTarget(pred_idx)])[0]
    except Exception:
        return None
    tixai_score = compute_tixai(gcam, mask_bin * 255)

    return {'image_id': img_id, 'true_class': row['label'], 'ita': ita,
            'fst_bucket': fst_bucket, 'tixai': tixai_score, 'mask_source': mask_source}


if TIXAI_USES_SYNTHETIC_MASKS and not CFG.get('allow_synthetic_masks_in_results', False):
    print(chr(0x26A0) + 'WARNING: ISIC2018 ground-truth masks are not attached -- this FST '
          'audit would run on fabricated synthetic masks, and any skin-tone-related '
          'TIxAI difference found this way is an artifact of the audit, not a finding '
          'about the model. Skipping.')
    fst_df = pd.DataFrame()
else:
    print('\nComputing Per-FST Explanation-Reliability Audit (NEW gap, not in original literature)...')
    fst_df = run_resumable_loop(
        test_df, CFG['out_dir'] / 'fst_tixai_results.csv', _fst_process_row,
        id_col='image_id', desc='FST-TIxAI'
    )

    if len(fst_df) > 10:
        fst_df['tixai'] = fst_df['tixai'].astype(float)
        fst_groups = {b: fst_df[fst_df['fst_bucket']==b]['tixai'].values
                      for b in ['I-II','III-IV','V-VI'] if (fst_df['fst_bucket']==b).sum() >= 5}
        print('\nPer-FST TIxAI (estimated via ITA proxy -- not clinical-grade FST labels):')
        for b, vals in fst_groups.items():
            print(f'FST {b:8s}: n={len(vals):4d} median TIxAI={np.median(vals):.4f} mean={np.mean(vals):.4f}')

        if len(fst_groups) >= 2:
            H_fst, p_fst = kruskal(*fst_groups.values())
            print(f'\nKruskal-Wallis across FST buckets: H={H_fst:.4f}, p={p_fst:.2e}')
            print(f'{"REJECT H0 -- explanation reliability differs by skin tone" if p_fst < 0.05 else "FAIL TO REJECT H0"}')

        fst_stats_df = pd.DataFrame([
            {'fst_bucket': b, 'n': len(v), 'median_tixai': np.median(v), 'mean_tixai': np.mean(v)}
            for b, v in fst_groups.items()
        ])
        fst_stats_df.to_csv(CFG['out_dir'] / 'Table_per_fst_tixai.csv', index=False)
        print('\nWARNING: CAVEAT: FST buckets here are estimated from ITA on peri-lesional pixels, not a')
        print('clinically-assigned Fitzpatrick score. Treat this as a directional signal; for a')
        print('publication-grade claim, validate against PAD-UFES-20 (CELL 9c) which has')
        print('self-reported skin-tone metadata, or against true Fitzpatrick17k labels.')
    else:
        print('WARNING: Too few samples with estimable ITA -- skipping FST stratification.')
        fst_stats_df = pd.DataFrame()



## CELL 10c — GAP-6: Temporal/Stochastic Saliency Stability (STS) (NEW — Novelty Add: is the explanation reproducible?)


In [ ]:
# ============================================================
# CELL 10c — Temporal/Stochastic Saliency Stability (STS)
# TIxAI (Cell 10) asks "is the explanation LOCALIZED correctly".
# STS asks a different question: "is the explanation REPRODUCIBLE at
# all?" -- it runs GradCAM++ STS_T times per image with MC-Dropout
# active and measures how much the saliency map itself moves across
# repeated stochastic passes on the SAME input, holding the explained
# class fixed. A clinician should not trust an explanation that fails
# either test. Unlike TIxAI, this needs no lesion segmentation mask.
# ============================================================
STS_T = 15
N_STS_EVAL = min(100, len(test_df))
sts_sample_df = test_df.sample(N_STS_EVAL, random_state=SEED)

model_sts = copy.deepcopy(model)
model_sts.use_mc_dropout = True
for _m in model_sts.modules():
    if isinstance(_m, nn.Dropout):
        _m.train()
model_sts.backbone.eval()
cam_sts = GradCAMPlusPlus(model=model_sts, target_layers=model_sts.get_cam_target_layers())


def _sts_process_row(row):
    img_bgr = cv2.imread(row['image_path'])
    if img_bgr is None:
        return None
    img_bgr = dull_razor_hair_removal(img_bgr)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_norm = cv2.resize(img_rgb, (CFG['img_size'], CFG['img_size'])).astype(np.float32) / 255.0
    img_t = torch.from_numpy(((img_norm - _mean_n) / _std_n).transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred_idx = int(torch.softmax(model(img_t), dim=1).argmax(1).item())

    cams = []
    for _ in range(STS_T):
        for _m in model_sts.modules():
            if isinstance(_m, nn.Dropout):
                _m.train()
        try:
            gcam = cam_sts(input_tensor=img_t, targets=[ClassifierOutputTarget(pred_idx)])[0]
        except Exception:
            continue
        cams.append(gcam.flatten())
    if len(cams) < 2:
        return None

    cams = np.stack(cams)
    corr_matrix = np.corrcoef(cams)
    if not np.isfinite(corr_matrix).all():
        return None
    upper = corr_matrix[np.triu_indices(len(cams), k=1)]
    sts_score = float(np.mean(upper))

    return {'image_id': str(row.get('image_id', row.name)), 'true_class': row['label'],
            'pred_class': CLASSES[pred_idx], 'correct': pred_idx == int(row['label_idx']),
            'sts': sts_score, 'n_valid_runs': len(cams)}


print('\nComputing Temporal/Stochastic Saliency Stability (STS)...')
sts_df = run_resumable_loop(
    sts_sample_df, CFG['out_dir'] / 'sts_results.csv', _sts_process_row,
    id_col='image_id', desc='STS'
)
sts_df = sts_df.dropna(subset=['sts'])
sts_df['sts'] = sts_df['sts'].astype(float)

print(f'\nSTS computed for {len(sts_df)} images (T={STS_T} stochastic passes each)')
print(f'Median STS: {sts_df["sts"].median():.4f} | Mean STS: {sts_df["sts"].mean():.4f}')

UNSTABLE_THRESHOLD = 0.5
n_unstable = (sts_df['sts'] < UNSTABLE_THRESHOLD).sum()
print(f'{n_unstable}/{len(sts_df)} ({100*n_unstable/len(sts_df):.1f}%) images have STS < {UNSTABLE_THRESHOLD} '
      f'-- explanation location is not reproducible across stochastic passes, regardless of TIxAI/accuracy.')

_merged_sts_tixai = sts_df.assign(image_id=lambda d: d['image_id'].astype(str)).merge(
    correct_df[['image_id', 'tixai']].assign(image_id=lambda d: d['image_id'].astype(str)),
    on='image_id', how='inner'
)
if len(_merged_sts_tixai) >= 5:
    rho, p_val = spearmanr(_merged_sts_tixai['sts'], _merged_sts_tixai['tixai'])
    print(f'\nSpearman(STS, TIxAI) on {len(_merged_sts_tixai)} shared images: rho={rho:.3f}, p={p_val:.4e}')
    print('Low |rho| -> STS is a genuinely distinct trust signal, not redundant with TIxAI.'
          if abs(rho) < 0.3 else
          'Moderate/high |rho| -> STS and TIxAI are correlated; report both but note the overlap.')

fig, ax = plt.subplots(figsize=(9, 6))
ax.hist(sts_df['sts'].values, bins=25, color='#00BCD4', edgecolor='white', alpha=0.85)
ax.axvline(UNSTABLE_THRESHOLD, color='red', linestyle='--', label=f'Unstable threshold ({UNSTABLE_THRESHOLD})')
ax.set_xlabel('Saliency Temporal Stability (STS) score')
ax.set_ylabel('Count')
ax.set_title('Distribution of Explanation Stability Across Stochastic (MC-Dropout) Passes\n'
             'Low STS = same model, same image, different explanation each time', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'Figure_STS_distribution.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise

sts_df.to_csv(CFG['out_dir'] / 'Table_STS_per_image.csv', index=False)
print('\nSTS results saved: Table_STS_per_image.csv, Figure_STS_distribution.png')



## CELL 11 — GAP-1 Statistical Tests + Violin Plots (unchanged)


In [ ]:
# ============================================================
# CELL 11 — Statistical Analysis: Kruskal-Wallis + Mann-Whitney
# ============================================================
correct_df = tixai_df[tixai_df['correct'] == True].dropna(subset=['tixai'])
correct_df['tixai'] = correct_df['tixai'].astype(float)
class_scores = {cls: correct_df[correct_df['true_class']==cls]['tixai'].values
                for cls in CLASSES if len(correct_df[correct_df['true_class']==cls]) > 0}

groups = [v for v in class_scores.values() if len(v) >= 5]
H_stat, p_kruskal = kruskal(*groups)
print(f'Kruskal-Wallis H-test:')
print(f'H = {H_stat:.4f}, df = {len(groups)-1}, p = {p_kruskal:.2e}')
print(f'Result: {"REJECT H0" if p_kruskal < 0.05 else "FAIL TO REJECT H0"}\n')

from statsmodels.stats.multitest import multipletests
print(f'Pairwise Mann-Whitney U (with Holm-Bonferroni correction):')

pairwise_results = []
p_values_raw = []
cls_list = list(class_scores.keys())
for i in range(len(cls_list)):
    for j in range(i+1, len(cls_list)):
        c1, c2 = cls_list[i], cls_list[j]
        s1, s2 = class_scores[c1], class_scores[c2]
        if len(s1) < 3 or len(s2) < 3:
            continue
        U, p = mannwhitneyu(s1, s2, alternative='two-sided')
        r = 1 - 2*U / (len(s1)*len(s2))
        p_values_raw.append(p)
        pairwise_results.append({
            'class_1': c1, 'class_2': c2, 'U': U, 'p_value_raw': p,
            'rank_biserial_r': abs(r),
            'effect_size': 'large' if abs(r)>0.5 else 'medium' if abs(r)>0.3 else 'small'
        })

reject, pvals_corrected, _, _ = multipletests(p_values_raw, alpha=0.05, method='holm')
for i, res in enumerate(pairwise_results):
    res['p_value_corr'] = pvals_corrected[i]
    res['significant'] = reject[i]


pairwise_df = pd.DataFrame(pairwise_results)
pairwise_df.to_csv(CFG['out_dir'] / 'Table_pairwise_mwu.csv', index=False)

primary_contrasts = [('NV','DF'), ('NV','VASC'), ('MEL','DF'), ('MEL','VASC')]
for c1, c2 in primary_contrasts:
    row = pairwise_df[(pairwise_df['class_1'].isin([c1,c2])) & (pairwise_df['class_2'].isin([c1,c2]))]
    if len(row) > 0:
        r = row.iloc[0]
        print(f'{c1} vs {c2}: p_corr={r["p_value_corr"]:.2e}, r={r["rank_biserial_r"]:.3f} ({r["effect_size"]})')

cls_count = [len(train_df[train_df['label']==cls]) for cls in cls_list]
cls_tixai = [np.median(class_scores[cls]) for cls in cls_list]
rho, p_spearman = spearmanr(cls_count, cls_tixai)
print(f'\nSpearman ρ (class frequency vs median TIxAI):')
print(f'ρ = {rho:.4f}, p = {p_spearman:.4f}')

fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor('#0D1117')
ax.set_facecolor('#161B22')
ordered_classes = sorted(cls_list, key=lambda c: np.median(class_scores[c]), reverse=True)
plot_data = [class_scores[c] for c in ordered_classes]
colors = [CLASS_COLORS[c] for c in ordered_classes]

vp = ax.violinplot(plot_data, positions=range(len(ordered_classes)), showmedians=True, showextrema=True)
for i, (body, color) in enumerate(zip(vp['bodies'], colors)):
    body.set_facecolor(color); body.set_alpha(0.8)
vp['cmedians'].set_color('white'); vp['cmedians'].set_linewidth(2.5)

ax.set_xticks(range(len(ordered_classes)))
ax.set_xticklabels([
    f'{c}\n(n={len(class_scores[c])}\nmed={np.median(class_scores[c]):.3f})'
    for c in ordered_classes
], fontsize=10, color='white')
ax.set_ylabel('TIxAI Score (Explanation Trustworthiness)', fontsize=12, color='white')
ax.set_title(
    'GAP #1: Per-Class TIxAI Distribution\n'
    'Rare classes (DF, VASC) produce systematically LESS trustworthy explanations\n'
    f'(Kruskal-Wallis H={H_stat:.1f}, p={p_kruskal:.2e})',
    fontsize=14, fontweight='bold', color='white', pad=15
)
ax.axhline(0.65, color='#4CAF50', linestyle='--', alpha=0.7, label='High Trust (>0.65)')
ax.axhline(0.45, color='#FF9800', linestyle='--', alpha=0.7, label='Moderate Trust (>0.45)')
ax.tick_params(colors='white')
ax.spines['bottom'].set_color('#30363D'); ax.spines['left'].set_color('#30363D')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(fontsize=10, facecolor='#161B22', labelcolor='white')
ax.grid(axis='y', alpha=0.2, color='white')
ymax = max(max(s) for s in plot_data) + 0.05
ax.set_ylim([-0.05, ymax + 0.1])

plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'Figure4_tixai_violin_plots.png', dpi=300, bbox_inches='tight', facecolor='#0D1117')
plt.show()
plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise
print('\nFigure 4 (TIxAI violin plots) saved.')



## CELL 11b — TIxAI vs Lesion Size Stratification (NEW — Novelty Add: is trust worse for small lesions?)


In [ ]:
# ============================================================
# CELL 11b — TIxAI vs Lesion Size Stratification
# HAM10000 lesions vary hugely in image-area fraction. Small lesions are
# harder to segment/attend to correctly -- this tests whether GradCAM
# localization quality (TIxAI) is itself worse for small lesions, which
# would be a direct, actionable warning for clinicians: don't trust the
# highlighted region as much on a small/early-stage lesion.
# ============================================================
assert (not TIXAI_USES_SYNTHETIC_MASKS) or CFG.get('allow_synthetic_masks_in_results', False), (
    'Refusing to compute lesion-size stratification on synthetic (non-GT) masks -- lesion area '
    "would be meaningless noise from the synthetic-mask generator, not real lesion size. Attach the "
    "ISIC2018 Task 1 dataset, or set CFG['allow_synthetic_masks_in_results']=True to proceed anyway for a demo run."
)


def compute_lesion_size_bucket(mask):
    mask_bin = (mask > 127).astype(np.float32)
    area = mask_bin.sum() / mask_bin.size
    if area < 0.05:
        return 'small'
    elif area < 0.15:
        return 'medium'
    return 'large'


def _lesion_size_row(row):
    img_id = str(row['image_id'])
    if img_id not in masks_dict:
        return None
    mask = cv2.imread(masks_dict[img_id], cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (CFG['img_size'], CFG['img_size']))
    return {'image_id': img_id, 'lesion_size_bucket': compute_lesion_size_bucket(mask)}


lesion_size_items = correct_df[['image_id']].drop_duplicates().reset_index(drop=True)
lesion_size_df = run_resumable_loop(
    lesion_size_items, CFG['out_dir'] / 'lesion_size_buckets.csv', _lesion_size_row,
    id_col='image_id', desc='LesionSize'
)

size_tixai_df = correct_df.assign(image_id=lambda d: d['image_id'].astype(str)).merge(
    lesion_size_df.assign(image_id=lambda d: d['image_id'].astype(str)),
    on='image_id', how='inner'
)

size_groups = {b: size_tixai_df[size_tixai_df['lesion_size_bucket'] == b]['tixai'].values
               for b in ['small', 'medium', 'large']}
size_groups = {b: v for b, v in size_groups.items() if len(v) >= 5}

print('\nTIxAI by lesion-size bucket:')
for b, v in size_groups.items():
    print(f'{b:7s}: n={len(v):4d}  median TIxAI={np.median(v):.4f}  mean={np.mean(v):.4f}')

if len(size_groups) >= 2:
    H_size, p_size = kruskal(*size_groups.values())
    print(f'\nKruskal-Wallis (TIxAI across lesion-size buckets): H={H_size:.4f}, p={p_size:.2e}')
    print('Result: ' + ('REJECT H0 -- lesion size significantly affects explanation trust'
                         if p_size < 0.05 else
                         'FAIL TO REJECT H0 -- no significant size effect detected'))
else:
    print('\nNot enough buckets with n>=5 samples to run Kruskal-Wallis.')
    H_size, p_size = float('nan'), float('nan')

order = [b for b in ['small', 'medium', 'large'] if b in size_groups]
if order:
    fig, ax = plt.subplots(figsize=(9, 6))
    bucket_colors = {'small': '#F44336', 'medium': '#FF9800', 'large': '#4CAF50'}
    bp = ax.boxplot([size_groups[b] for b in order], labels=order, patch_artist=True, showmeans=True)
    for patch, b in zip(bp['boxes'], order):
        patch.set_facecolor(bucket_colors[b])
        patch.set_alpha(0.7)
    ax.set_ylabel('TIxAI Score')
    ax.set_xlabel('Lesion size bucket (fraction of image area)')
    ax.set_title(f'Explanation Trust vs Lesion Size\n(Kruskal-Wallis H={H_size:.2f}, p={p_size:.2e})', fontweight='bold')
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(CFG['out_dir'] / 'Figure_TIxAI_vs_lesion_size.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise
else:
    print('\nNo lesion-size buckets with n>=5 samples -- skipping plot.')

lesion_size_summary_rows = [{'bucket': b, 'n': len(v), 'median_tixai': np.median(v), 'mean_tixai': np.mean(v)}
                             for b, v in size_groups.items()]
lesion_size_summary_rows.append({'bucket': 'kruskal_H_p', 'n': None, 'median_tixai': H_size, 'mean_tixai': p_size})
pd.DataFrame(lesion_size_summary_rows).to_csv(CFG['out_dir'] / 'Table_lesion_size_tixai.csv', index=False)
print('\nLesion-size stratification results saved: Table_lesion_size_tixai.csv, Figure_TIxAI_vs_lesion_size.png')


## CELL 11c — Explanation Reliability Calibration Curve (ERCC) (NEW — Novelty Add)

In [ ]:
# ============================================================
# CELL 11c — Explanation Reliability Calibration Curve (ERCC)
# (NEW — Novelty Add: isotonic regression mapping confidence -> expected
# TIxAI. No prior work calibrates SPATIAL saliency quality against
# confidence -- Guo et al. 2017 and similar calibrate the probability
# itself, not whether the explanation can be trusted.)
#
# Also answers a question CELL 10's own design doc flags directly: does
# confidence predict TIxAI well enough to make TIxAI redundant? If rho is
# high, confidence is a cheap proxy for explanation reliability. If not,
# TIxAI provides genuinely independent information.
#
# FIX (Executive Summary bug, audit 2026-07-05): the Spearman rho below
# used to be stored in a bare `rho`, which later cells (e.g. GAP-10's own
# Spearman test) overwrite before CELL 18 reads it. Renamed to `rho_ercc`
# so CELL 18 reads the real ERCC statistic.
# ============================================================
from sklearn.isotonic import IsotonicRegression

ercc_df = correct_df.dropna(subset=['tixai', 'confidence']).copy()
conf_arr = ercc_df['confidence'].values.astype(float)
tixai_arr = ercc_df['tixai'].values.astype(float)

if len(ercc_df) < 20:
    print(f'ERCC: only {len(ercc_df)} samples with both confidence and TIxAI -- skipping '
          f'(need at least 20 for a meaningful isotonic fit).')
else:
    order = np.argsort(conf_arr)
    conf_sorted, tixai_sorted = conf_arr[order], tixai_arr[order]

    ir = IsotonicRegression(out_of_bounds='clip')
    ir.fit(conf_sorted, tixai_sorted)
    tixai_calibrated = ir.predict(conf_sorted)

    rho_ercc, p_ercc = spearmanr(conf_arr, tixai_arr)
    print(f'Spearman(confidence, TIxAI): rho={rho_ercc:.3f}, p={p_ercc:.4f}')
    if p_ercc < 0.01 and rho_ercc > 0.5:
        print('-> Confidence is a STRONG proxy for TIxAI here -- report this explicitly; it '
              'weakens (does not eliminate) the case for TIxAI as an independent signal.')
    elif p_ercc < 0.01:
        print('-> Confidence correlates with TIxAI but weakly -- both signals add information.')
    else:
        print('-> Confidence does NOT reliably predict explanation quality -- TIxAI provides '
              'genuinely independent information beyond what confidence already tells you.')

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(conf_sorted, tixai_sorted, alpha=0.15, s=8, color='gray',
               label='Observed (correct predictions)')
    ax.plot(conf_sorted, tixai_calibrated, 'b-', lw=2, label='Isotonic fit (ERCC)')
    ax.set_xlabel('Model confidence (calibrated softmax max)')
    ax.set_ylabel('TIxAI score (explanation reliability)')
    ax.set_title(f'Explanation Reliability Calibration Curve (ERCC)\nSpearman rho={rho_ercc:.3f}, p={p_ercc:.2e}')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(CFG['out_dir'] / 'Figure_ERCC.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise

    pd.DataFrame({'confidence': conf_sorted, 'tixai_observed': tixai_sorted,
                  'tixai_calibrated': tixai_calibrated}).to_csv(
        CFG['out_dir'] / 'Table_ERCC.csv', index=False)
    print('ERCC figure and table saved.')



## CELL 12 — GAP-4: Multi-XAI Disagreement Index (MODIFIED: resumable, faster SHAP, full test set)


In [ ]:
# ============================================================
# CELL 12 — GAP #4: Multi-XAI Disagreement Index (XDI)
# FIXES applied:
# - Switched shap.DeepExplainer -> shap.GradientExplainer: faster and
# much lighter on GPU memory for CNNs, which is what made running
# XDI on the FULL test set (instead of a 200-image sample) feasible.
# - Resumable via run_resumable_loop -- safe to stop/restart.
# - N_XDI_SAMPLES is now configurable; set to len(test_df) for the
# full-coverage run once you've confirmed GradientExplainer fits
# your time budget on a small sample first.
# - FIX (Executive Summary bug, audit 2026-07-05): the MWU p-value below
# used to be stored in a bare `p`, which CELL 17's saliency-geometry loop
# later overwrites (same variable name, different test) before CELL 18
# reads it. Renamed to `p_xdi` so CELL 18 reads the real XDI statistic.
# ============================================================
import shap

cam_dense = GradCAMPlusPlus(model=model, target_layers=model.get_cam_target_layers())

np.random.seed(SEED)
bg_indices = np.random.choice(len(train_ds), min(50, len(train_ds)), replace=False)
background = torch.cat([train_ds[i][0].unsqueeze(0) for i in bg_indices]).to(DEVICE)
# GradientExplainer (FIX): far lower memory + faster than DeepExplainer for CNNs,
# and doesn't require patching every nonlinearity the way DeepExplainer does.
explainer = shap.GradientExplainer(model, background)


def compute_saliency_maps(img_t, pred_idx):
    """Expensive part (Grad-CAM++ + SHAP) -- computed ONCE per image."""
    targets = [ClassifierOutputTarget(pred_idx)]
    gradcam_map = cam_dense(input_tensor=img_t, targets=targets)[0]

    try:
        shap_values = explainer.shap_values(img_t, nsamples=50)
        # FIX: shap.GradientExplainer (unlike the old DeepExplainer) returns
        # a single ndarray with a trailing per-class axis for multi-output
        # models -- shape (batch, C, H, W, n_classes) -- rather than a
        # python list of per-class arrays. The `else` branch below used to
        # just strip the batch dim (shap_values[0]), which still left the
        # (C, H, W, n_classes) per-class axis in place; summing over axis=0
        # (channels) then produced a (H, W, n_classes) map instead of
        # (H, W), which failed to broadcast against gradcam_map's (H, W)
        # shape 100% of the time (confirmed: "operands could not be
        # broadcast together with shapes (224,224) (224,224,7)").
        if isinstance(shap_values, list):
            shap_img = shap_values[pred_idx][0]  # (C, H, W)
        elif shap_values.ndim == 5:
            shap_img = shap_values[0, ..., pred_idx]  # select predicted class -> (C, H, W)
        else:
            shap_img = shap_values[0]  # already single-output -> (C, H, W)
        shap_map = np.abs(shap_img).sum(axis=0)
        shap_map = (shap_map - shap_map.min()) / (shap_map.max() - shap_map.min() + 1e-8)
    except Exception:
        return gradcam_map, None
    return gradcam_map, shap_map


def xdi_at_threshold(gradcam_map, shap_map, threshold=0.5):
    """Cheap part: binarize the (already-computed) maps at a given
    top-k% mass threshold and compute 1-IoU. Split out from
    compute_saliency_maps so the multi-threshold robustness check below
    (k=25/50/75%, per the reviewer-risk analysis) does not re-run SHAP --
    by far the expensive step -- three times per image.
    """
    if shap_map is None:
        return None
    cam_thresh = np.percentile(gradcam_map, (1 - threshold) * 100)
    shap_thresh = np.percentile(shap_map, (1 - threshold) * 100)
    cam_bin = (gradcam_map >= cam_thresh).astype(np.uint8)
    shap_bin = (shap_map >= shap_thresh).astype(np.uint8)

    if cam_bin.shape != shap_bin.shape:
        shap_bin = cv2.resize(shap_bin.astype(np.float32), (cam_bin.shape[1], cam_bin.shape[0]))
        shap_bin = (shap_bin > 0.5).astype(np.uint8)

    intersection = (cam_bin & shap_bin).sum()
    union = (cam_bin | shap_bin).sum()
    iou = intersection / (union + 1e-8)
    return 1.0 - float(iou)


XDI_THRESHOLDS = (0.25, 0.50, 0.75)  # NEW: robustness sweep, not just k=50%

def compute_xdi(img_t, pred_idx, threshold=0.5):
    """Kept for backward compatibility with any external caller expecting
    a single-threshold API; the loop below calls compute_saliency_maps +
    xdi_at_threshold directly so it only pays the SHAP cost once per image
    while still reporting all three thresholds.
    """
    gradcam_map, shap_map = compute_saliency_maps(img_t, pred_idx)
    return xdi_at_threshold(gradcam_map, shap_map, threshold), gradcam_map, shap_map


# NEW: full test set by default (was hardcoded to 200). GradientExplainer +
# the resumable loop's time-budget guard make this safe -- it'll just pick
# up where it left off if the session ends mid-run.
N_XDI_SAMPLES = len(test_df) # was: min(200, len(test_df))
xdi_sample_df = test_df.sample(min(N_XDI_SAMPLES, len(test_df)), random_state=SEED)


def _xdi_process_row(row):
    img_id = str(row.get('image_id', row.name))
    true_cls = row['label']

    img_bgr = cv2.imread(row['image_path'])
    if img_bgr is None:
        return None
    img_bgr = dull_razor_hair_removal(img_bgr)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_norm = cv2.resize(img_rgb, (224,224)).astype(np.float32)/255.0
    img_t = torch.from_numpy(((img_norm-_mean_n)/_std_n).transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(img_t) / T_dense
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_idx = int(probs.argmax())
        pred_cls = CLASSES[pred_idx]
        conf = float(probs.max())

    gcam, shap_m = compute_saliency_maps(img_t, pred_idx)
    if shap_m is None:
        return None

    xdi_by_k = {f'xdi_{int(k*100)}': xdi_at_threshold(gcam, shap_m, k) for k in XDI_THRESHOLDS}
    result = {'image_id': img_id, 'true_class': true_cls, 'pred_class': pred_cls,
              'confidence': conf, 'correct': pred_idx == int(row['label_idx'])}
    result.update(xdi_by_k)
    result['xdi'] = xdi_by_k['xdi_50']  # keep 'xdi' = k=50% for CELL 12b and downstream compatibility
    return result


print(f'\nComputing XDI for up to {N_XDI_SAMPLES} test samples (resumable)...')
xdi_df = run_resumable_loop(
    xdi_sample_df, CFG['out_dir'] / 'xdi_results.csv', _xdi_process_row,
    id_col='image_id', desc='XDI', save_every=25 # SHAP is slow -- save more often
)

print(f'\nXDI Results:')
# FIX: if every row's process_fn raised (run_resumable_loop already prints
# a loud warning for this case), xdi_df has zero rows and no 'xdi' column
# at all, so xdi_df["xdi"].mean() used to fail with a bare, uninformative
# KeyError('xdi') instead of pointing at the real cause.
if 'xdi' not in xdi_df.columns or len(xdi_df) == 0:
    print('No successful XDI rows were computed -- see the "row(s) raised an exception"')
    print('warning above for the actual failure. Skipping XDI summary statistics.')
else:
    print(f'Mean XDI: {xdi_df["xdi"].mean():.4f} ± {xdi_df["xdi"].std():.4f}')
    print(f'Correct preds mean XDI: {xdi_df[xdi_df.correct]["xdi"].mean():.4f}')
    print(f'Incorrect preds mean XDI: {xdi_df[~xdi_df.correct]["xdi"].mean():.4f}')

    correct_xdi = xdi_df[xdi_df.correct]['xdi'].dropna().values
    incorrect_xdi = xdi_df[~xdi_df.correct]['xdi'].dropna().values
    # FIX (2026-07-05): always bind p_xdi so CELL 18 never falls to 'n/a'
    # unexpectedly. It stays NaN only in the degenerate too-few-samples case,
    # which CELL 18 then reports explicitly rather than silently as 'n/a'.
    p_xdi = float('nan')
    if len(correct_xdi) > 5 and len(incorrect_xdi) > 5:
        U, p_xdi = mannwhitneyu(correct_xdi, incorrect_xdi, alternative='less')
        print(f'\nMWU test (correct XDI < incorrect XDI): U={U:.0f}, p={p_xdi:.4f}')
        print(f'-> XAI disagreement IS a signal of unreliability: {"YES" if p_xdi < 0.05 else "NO"}')

    # NEW: threshold-robustness check (k=25/50/75%) -- a single-threshold
    # result is a fair reviewer objection ("does this hold at other
    # binarization levels, or did you pick the one that worked?").
    print('\nXDI threshold-robustness check (k = 25%, 50%, 75%):')
    threshold_robustness = []
    for k in XDI_THRESHOLDS:
        col = f'xdi_{int(k*100)}'
        if col not in xdi_df.columns:
            continue
        c_vals = xdi_df[xdi_df.correct][col].dropna().values
        i_vals = xdi_df[~xdi_df.correct][col].dropna().values
        if len(c_vals) <= 5 or len(i_vals) <= 5:
            print(f'k={k:.0%}: insufficient samples, skipped')
            continue
        U_k, p_k = mannwhitneyu(c_vals, i_vals, alternative='less')
        r_k = 1 - (2 * U_k) / (len(c_vals) * len(i_vals))  # rank-biserial effect size
        sig = p_k < 0.01
        threshold_robustness.append({'k': k, 'p_value': p_k, 'rank_biserial_r': r_k, 'significant_p<0.01': sig})
        print(f'k={k:.0%}: Mann-Whitney p={p_k:.4f}, rank-biserial r={r_k:.3f}, '
              f'{"significant" if sig else "NOT significant"}')
    xdi_robustness_df = pd.DataFrame(threshold_robustness)
    xdi_robustness_df.to_csv(CFG['out_dir'] / 'Table_XDI_threshold_robustness.csv', index=False)
    if len(threshold_robustness) == len(XDI_THRESHOLDS):
        all_sig = all(r['significant_p<0.01'] for r in threshold_robustness)
        print(f'\n-> XDI separation holds at ALL three thresholds: '
              f'{"YES" if all_sig else "NO -- report as threshold-sensitive, do not claim a single robust effect"}')



## CELL 12b — XDI vs Uncertainty-Deferral Baseline Comparison (NEW — Novelty Add #2)


In [ ]:
# ============================================================
# CELL 12b — Selective-accuracy/coverage curve: XDI vs MC-Dropout
# entropy vs raw confidence as deferral signals.
#
# NEW: This is the "missing experiment" your audit's Reviewer-#2
# simulation flagged: XDI is only a useful CONTRIBUTION if it beats
# (or meaningfully complements) cheaper uncertainty signals at
# selecting which predictions to defer to a clinician.
# ============================================================

def mc_dropout_predict_single(model, img_t, T=10):
    model_mc_local = copy.deepcopy(model)
    model_mc_local.use_mc_dropout = True
    for _m in model_mc_local.modules():
        if isinstance(_m, nn.Dropout):
            _m.train()
    model_mc_local.backbone.eval()
    preds = []
    with torch.no_grad():
        for _ in range(T):
            out = model_mc_local(img_t)
            preds.append(torch.softmax(out, dim=1).cpu().numpy())
    preds = np.stack(preds)
    mean_p = preds.mean(axis=0)
    entropy = -np.sum(mean_p * np.log(mean_p + 1e-9), axis=1)
    return float(entropy[0])


_ckpt12b = load_ckpt('cell12b_entropy')
if _ckpt12b is not None:
    xdi_entropy_df = _ckpt12b
else:
    # Build ONE shared MC-dropout model (avoid rebuilding per-row -- expensive)
    model_mc_shared = copy.deepcopy(model)
    model_mc_shared.use_mc_dropout = True
    for _m in model_mc_shared.modules():
        if isinstance(_m, nn.Dropout):
            _m.train()
    model_mc_shared.backbone.eval()

    entropy_rows = []
    for idx, row in tqdm(xdi_df.iterrows(), total=len(xdi_df), desc='MC-entropy for baseline'):
        img_bgr = cv2.imread(test_df[test_df['image_id'].astype(str) == str(row['image_id'])].iloc[0]['image_path']) \
                  if (test_df['image_id'].astype(str) == str(row['image_id'])).any() else None
        if img_bgr is None:
            continue
        img_bgr = dull_razor_hair_removal(img_bgr)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_norm = cv2.resize(img_rgb, (224,224)).astype(np.float32)/255.0
        img_t = torch.from_numpy(((img_norm-_mean_n)/_std_n).transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            preds = []
            for _ in range(10):
                out = model_mc_shared(img_t)
                preds.append(torch.softmax(out, dim=1).cpu().numpy())
        preds = np.stack(preds)
        mean_p = preds.mean(axis=0)
        ent = float(-np.sum(mean_p * np.log(mean_p + 1e-9), axis=1)[0])
        entropy_rows.append({'image_id': row['image_id'], 'mc_entropy': ent})

    xdi_entropy_df = xdi_df.merge(pd.DataFrame(entropy_rows), on='image_id', how='inner')
    save_ckpt('cell12b_entropy', xdi_entropy_df)


def selective_curve(scores, correct, ascending=True):
    """scores = unreliability signal (higher = less trust). Sweep retained-coverage fraction."""
    order = np.argsort(scores if ascending else [-s for s in scores])
    correct_sorted = np.array(correct)[order]
    coverages = np.linspace(0.1, 1.0, 19)
    accs = [correct_sorted[:max(1, int(c*len(correct_sorted)))].mean() for c in coverages]
    return coverages, accs

correct_arr = xdi_entropy_df['correct'].values
cov_xdi, acc_xdi = selective_curve(xdi_entropy_df['xdi'].values, correct_arr)
cov_ent, acc_ent = selective_curve(xdi_entropy_df['mc_entropy'].values, correct_arr)
cov_conf, acc_conf = selective_curve((1 - xdi_entropy_df['confidence'].values), correct_arr)

fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(cov_xdi, acc_xdi, 'o-', label='XDI deferral (novel)', color='#F44336', linewidth=2)
ax.plot(cov_ent, acc_ent, 's-', label='MC-Dropout entropy deferral', color='#2196F3', linewidth=2)
ax.plot(cov_conf, acc_conf, '^-', label='Confidence deferral (baseline)', color='#9E9E9E', linewidth=2)
ax.set_xlabel('Coverage (fraction of cases AI handles)', fontsize=12)
ax.set_ylabel('Accuracy on retained (non-deferred) cases', fontsize=12)
ax.set_title('GAP #4 validation: Does XDI beat uncertainty-based deferral?\n'
             '(Higher curve at a given coverage = better deferral signal)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'Figure_XDI_vs_uncertainty_deferral.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise

# AUC-under-the-selective-curve as a single summary number per signal
auc_xdi = np.trapz(acc_xdi, cov_xdi)
auc_ent = np.trapz(acc_ent, cov_ent)
auc_conf = np.trapz(acc_conf, cov_conf)
print('\nArea under selective-accuracy curve (higher = better deferral signal):')
print(f'XDI: {auc_xdi:.4f}')
print(f'MC-entropy: {auc_ent:.4f}')
print(f'Confidence: {auc_conf:.4f}')
xdi_wins = auc_xdi > max(auc_ent, auc_conf)
if xdi_wins:
    verdict = 'YES'
else:
    verdict = ("NO -- demote XDI to a diagnostic visualization, not a deferral signal "
               "(per the audit's own H2-false contingency)")
print('\n-> XDI beats both baselines: '+ verdict)
pd.DataFrame([
    {'signal':'XDI','auc_selective':auc_xdi},
    {'signal':'MC-entropy','auc_selective':auc_ent},
    {'signal':'Confidence','auc_selective':auc_conf},
]).to_csv(CFG['out_dir'] / 'Table_XDI_vs_baselines.csv', index=False)



## CELL 13 — GAP-3: Confusion Saliency Matrix (MODIFIED: resumable, batched CAM caching)


In [ ]:
# ============================================================
# CELL 13 — GAP #3: Confusion Saliency Matrix (CSM)
# FIX: misclassified-sample CAMs are computed via a resumable loop and
# cached to disk as .npy files keyed by image_id, so a killed session
# doesn't lose the (often slow) Grad-CAM++ pass over the whole test set.
# ============================================================
cam_dense = GradCAMPlusPlus(model=model, target_layers=model.get_cam_target_layers())
model.eval()

CSM_CACHE_DIR = CFG['out_dir'] / 'csm_cam_cache'
CSM_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _csm_process_row(row):
    img_id = str(row.get('image_id', row.name))
    img_bgr = cv2.imread(row['image_path'])
    if img_bgr is None:
        return None
    img_bgr = dull_razor_hair_removal(img_bgr)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_norm = cv2.resize(img_rgb, (224,224)).astype(np.float32)/255.0
    img_t = torch.from_numpy(((img_norm-_mean_n)/_std_n).transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(img_t) / T_dense
        pred_idx = int(torch.softmax(logits,1).argmax(1).item())

    true_idx = int(row['label_idx'])
    if pred_idx == true_idx:
        return {'image_id': img_id, 'true_class': CLASSES[true_idx], 'pred_class': CLASSES[true_idx],
                'misclassified': False, 'cam_path': None}

    true_cls = CLASSES[true_idx]; pred_cls = CLASSES[pred_idx]
    try:
        targets = [ClassifierOutputTarget(pred_idx)]
        gcam = cam_dense(input_tensor=img_t, targets=targets)[0]
        cam_path = CSM_CACHE_DIR / f'{img_id}.npy'
        np.save(cam_path, gcam.astype(np.float16)) # fp16 halves disk usage for the cache
    except Exception:
        return None

    return {'image_id': img_id, 'true_class': true_cls, 'pred_class': pred_cls,
            'misclassified': True, 'cam_path': str(cam_path)}


print('Computing Confusion Saliency Matrix (resumable CAM cache)...')
csm_index_df = run_resumable_loop(
    test_df, CFG['out_dir'] / 'csm_index.csv', _csm_process_row,
    id_col='image_id', desc='CSM'
)

confusion_cams = defaultdict(list)
confusion_counts = defaultdict(int)
for _, r in csm_index_df[csm_index_df['misclassified'] == True].iterrows():
    if r['cam_path'] and Path(r['cam_path']).exists():
        gcam = np.load(r['cam_path']).astype(np.float32)
        confusion_cams[(r['true_class'], r['pred_class'])].append(gcam)
        confusion_counts[(r['true_class'], r['pred_class'])] += 1

print(f'\nCSM computed. Top confusion pairs:')
top_confusions = sorted(confusion_counts.items(), key=lambda x: x[1], reverse=True)[:10]
for (t, p), cnt in top_confusions:
    print(f'{t} -> {p}: {cnt} misclassified samples')

fig, axes = plt.subplots(len(CLASSES), len(CLASSES), figsize=(3*len(CLASSES), 3*len(CLASSES)))
fig.patch.set_facecolor('#0D1117')

for ri, true_cls in enumerate(CLASSES):
    for ci, pred_cls in enumerate(CLASSES):
        ax = axes[ri, ci]
        ax.set_facecolor('#161B22')
        if ri == ci:
            ax.text(0.5, 0.5, f'OK\n{true_cls}', ha='center', va='center',
                    color='#4CAF50', fontsize=11, fontweight='bold', transform=ax.transAxes)
        elif (true_cls, pred_cls) in confusion_cams and len(confusion_cams[(true_cls, pred_cls)]) >= 5:
            mean_cam = np.mean(confusion_cams[(true_cls, pred_cls)], axis=0)
            ax.imshow(mean_cam, cmap='jet', vmin=0, vmax=1)
            count = confusion_counts[(true_cls, pred_cls)]
            ax.set_title(f'n={count}', fontsize=7, color='white', pad=1)
        else:
            n_have = confusion_counts.get((true_cls, pred_cls), 0)
            label = '—' if n_have == 0 else 'n='+ str(n_have) + '(too few)'
            ax.text(0.5, 0.5, label, ha='center', va='center', color='gray', fontsize=9, transform=ax.transAxes)
        ax.axis('off')
        if ci == 0:
            ax.set_ylabel(true_cls, color=CLASS_COLORS.get(true_cls,'white'), fontsize=9,
                          fontweight='bold', rotation=0, labelpad=40, va='center')
            ax.yaxis.set_visible(True)
        if ri == 0:
            ax.set_title(pred_cls, color=CLASS_COLORS.get(pred_cls,'white'), fontsize=9, fontweight='bold', pad=5)

fig.text(0.5, 0.02, 'Predicted Class', ha='center', fontsize=13, color='white', fontweight='bold')
fig.text(0.01, 0.5, 'True Class', va='center', rotation='vertical', fontsize=13, color='white', fontweight='bold')
plt.suptitle(
    'GAP #3: Confusion Saliency Matrix\n'
    'Mean Grad-CAM++ maps when model confuses one class for another\n'
    '(Brighter = higher attention; Row=True class, Col=Predicted class)',
    fontsize=13, fontweight='bold', color='white', y=1.01
)
plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'Figure5_confusion_saliency_matrix.png', dpi=250, bbox_inches='tight', facecolor='#0D1117')
plt.show()
plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise
print('\nFigure 5 (Confusion Saliency Matrix) saved.')



## CELL 14 — GAP-2: Explanation Failure Prediction (MODIFIED: resumable feature extraction)


In [ ]:
# ============================================================
# CELL 14 — GAP #2: Explanation Failure Prediction
# FIX: the per-image embedding+MC-dropout extraction loop is now
# resumable (it's the slow part); the final CV-trained meta-model
# is cheap and just reruns on whatever feature rows exist.
# ============================================================
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report as cr

correct_df = tixai_df[tixai_df['correct'] == True].dropna(subset=['tixai'])
correct_df['tixai'] = correct_df['tixai'].astype(float)

embeddings = {}
def hook_fn(module, input, output):
    embeddings['feat'] = output.detach().cpu().numpy()

hook = model.backbone.features.denseblock4.register_forward_hook(hook_fn)

model_mc = copy.deepcopy(model)
model_mc.use_mc_dropout = True
for _m in model_mc.modules():
    if isinstance(_m, nn.Dropout):
        _m.train()
model_mc.backbone.eval()

def mc_dropout_predict(model_mc, img_t, T=10):
    preds = []
    with torch.no_grad():
        for _ in range(T):
            out = model_mc(img_t)
            preds.append(torch.softmax(out, dim=1).cpu().numpy())
    preds = np.stack(preds)
    mean_p = preds.mean(axis=0)
    entropy = -np.sum(mean_p * np.log(mean_p + 1e-9), axis=1)
    variance = preds.var(axis=0).sum(axis=1)
    return mean_p, entropy, variance


def _meta_feature_row(trow):
    img_id = str(trow['image_id'])
    img_match = test_df[test_df.get('image_id', pd.Series(dtype=str)).astype(str) == img_id]
    if len(img_match) == 0:
        return None
    img_row = img_match.iloc[0]

    img_bgr = cv2.imread(img_row['image_path'])
    if img_bgr is None:
        return None
    img_bgr = dull_razor_hair_removal(img_bgr)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_norm = cv2.resize(img_rgb, (224,224)).astype(np.float32)/255.0
    img_t = torch.from_numpy(((img_norm-_mean_n)/_std_n).transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(img_t)
        probs = torch.softmax(logits/T_dense, dim=1).cpu().numpy()[0]
        conf = float(probs.max())
        pred_c = int(probs.argmax())

    if 'feat' in embeddings:
        feat_vec = embeddings['feat'].mean(axis=(2,3)).flatten()
    else:
        feat_vec = np.zeros(1024)

    mean_mc, ent, var = mc_dropout_predict(model_mc, img_t, T=10)

    class_onehot = np.zeros(CFG['n_classes'])
    class_onehot[pred_c] = 1.0
    feature = np.concatenate([[conf], [float(ent[0])], [float(var[0])], class_onehot])

    return {
        'image_id': img_id,
        'feature_json': json.dumps(feature.tolist()),
        'embedding_json': json.dumps(feat_vec.tolist()),
        'label': 1 if float(trow['tixai']) < 0.45 else 0
    }


print('\nExtracting meta-model features (embeddings + MC Dropout, resumable)...')
meta_raw_df = run_resumable_loop(
    correct_df, CFG['out_dir'] / 'meta_features_raw.csv', _meta_feature_row,
    id_col='image_id', desc='Meta-features'
)
hook.remove()

from sklearn.decomposition import PCA
if len(meta_raw_df) > 50:
    X_meta = np.array([json.loads(j) for j in meta_raw_df['feature_json']])
    emb_arr = np.array([json.loads(j) for j in meta_raw_df['embedding_json']])
    y = meta_raw_df['label'].values

    print(f'\nMeta-model dataset: {emb_arr.shape[0]} samples')
    print(f'Class balance: reliable={sum(y==0)}, unreliable={sum(y==1)}')

    from sklearn.pipeline import Pipeline
    from sklearn.compose import ColumnTransformer

    # FIX: the previous MetaFeatureCombiner closed over the FULL-dataset
    # X_meta array and np.hstack'd it onto whatever row-subset cross_val_predict
    # passed through the pipeline for the current fold -- since cross_val_predict
    # always slices by row count (n_train_fold or n_test_fold), while X_meta kept
    # its full n_total row count, this raised a hard shape-mismatch ValueError on
    # the very first fold (confirmed: "size 100 and the array at index 1 has size
    # 80"for a 5-fold split of n=100). It also required BaseEstimator/
    # TransformerMixin which were never imported (NameError before even reaching
    # the shape bug).
    #
    # Correct fix: concatenate X_meta and emb_arr ONCE, row-aligned, into a single
    # combined_X BEFORE cross-validation (this is safe -- no fitted transform is
    # applied yet, it's just a raw feature concatenation). Then use a
    # ColumnTransformer inside the Pipeline so PCA is fit ONLY on the embedding
    # columns of whatever fold-subset of rows is passed in, while the raw
    # meta-feature columns (confidence/entropy/variance/one-hot) pass through
    # unchanged. This way PCA is refit per-fold (no leakage) and every step
    # correctly operates on the same row-subset cross_val_predict provides.
    n_meta_cols = X_meta.shape[1]
    combined_X = np.hstack([X_meta, emb_arr])
    meta_col_idx = list(range(n_meta_cols))
    emb_col_idx = list(range(n_meta_cols, combined_X.shape[1]))

    # FIX: n_components was sized from the FULL dataset (emb_arr.shape[0]),
    # but this PCA is fit INSIDE each cross_val_predict fold on only the
    # training subset of that data. With 5-fold CV, each training fold has
    # roughly 4/5 of the total samples -- for a small meta-model dataset
    # this is smaller than 64, so PCA raised "n_components=64 must be
    # between 0 and min(n_samples, n_features)=55" on the very first fold.
    # Size the cap from the worst-case (smallest) fold instead of the full
    # dataset so it's guaranteed to fit every fold, not just the unsplit data.
    N_CV_SPLITS = 5
    max_fold_train_samples = int(emb_arr.shape[0] * (N_CV_SPLITS - 1) / N_CV_SPLITS)
    n_pca_components = min(64, max_fold_train_samples - 1, emb_arr.shape[1])

    preprocessor = ColumnTransformer([
        ('meta_passthrough', 'passthrough', meta_col_idx),
        ('emb_pca', PCA(n_components=n_pca_components, random_state=SEED), emb_col_idx),
    ])

    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('scaler', StandardScaler()),
        ('classifier', GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=SEED))
    ])
    cv = StratifiedKFold(n_splits=N_CV_SPLITS, shuffle=True, random_state=SEED)
    y_prob_meta = cross_val_predict(pipeline, combined_X, y, cv=cv, method='predict_proba')[:, 1]
    y_pred_meta = (y_prob_meta >= 0.5).astype(int)

    auc_meta = roc_auc_score(y, y_prob_meta)
    print(f'\nMeta-model (GAP #2) 5-Fold CV Results:')
    print(f'Mean AUC-ROC: {auc_meta:.4f}')
    print(cr(y, y_pred_meta, target_names=['Reliable','Unreliable']))

    pipeline.fit(combined_X, y)
    fitted_pca = pipeline.named_steps['preprocess'].named_transformers_['emb_pca']
    feat_names = (['confidence','entropy','mc_variance'] +
                  [f'class_{c}' for c in CLASSES] +
                  [f'emb_{i}' for i in range(fitted_pca.n_components_)])
    fi = pd.DataFrame({'feature': feat_names, 'importance': pipeline.named_steps['classifier'].feature_importances_})
    fi = fi.sort_values('importance', ascending=False).head(15)
    print('\nTop meta-model features:')
    print(fi.to_string(index=False))
    fi.to_csv(CFG['out_dir'] / 'meta_model_feature_importance.csv', index=False)

    # NEW: BCa bootstrap CI on the primary (0.45-threshold) AUC -- every
    # headline number in this notebook should carry an interval, and this
    # one previously didn't.
    try:
        auc_boot = bootstrap((y, y_prob_meta), statistic=roc_auc_score,
                              n_resamples=2000, method='BCa', random_state=SEED,
                              paired=True, vectorized=False)
        print(f'\nAUC = {auc_meta:.4f}, 95% BCa CI [{auc_boot.confidence_interval.low:.4f}, '
              f'{auc_boot.confidence_interval.high:.4f}] (2000 resamples)')
    except Exception as e:
        print(f'\nWARNING: BCa bootstrap CI for AUC failed ({type(e).__name__}: {e}) -- '
              f'likely a resample landed with only one class present (small/imbalanced '
              f'"unreliable" label group). Point estimate AUC={auc_meta:.4f} is still valid; '
              f'the CI just could not be computed this run.')

    # NEW: TIxAI-threshold sensitivity for the failure-prediction LABEL.
    # The 0.45 cutoff baked into _meta_feature_row above is an arbitrary
    # round number -- re-derive the label at 0.35, 0.45, and the empirical
    # median TIxAI. This reuses the SAME already-extracted features (no
    # re-running the expensive per-image embedding/MC-dropout loop) --
    # only the label (and therefore the cheap CV refit) changes per threshold.
    print('\nTIxAI-threshold sensitivity for the failure-prediction label:')
    raw_tixai = meta_raw_df.merge(
        correct_df[['image_id', 'tixai']], on='image_id', how='left'
    )['tixai'].values
    median_tixai = float(np.median(raw_tixai))
    threshold_sensitivity = []
    for thresh_name, thresh_val in [('0.35', 0.35), ('0.45', 0.45),
                                     (f'median={median_tixai:.3f}', median_tixai)]:
        y_t = (raw_tixai < thresh_val).astype(int)
        pos_rate = float(y_t.mean())
        if pos_rate < 0.05 or pos_rate > 0.95:
            print(f'threshold={thresh_name}: skipped (positive rate={pos_rate:.1%}, '
                  f'too imbalanced for reliable CV)')
            threshold_sensitivity.append({'threshold': thresh_name, 'pos_rate': pos_rate, 'auc': None})
            continue
        y_prob_t = cross_val_predict(pipeline, combined_X, y_t, cv=cv, method='predict_proba')[:, 1]
        auc_t = roc_auc_score(y_t, y_prob_t)
        print(f'threshold={thresh_name}: pos_rate={pos_rate:.1%}, AUC={auc_t:.4f}')
        threshold_sensitivity.append({'threshold': thresh_name, 'pos_rate': pos_rate, 'auc': auc_t})
    pd.DataFrame(threshold_sensitivity).to_csv(CFG['out_dir'] / 'gap2_threshold_sensitivity.csv', index=False)
    valid_aucs = [r['auc'] for r in threshold_sensitivity if r['auc'] is not None]
    if len(valid_aucs) >= 2:
        print(f'\n-> AUC range across thresholds: [{min(valid_aucs):.4f}, {max(valid_aucs):.4f}] '
              f'({"stable" if max(valid_aucs) - min(valid_aucs) < 0.05 else "threshold-sensitive -- report all three, not just 0.45"})')
else:
    print('WARNING: Insufficient samples for meta-model. Need more images with ISIC 2018 masks.')
    auc_meta = float('nan')


## CELL 14b — Uncertainty-Gated Explanation Suppression Policy (NEW — Novelty Add)

In [ ]:
# ============================================================
# CELL 14b — Uncertainty-Gated Explanation Suppression Policy
# (NEW — Novelty Add: a concrete clinical-deployment rule that combines
# BOTH signals from GAP-2's meta-model features -- entropy AND TIxAI --
# rather than either alone. Reuses CELL 14's already-extracted MC-dropout
# entropy (packed into meta_raw_df's feature_json as index 1) so this
# costs no extra GPU forward passes.
# ============================================================
if 'meta_raw_df' in dir() and len(meta_raw_df) > 0:
    _entropy_vals = np.array([json.loads(j)[1] for j in meta_raw_df['feature_json']])
    suppress_df = meta_raw_df[['image_id']].copy()
    suppress_df['entropy'] = _entropy_vals
    suppress_df = suppress_df.merge(correct_df[['image_id', 'tixai']], on='image_id', how='left')
    suppress_df = suppress_df.dropna(subset=['tixai'])

    ENTROPY_THRESH = float(np.median(suppress_df['entropy']))  # data-driven, not an arbitrary constant
    TIXAI_THRESH = 0.45  # matches the GAP-2 failure-prediction label cutoff for consistency

    def uncertainty_gated_display(entropy, tixai, entropy_thresh=ENTROPY_THRESH, tixai_thresh=TIXAI_THRESH):
        """
        Per-image clinical display decision:
          'SHOW'     -- both signals reliable, display the heatmap normally
          'WARN'     -- one signal is poor, display with a caution badge
          'SUPPRESS' -- both signals are poor, withhold the heatmap and
                        flag the case for clinician review instead
        """
        decisions = []
        for e, t in zip(entropy, tixai):
            if e > entropy_thresh and t < tixai_thresh:
                decisions.append('SUPPRESS')
            elif e > entropy_thresh or t < tixai_thresh:
                decisions.append('WARN')
            else:
                decisions.append('SHOW')
        return decisions

    suppress_df['decision'] = uncertainty_gated_display(suppress_df['entropy'].values, suppress_df['tixai'].values)
    decision_counts = suppress_df['decision'].value_counts()
    print(f'Uncertainty-gated suppression policy (entropy_thresh={ENTROPY_THRESH:.3f} [median], '
          f'tixai_thresh={TIXAI_THRESH}):')
    print(decision_counts.to_string())
    print(f'\n-> {100 * decision_counts.get("SUPPRESS", 0) / len(suppress_df):.1f}% of correctly-classified '
          f'test cases would have their heatmap suppressed in deployment under this policy.')

    # Per-class breakdown -- rare classes should show up here if the GAP-1 finding holds
    class_col = correct_df.set_index('image_id')['true_class']
    suppress_df['true_class'] = suppress_df['image_id'].map(class_col)
    print('\nSuppression rate by class:')
    print((suppress_df.groupby('true_class')['decision']
           .apply(lambda s: (s == 'SUPPRESS').mean())
           .sort_values(ascending=False).to_string()))

    suppress_df.to_csv(CFG['out_dir'] / 'Table_uncertainty_gated_suppression.csv', index=False)
else:
    print('CELL 14b: skipped -- meta_raw_df not available (CELL 14 did not produce enough samples).')


## CELL 14c — Per-FST MC-Dropout Uncertainty Stratification (NEW — Novelty Add)

In [ ]:
# ============================================================
# CELL 14c — Per-FST MC-Dropout Uncertainty Stratification
# (NEW — Novelty Add: is the model itself MORE UNCERTAIN, not just
# less-well-explained, on darker skin tones? CELL 10b already audits
# per-FST TIxAI [explanation reliability]; this is a genuinely different
# signal [predictive uncertainty] on the same FST buckets. BOSQUE 2025
# discusses epistemic uncertainty in fairness generally but does not
# stratify per-image entropy by (ITA-estimated) FST -- this does.
#
# Reuses fst_df (CELL 10b, has fst_bucket per image_id) and meta_raw_df
# (CELL 14, has entropy packed into feature_json) -- no new GPU forward
# passes needed, just a join of two already-computed tables.
# ============================================================
if 'fst_df' in dir() and len(fst_df) > 10 and 'meta_raw_df' in dir() and len(meta_raw_df) > 0:
    _entropy_vals_fst = np.array([json.loads(j)[1] for j in meta_raw_df['feature_json']])
    entropy_lookup = pd.DataFrame({'image_id': meta_raw_df['image_id'].values, 'entropy': _entropy_vals_fst})

    fst_entropy_df = fst_df[['image_id', 'fst_bucket']].merge(entropy_lookup, on='image_id', how='inner')

    if len(fst_entropy_df) < 10:
        print(f'CELL 14c: only {len(fst_entropy_df)} images have both an FST bucket (CELL 10b) and '
              f'an entropy value (CELL 14) -- these come from different loops over test_df with '
              f'their own filters, so overlap can be small. Skipping.')
    else:
        fst_entropy_groups = {b: fst_entropy_df[fst_entropy_df['fst_bucket'] == b]['entropy'].values
                               for b in ['I-II', 'III-IV', 'V-VI']
                               if (fst_entropy_df['fst_bucket'] == b).sum() >= 5}
        print('Per-FST MC-Dropout predictive entropy (higher = more uncertain):')
        for b, vals in fst_entropy_groups.items():
            print(f'FST {b:8s}: n={len(vals):4d} mean entropy={np.mean(vals):.4f} +/- {np.std(vals):.4f}')

        if len(fst_entropy_groups) >= 2:
            H_ent, p_ent = kruskal(*fst_entropy_groups.values())
            print(f'\nKruskal-Wallis across FST buckets (entropy): H={H_ent:.4f}, p={p_ent:.4f}')
            print(f'{"REJECT H0 -- model is systematically more/less uncertain by skin tone" if p_ent < 0.05 else "FAIL TO REJECT H0"}')
            pd.DataFrame([
                {'fst_bucket': b, 'n': len(v), 'mean_entropy': np.mean(v), 'std_entropy': np.std(v)}
                for b, v in fst_entropy_groups.items()
            ]).to_csv(CFG['out_dir'] / 'Table_per_fst_entropy.csv', index=False)
        print('\nCAVEAT: same ITA-based FST proxy caveat as CELL 10b -- not clinical-grade labels.')
else:
    print('CELL 14c: skipped -- requires both fst_df (CELL 10b) and meta_raw_df (CELL 14) to be populated.')



## CELL 15 — GAP-8: Composite Clinical Risk Score (MODIFIED: resumable + confusion-pair risk term, Novelty Add #7)


In [ ]:
# ============================================================
# CELL 15 — GAP #8: Composite Clinical Risk Score (CCRS)
# + Conformal Selective Prediction (Deferral)
# FIXES applied:
# - The per-test-row CCRS loop is now resumable.
# - Added a 6th weight term: confusion-pair risk, built from the
# Confusion Saliency Matrix counts in CELL 13 (Novelty Add #7 --
# defer harder on clinically dangerous confusions like MEL<->NV
# than low-stakes ones like BKL<->NV).
# - FIX (ground-truth leakage, audit 2026-07-05): confusion_pair_risk()
# used to take the sample's own TRUE label as an argument and return 0.0
# whenever the prediction was correct -- i.e. the risk score that decides
# whether to defer to a clinician was computed using the answer to the
# diagnosis. In real deployment the true label is never known in advance,
# so this was not a deployable score, and it mechanically produced the
# "Error rate in deferred: 1.0000" result (every deferred case was wrong
# by construction, not because the score genuinely identified risk).
# Fixed by keying the pair-risk term ONLY on the predicted class and an
# aggregate, population-level error rate for that predicted class (drawn
# from CELL 13's confusion_counts across the whole evaluated set) --
# this no longer depends on knowing the current sample's true label.
# `true_class` is still recorded on each output row for POST-HOC
# evaluation (accuracy within the deferred/non-deferred groups), which is
# normal and necessary -- the fix removes it only from the score's own
# inputs.
# - FIX (imputed-TIxAI coverage, audit 2026-07-05): tixai_lookup only had
# entries for CORRECTLY classified test images, because CELL 10's own
# TIxAI computation is deliberately restricted to correct predictions
# (a documented GAP-1 methodology choice, isolating explanation quality
# from classification error). That left every misclassified test image
# (52.9% of the test set) resting on a hardcoded 0.5 placeholder in the
# CCRS score, instead of a real explanation-quality signal. Nothing about
# computing Grad-CAM++ for the model's OWN predicted class requires the
# prediction to be correct, so both _tauval_process_row and
# _ccrs_process_row below now compute a real, predicted-class-conditioned
# TIxAI for every row that CELL 10 didn't already score, instead of
# imputing. GAP-1's own per-class analysis in CELL 10 is untouched.
# ============================================================

assert (not TIXAI_USES_SYNTHETIC_MASKS) or CFG.get('allow_synthetic_masks_in_results', False), (
    'Refusing to compute this cell on synthetic (non-GT) lesion masks. Attach the ISIC2018 '
    "Task 1 dataset, or set CFG['allow_synthetic_masks_in_results']=True to proceed anyway for a demo run."
)

MALIGNANCY_PRIOR = {
    'MEL': 0.95, 'BCC': 0.85, 'AKIEC': 0.70,
    'BKL': 0.10, 'NV': 0.05, 'DF': 0.05, 'VASC': 0.15,
}

# FIX: predicted classes historically implicated in the model's dangerous
# confusions with MEL (per the original _DANGEROUS_PAIRS set, expressed
# here as PREDICTED-class-only so no true label is required at score time).
# Predicting one of these still carries elevated risk of an underlying
# missed melanoma, independent of what the current sample's true class is.
DANGEROUS_PRED_CLASSES = {'NV', 'BKL', 'BCC', 'AKIEC'}

def confusion_pair_risk(pred_class, confusion_counts, dangerous_classes):
    """Population-level risk prior for a PREDICTED class: how often has the
    model been wrong when it predicted this class, aggregated across
    confusion_counts (built in CELL 13). Does not use the current sample's
    true label -- only its prediction and an aggregate historical rate."""
    total_errors_for_pred = sum(
        v for (tc, pc), v in confusion_counts.items() if pc == pred_class
    )
    if pred_class in dangerous_classes:
        return 0.7
    return float(np.clip(0.1 + 0.03 * np.log1p(total_errors_for_pred), 0.1, 0.5))


def compute_ccrs(pred_class, confidence, tixai_score, conf_set_size,
                  confusion_counts, dangerous_classes,
                  weights=(0.25, 0.25, 0.20, 0.10, 0.20), n_classes=7):
    # FIX (2026-07-05): XDI removed from the CCRS entirely. GAP-4 established XDI does
    # NOT separate correct from incorrect predictions (Mann-Whitney p=0.977, null at
    # all three thresholds), so weighting it into the deferral score added noise, not
    # signal, and contradicted GAP-4's own report. XDI is retained ONLY as a diagnostic
    # value in the per-image CCRS CSV; it no longer influences the deferral decision.
    # Remaining five terms (confidence, TIxAI, malignancy, conformal-set, confusion-pair)
    # have weights redistributed to sum to 1.0.
    w1, w2, w3, w4, w5 = weights
    mal_prior = MALIGNANCY_PRIOR.get(pred_class, 0.3)
    pair_risk = confusion_pair_risk(pred_class, confusion_counts, dangerous_classes)
    ccrs = (w1 * (1 - confidence) +
            w2 * (1 - tixai_score) +
            w3 * mal_prior * (1 - confidence) +
            w4 * (conf_set_size / n_classes) +
            w5 * pair_risk)
    return float(np.clip(ccrs, 0, 1))


def build_conformal_prediction_set(logits, calibration_labels, calibration_logits, alpha=0.1):
    cal_probs = torch.softmax(torch.tensor(calibration_logits), dim=1).numpy()
    nonconformity_scores = [1 - cal_probs[i, calibration_labels[i]] for i in range(len(calibration_labels))]
    q_level = np.ceil((1 - alpha) * (len(nonconformity_scores) + 1)) / len(nonconformity_scores)
    q_hat = np.quantile(nonconformity_scores, min(q_level, 1.0))
    test_probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    pred_set = [i for i in range(test_probs.shape[1]) if 1 - test_probs[0, i] <= q_hat]
    return pred_set, float(q_hat)


cal_split_df, tauval_split_df = train_test_split(
    val_df, test_size=0.5, random_state=SEED, stratify=val_df['label_idx']
)

_ckpt_cal = load_ckpt('cell15_cal_logits')
if _ckpt_cal is not None:
    cal_logits, cal_labels = _ckpt_cal['cal_logits'], _ckpt_cal['cal_labels']
else:
    print('\nCollecting calibration logits from val set (calibration half only, n='+ str(len(cal_split_df)) + ')...')
    cal_ds_split = SkinDataset(cal_split_df, val_transform, use_hair_removal=True)
    cal_loader_split = DataLoader(cal_ds_split, batch_size=CFG['batch_size'], shuffle=False, num_workers=0, pin_memory=True)
    cal_logits_list, cal_labels_list = [], []
    model.eval()
    with torch.no_grad():
        for imgs, labels, _ in tqdm(cal_loader_split, desc='Cal logits', leave=False):
            imgs = imgs.to(DEVICE)
            lgs = model(imgs).cpu().numpy()
            cal_logits_list.extend(lgs)
            cal_labels_list.extend(labels.numpy())
    cal_logits = np.array(cal_logits_list)
    cal_labels = np.array(cal_labels_list)
    save_ckpt('cell15_cal_logits', {'cal_logits': cal_logits, 'cal_labels': cal_labels})


def _tauval_process_row(vrow):
    vimg_id = str(vrow.get('image_id', vrow.name))
    vimg_bgr = cv2.imread(vrow['image_path'])
    if vimg_bgr is None:
        return None
    vimg_bgr = dull_razor_hair_removal(vimg_bgr)
    vimg_rgb = cv2.cvtColor(vimg_bgr, cv2.COLOR_BGR2RGB)
    vimg_norm = cv2.resize(vimg_rgb, (224, 224)).astype(np.float32) / 255.0
    vimg_t = torch.from_numpy(((vimg_norm - _mean_n) / _std_n).transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        v_logits = model(vimg_t).cpu()
    v_probs = torch.softmax(v_logits / T_dense, dim=1).numpy()[0]
    v_pred_idx = int(v_probs.argmax())
    v_pred_cls = CLASSES[v_pred_idx]
    v_conf = float(v_probs.max())
    v_pred_set, _ = build_conformal_prediction_set(v_logits.numpy(), cal_labels, cal_logits, alpha=CFG['deferral_alpha'])
    v_correct = v_pred_idx == int(vrow['label_idx'])
    # FIX: previously gated behind `if v_correct:` and defaulted to 0.5
    # otherwise. Grad-CAM++ here is already computed against the model's
    # PREDICTED class (v_pred_idx), not the true class, so nothing about
    # this computation actually requires the prediction to be correct --
    # only a genuine exception (bad mask, bad image) should fall back to 0.5.
    try:
        v_gcam = cam_dense(input_tensor=vimg_t, targets=[ClassifierOutputTarget(v_pred_idx)])[0]
        if not USE_SYNTHETIC_MASKS and vimg_id in masks_dict:
            v_tixai = compute_tixai(v_gcam, masks_dict[vimg_id])
        else:
            v_tixai = compute_tixai(v_gcam, make_synthetic_mask(CFG['img_size']))
    except Exception:
        v_tixai = 0.5
    v_xdi = 0.5 # documented neutral default during val calibration; SHAP-based XDI not computed here
    return {'image_id': vimg_id, 'pred_cls': v_pred_cls, 'confidence': v_conf, 'tixai': v_tixai,
            'xdi': v_xdi, 'conf_set_size': len(v_pred_set), 'correct': v_correct,
            'true_class': CLASSES[int(vrow['label_idx'])]}


print('\nCollecting tau-calibration measurements (resumable, n='+ str(len(tauval_split_df)) + ')...')
tauval_df = run_resumable_loop(
    tauval_split_df, CFG['out_dir'] / 'cell15_tauval.csv', _tauval_process_row,
    id_col='image_id', desc='TauCal'
)
tauval_rows = tauval_df.to_dict('records')

print('\nRunning CCRS weight ablation sweep on the tau-calibration split...')
WEIGHT_CANDIDATES = [  # (confidence, TIxAI, malignancy, conf_set_size, confusion_pair); XDI removed
    (0.25, 0.25, 0.20, 0.10, 0.20), # Baseline
    (0.45, 0.15, 0.15, 0.05, 0.20), # Confidence-heavy
    (0.15, 0.45, 0.15, 0.05, 0.20), # TIxAI-heavy
    (0.20, 0.20, 0.35, 0.05, 0.20), # Malignancy-heavy
    (0.15, 0.15, 0.15, 0.15, 0.40), # Confusion-pair-heavy
]
sweep_results = []
for cand in WEIGHT_CANDIDATES:
    cand_ccrs = np.array([
        compute_ccrs(r['pred_cls'], r['confidence'], r['tixai'], r['conf_set_size'],
                     confusion_counts, DANGEROUS_PRED_CLASSES, weights=cand)
        for r in tauval_rows
    ])
    cand_correct = np.array([r['correct'] for r in tauval_rows])
    cand_tau = float(np.percentile(cand_ccrs, 70))
    cand_deferred = cand_ccrs >= cand_tau
    cand_ai_acc = float(cand_correct[~cand_deferred].mean()) if (~cand_deferred).sum() > 0 else float('nan')
    sweep_results.append({'weights': cand, 'tau': cand_tau, 'ai_accuracy': cand_ai_acc,
                          'deferral_rate': float(cand_deferred.mean())})
    print('weights='+ str(cand) + '-> tau='+ str(round(cand_tau, 4)) +
          'AI-handled acc='+ str(round(cand_ai_acc, 4)) +
          'deferral rate='+ str(round(100 * cand_deferred.mean(), 1)) + '%')

best = max(sweep_results, key=lambda r: (r['ai_accuracy'] if not np.isnan(r['ai_accuracy']) else -1.0))
OPTIMAL_WEIGHTS = best['weights']
tau = best['tau']
print('Selected weights (max AI-handled accuracy on tau-calibration split): '+ str(OPTIMAL_WEIGHTS) +
      '(tau='+ str(round(tau, 4)) + ')')

tixai_lookup = {str(r['image_id']): r['tixai'] for _, r in tixai_df.iterrows()
                if r['correct'] and r['tixai'] is not None}
xdi_lookup = {str(r['image_id']): r['xdi'] for _, r in xdi_df.iterrows()}

def _ccrs_process_row(row):
    img_id = str(row.get('image_id', row.name))
    true_cls = row['label']
    img_bgr = cv2.imread(row['image_path'])
    if img_bgr is None:
        return None
    img_bgr = dull_razor_hair_removal(img_bgr)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_norm = cv2.resize(img_rgb, (224,224)).astype(np.float32)/255.0
    img_t = torch.from_numpy(((img_norm-_mean_n)/_std_n).transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        test_logits = model(img_t).cpu()
        probs = torch.softmax(test_logits/T_dense, dim=1).numpy()[0]
        pred_idx = int(probs.argmax())
        pred_cls = CLASSES[pred_idx]
        confidence = float(probs.max())

    pred_set, q_hat = build_conformal_prediction_set(test_logits.numpy(), cal_labels, cal_logits, alpha=CFG['deferral_alpha'])

    # FIX (audit 2026-07-05): tixai_lookup only covers CELL 10's correctly-
    # classified subset. Previously every other row (all 524 misclassified
    # test images) fell back to a hardcoded 0.5. Compute a real, predicted-
    # class-conditioned TIxAI here instead -- this needs no true label, only
    # the model's own prediction and the lesion mask.
    if img_id in tixai_lookup:
        tixai_s = tixai_lookup[img_id]
        tixai_imputed = False
    else:
        try:
            gcam = cam_dense(input_tensor=img_t, targets=[ClassifierOutputTarget(pred_idx)])[0]
            if not USE_SYNTHETIC_MASKS and img_id in masks_dict:
                tixai_s = compute_tixai(gcam, masks_dict[img_id])
            else:
                tixai_s = compute_tixai(gcam, make_synthetic_mask(CFG['img_size']))
            tixai_imputed = False
        except Exception:
            tixai_s = 0.5
            tixai_imputed = True
    xdi_s = xdi_lookup.get(img_id, 0.5)
    # NOTE: true_cls is NOT passed into compute_ccrs -- it is recorded below
    # only for post-hoc evaluation (accuracy within the deferred/non-deferred
    # groups), never as an input to the risk score itself.
    ccrs = compute_ccrs(pred_cls, confidence, tixai_s, len(pred_set),
                         confusion_counts, DANGEROUS_PRED_CLASSES, weights=OPTIMAL_WEIGHTS)

    return {'image_id': img_id, 'true_class': true_cls, 'pred_class': pred_cls,
            'correct': pred_idx == int(row['label_idx']),
            'confidence': confidence, 'tixai': tixai_s, 'xdi': xdi_s,
            'tixai_imputed': tixai_imputed,
            'conf_set_size': len(pred_set), 'ccrs': ccrs,
            'malignancy': MALIGNANCY_PRIOR.get(pred_cls, 0.3)}


print('\nComputing Composite Clinical Risk Scores (resumable)...')
ccrs_df = run_resumable_loop(
    test_df, CFG['out_dir'] / 'ccrs_results.csv', _ccrs_process_row,
    id_col='image_id', desc='CCRS'
)

ccrs_test_ids = set(ccrs_df['image_id'].astype(str))
n_tixai_imputed = int(ccrs_df['tixai_imputed'].sum()) if 'tixai_imputed' in ccrs_df.columns else 0
n_xdi_imputed = sum(1 for tid in ccrs_test_ids if tid not in xdi_lookup)
n_total_ccrs = len(ccrs_test_ids)
print('CCRS imputation check: TIxAI defaulted to 0.5 for '+ str(n_tixai_imputed) + '/'+ str(n_total_ccrs) +
      '('+ str(round(100*n_tixai_imputed/max(n_total_ccrs,1),1)) + '%) -- counts only rows where computing '
      'Grad-CAM++/TIxAI for the predicted class itself raised an exception; every other row (including all '
      'misclassified images) now gets a real, predicted-class-conditioned TIxAI instead of a placeholder.')
print('CCRS imputation check: XDI defaulted to 0.5 for '+ str(n_xdi_imputed) + '/'+ str(n_total_ccrs) +
      '('+ str(round(100*n_xdi_imputed/max(n_total_ccrs,1),1)) + '%)')
if n_tixai_imputed > 0.3 * n_total_ccrs or n_xdi_imputed > 0.3 * n_total_ccrs:
    print('WARNING: More than 30% of CCRS rows rest on an imputed 0.5 default. With the CELL 12 fix')
    print('(full-test-set XDI), this should now be near 0% for XDI -- re-run CELL 12 to full coverage first.')

ccrs_df['deferred'] = ccrs_df['ccrs'] >= tau
n_deferred = ccrs_df['deferred'].sum()
n_ai_handled = (~ccrs_df['deferred']).sum()
ai_accuracy = ccrs_df[~ccrs_df['deferred']]['correct'].mean()
deferred_error_rate = ccrs_df[ccrs_df['deferred']]['correct'].mean()

print(f'\nClinical Deferral Results (τ={tau:.3f}):')
print(f'Total test samples: {len(ccrs_df):,}')
print(f'AI-handled (confident): {n_ai_handled:,} ({100*n_ai_handled/len(ccrs_df):.1f}%)')
print(f'Deferred to clinician: {n_deferred:,} ({100*n_deferred/len(ccrs_df):.1f}%)')
print(f'AI accuracy (non-defer):{ai_accuracy:.4f}')
print(f'Error rate in deferred: {1-deferred_error_rate:.4f}')
print(f'Overall accuracy: {ccrs_df["correct"].mean():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0D1117')
ax1 = axes[0]; ax1.set_facecolor('#161B22')
ax1.hist(ccrs_df[~ccrs_df['deferred']]['ccrs'], bins=30, alpha=0.8, color='#4CAF50',
         label='AI-handled', edgecolor='white', linewidth=0.3)
ax1.hist(ccrs_df[ccrs_df['deferred']]['ccrs'], bins=30, alpha=0.8, color='#F44336',
         label='Deferred to clinician', edgecolor='white', linewidth=0.3)
ax1.axvline(tau, color='yellow', linestyle='--', linewidth=2, label=f'τ={tau:.3f}')
ax1.set_xlabel('Composite Clinical Risk Score (CCRS)', color='white', fontsize=12)
ax1.set_ylabel('Count', color='white', fontsize=12)
ax1.set_title('CCRS Distribution with Deferral Threshold', color='white', fontsize=13, fontweight='bold')
ax1.tick_params(colors='white'); ax1.legend(fontsize=10, facecolor='#161B22', labelcolor='white')
ax1.spines[:].set_color('#30363D')

ax2 = axes[1]; ax2.set_facecolor('#161B22')
per_class_defer = ccrs_df.groupby('true_class')['deferred'].mean().reindex(CLASSES)
bar_colors = [CLASS_COLORS[c] for c in CLASSES]
ax2.bar(CLASSES, per_class_defer.values * 100, color=bar_colors, edgecolor='white', linewidth=1.2)
ax2.set_xlabel('Lesion Class', color='white', fontsize=12)
ax2.set_ylabel('Deferral Rate (%)', color='white', fontsize=12)
ax2.set_title('Per-Class Deferral Rate\n(Rare classes should defer more)', color='white', fontsize=13, fontweight='bold')
ax2.tick_params(colors='white', axis='x', rotation=30); ax2.tick_params(colors='white', axis='y')
ax2.spines[:].set_color('#30363D')

plt.suptitle('GAP #8: Composite Clinical Risk Score + Conformal Deferral (now with confusion-pair risk)',
             fontsize=14, fontweight='bold', color='white')
plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'Figure6_ccrs_deferral.png', dpi=300, bbox_inches='tight', facecolor='#0D1117')
plt.show()
plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise
print('\nFigure 6 (CCRS + Deferral) saved.')



## CELL 15b — Per-FST Conformal Coverage Audit (NEW — Novelty Add #8)


In [ ]:
# ============================================================
# CELL 15b — Mondrian (group-conditional) conformal coverage by FST
#
# NEW: Standard conformal prediction (CELL 15) guarantees coverage
# OVERALL, not per demographic group. This builds a separate q_hat
# per estimated FST bucket (from CELL 10b's ITA estimates), giving a
# per-skin-tone coverage guarantee -- combining fairness + conformal
# prediction in a way your audit flagged as unaddressed in dermoscopy.
#
# FIX (dead-code bug, audit 2026-07-05): the evaluation step used to
# intersect `fst_df` (FST buckets estimated ONLY for test_df in CELL 10b)
# with `tauval_split_df` (a split of val_df) by image_id. Since test_df
# and val_df are disjoint by construction, that intersection was always
# empty, so every bucket printed "too few eval samples -- skipped" and
# this cell never actually computed a coverage number. Fixed by
# estimating FST buckets directly on tauval_split_df (same ITA estimator
# already used for cal_split_df below), so calibration and evaluation
# both draw their FST bucket labels from data that's actually in scope.
# ============================================================
if TIXAI_USES_SYNTHETIC_MASKS and not CFG.get('allow_synthetic_masks_in_results', False):
    print(chr(0x26A0) + 'WARNING: ISIC2018 ground-truth masks are not attached -- skipping '
          'per-FST conformal coverage (it depends on Cell 10b\'s TIxAI/FST estimates, which '
          'are themselves skipped under synthetic masks). A coverage-gap finding here would '
          'otherwise be indistinguishable from a real fairness result.')
elif 'fst_df' in dir() and len(fst_df) > 20:
    # Estimate FST for the calibration split directly (cal_split_df wasn't
    # processed by CELL 10b, which only ran on test_df) -- reuse the same
    # ITA estimator on a quick pass.
    def _cal_fst_row(row):
        img_bgr = cv2.imread(row['image_path'])
        if img_bgr is None:
            return None
        img_bgr = dull_razor_hair_removal(img_bgr)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rs = cv2.resize(img_rgb, (CFG['img_size'], CFG['img_size']))
        # crude full-image skin mask (no lesion mask needed here -- just for ITA)
        mask_bin = (make_synthetic_mask(CFG['img_size']) > 127).astype(np.uint8)
        ita = estimate_ita(img_rs, mask_bin)
        bucket = ita_to_fst_bucket(ita)
        if bucket is None:
            return None
        return {'image_id': str(row.get('image_id', row.name)), 'fst_bucket': bucket}

    cal_fst_df = run_resumable_loop(
        cal_split_df, CFG['out_dir'] / 'cell15b_cal_fst.csv', _cal_fst_row,
        id_col='image_id', desc='Cal-FST-estimate'
    )

    # FIX: tauval_split_df (the evaluation half of val_df) never had FST
    # buckets estimated at all -- fst_df only covers test_df. Run the SAME
    # estimator on tauval_split_df so the evaluation loop below has FST
    # labels for the split it's actually iterating over.
    tauval_fst_df = run_resumable_loop(
        tauval_split_df, CFG['out_dir'] / 'cell15b_tauval_fst.csv', _cal_fst_row,
        id_col='image_id', desc='TauVal-FST-estimate'
    )
    tauval_split_fst = tauval_split_df.merge(tauval_fst_df, on='image_id', how='inner')

    cal_split_fst = cal_split_df.merge(cal_fst_df, on='image_id', how='inner')
    cal_logits_df = pd.DataFrame(cal_logits)
    cal_logits_df['image_id'] = cal_split_df.reset_index(drop=True)['image_id'].astype(str).values[:len(cal_logits_df)]

    print('\nPer-FST Mondrian Conformal Coverage (target = 90%):')
    fst_coverage_results = []
    for bucket in ['I-II', 'III-IV', 'V-VI']:
        bucket_ids = set(cal_split_fst[cal_split_fst['fst_bucket'] == bucket]['image_id'].astype(str))
        if len(bucket_ids) < 10:
            print(f'FST {bucket}: too few calibration samples (n={len(bucket_ids)}) -- skipped.')
            continue
        bucket_mask = cal_logits_df['image_id'].isin(bucket_ids).values
        if bucket_mask.sum() < 10:
            continue
        b_cal_logits = cal_logits[bucket_mask]
        b_cal_labels = cal_labels[bucket_mask]

        # Evaluate coverage on the held-out tauval split's rows that share this
        # bucket -- both the calibration bucket_ids above and this eval set now
        # come from FST estimates computed on their OWN split (cal_split_fst /
        # tauval_split_fst respectively), not a cross-split lookup.
        tauval_ids_in_bucket = set(tauval_split_fst[tauval_split_fst['fst_bucket'] == bucket]['image_id'].astype(str))
        if len(tauval_ids_in_bucket) < 5:
            print(f'FST {bucket}: too few eval samples for coverage check -- skipped.')
            continue

        n_covered, n_eval = 0, 0
        for img_id in list(tauval_ids_in_bucket)[:200]: # cap for runtime
            row = tauval_split_df[tauval_split_df['image_id'].astype(str) == img_id]
            if len(row) == 0:
                continue
            row = row.iloc[0]
            img_bgr = cv2.imread(row['image_path'])
            if img_bgr is None:
                continue
            img_bgr = dull_razor_hair_removal(img_bgr)
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            img_norm = cv2.resize(img_rgb, (224,224)).astype(np.float32)/255.0
            img_t = torch.from_numpy(((img_norm-_mean_n)/_std_n).transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                lg = model(img_t).cpu().numpy()
            pred_set, _ = build_conformal_prediction_set(lg, b_cal_labels, b_cal_logits, alpha=CFG['deferral_alpha'])
            true_idx = int(row['label_idx'])
            n_covered += int(true_idx in pred_set)
            n_eval += 1

        if n_eval > 0:
            cov = n_covered / n_eval
            fst_coverage_results.append({'fst_bucket': bucket, 'n_cal': bucket_mask.sum(),
                                          'n_eval': n_eval, 'empirical_coverage': cov})
            print(f'FST {bucket:8s}: n_cal={bucket_mask.sum():4d} n_eval={n_eval:4d} '
                  f'empirical coverage={cov:.3f} (target 0.900)')

    if fst_coverage_results:
        fst_cov_df = pd.DataFrame(fst_coverage_results)
        fst_cov_df.to_csv(CFG['out_dir'] / 'Table_per_fst_conformal_coverage.csv', index=False)
        max_gap = fst_cov_df['empirical_coverage'].max() - fst_cov_df['empirical_coverage'].min()
        print(f'\nMax coverage gap across FST buckets: {max_gap:.3f}')
        print('(A large gap here is itself a fairness finding -- standard non-grouped conformal')
        print('prediction may be UNDER-covering one skin-tone group while over-covering another.)')
    else:
        print('\nNo FST bucket had enough calibration AND evaluation samples to report coverage -- '
              'likely V-VI given its small size in this dataset. Report per-bucket sample sizes '
              'explicitly rather than an omnibus number if this remains sparse.')
else:
    print('ℹ CELL 10b (FST estimation) did not produce enough rows -- skipping per-FST conformal audit.')



## CELL 15c — Conformal Prediction Set Size as a Trust Proxy (NEW — Novelty Add: cheap, GradCAM-free trust signal)


In [ ]:
# ============================================================
# CELL 15c — Conformal Prediction Set Size as a Trust Proxy
# CELL 15 already computes a conformal prediction set per test image
# (build_conformal_prediction_set) as part of CCRS -- this reuses that
# result rather than recomputing conformal sets from scratch. A smaller
# set means the model is more confident; here we test whether set size
# is a CHEAP proxy for TIxAI-style explanation trust (no GradCAM/backward
# pass needed at inference time), by correlating it against the genuine
# (non-imputed) TIxAI scores from CELL 10.
# ============================================================
conf_trust_df = ccrs_df[ccrs_df['image_id'].astype(str).isin(tixai_lookup.keys())].copy()
conf_trust_df['tixai_real'] = conf_trust_df['image_id'].astype(str).map(tixai_lookup)

print(f'\nConformal set size vs TIxAI: n={len(conf_trust_df)} images with genuine (non-imputed) TIxAI scores')
print(f"Set size distribution: {conf_trust_df['conf_set_size'].value_counts().sort_index().to_dict()}")

if len(conf_trust_df) >= 5:
    rho, p_val = spearmanr(conf_trust_df['conf_set_size'], conf_trust_df['tixai_real'])
    print(f'\nSpearman(conformal set size, TIxAI): rho={rho:.4f}, p={p_val:.4e}')
    if p_val < 0.01 and abs(rho) > 0.3:
        print('|rho| > 0.3 and p < 0.01 -> conformal set size is a usable, GradCAM-free proxy for '
              'explanation trust (cheap to compute at inference time, no backward pass needed).')
    else:
        print('Correlation too weak/unreliable -> conformal set size should NOT replace TIxAI as a trust '
              'signal, only complement it (e.g., as one CCRS term, which CELL 15 already does).')
else:
    print('\nToo few images with genuine TIxAI scores to test this correlation.')
    rho, p_val = float('nan'), float('nan')

sizes_present = sorted(conf_trust_df['conf_set_size'].unique())
if sizes_present:
    fig, ax = plt.subplots(figsize=(9, 6))
    box_data = [conf_trust_df[conf_trust_df['conf_set_size'] == s]['tixai_real'].values for s in sizes_present]
    bp = ax.boxplot(box_data, labels=sizes_present, patch_artist=True, showmeans=True)
    for patch in bp['boxes']:
        patch.set_facecolor('#2196F3')
        patch.set_alpha(0.7)
    ax.set_xlabel('Conformal prediction set size')
    ax.set_ylabel('TIxAI score (genuine, non-imputed)')
    ax.set_title(f'Conformal Set Size vs Explanation Trust (rho={rho:.3f})\n'
                 '(smaller set = model more confident)', fontweight='bold')
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(CFG['out_dir'] / 'Figure_conformal_set_size_vs_tixai.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise
else:
    print('\nNo images with both a conformal set size and a genuine TIxAI score -- skipping plot.')

conf_trust_df[['image_id', 'conf_set_size', 'tixai_real']].to_csv(
    CFG['out_dir'] / 'Table_conformal_set_size_vs_tixai.csv', index=False)
print('\nConformal-set-size trust-proxy results saved: Table_conformal_set_size_vs_tixai.csv, '
      'Figure_conformal_set_size_vs_tixai.png')



## CELL 16 — Accuracy vs Trust Dissociation (MODIFIED: resumable ConvNeXt TIxAI pass)


In [ ]:
# ============================================================
# CELL 16 — Accuracy vs Explanation Trust DISSOCIATION (Headline Figure)
# FIX: resumable via run_resumable_loop.
# FIX (fairness-of-comparison guard, audit 2026-07-05): this cell's whole
# point is comparing DenseNet121 against ConvNeXt-Tiny on trust metrics.
# That comparison is only meaningful if both models were actually trained
# to genuine convergence -- if one stopped at a deliberate phase boundary
# (CFG['training_phase']==1) or a time-budget cutoff rather than
# early-stopping on a plateaued val_f1, any accuracy/trust gap between them
# is confounded with training completeness, not a clean architecture
# comparison. Check both models' *_status.json before presenting this as
# a "headline" finding.
# ============================================================
assert (not TIXAI_USES_SYNTHETIC_MASKS) or CFG.get('allow_synthetic_masks_in_results', False), (
    'Refusing to compute this cell on synthetic (non-GT) lesion masks. Attach the ISIC2018 '
    "Task 1 dataset, or set CFG['allow_synthetic_masks_in_results']=True to proceed anyway for a demo run."
)

def _model_convergence_reasons():
    reasons = {}
    for name in ('DenseNet121', 'ConvNeXt-Tiny'):
        status_path = CFG['out_dir'] / f'{name}_status.json'
        if status_path.exists():
            with open(status_path) as f:
                reasons[name] = json.load(f).get('reason', 'unknown')
        else:
            reasons[name] = 'unknown (no status file)'
    return reasons

_convergence_status = _model_convergence_reasons()

def _model_converged(name, full_target=None):
    # A model counts as "converged enough to compare" if it either early-stopped
    # on a val-F1 plateau OR trained to its full intended epoch budget -- and did
    # NOT stop merely because the wall-clock time budget ran out. `full_target`
    # overrides the per-call target_epochs written to status (needed for DenseNet,
    # whose phase-1 status records target_epochs=60 even though the real goal is
    # CFG['epochs']).
    sp = CFG['out_dir'] / f'{name}_status.json'
    if not sp.exists():
        return False
    st = json.load(open(sp))
    if st.get('reason') == 'time_budget':
        return False
    goal = full_target if full_target is not None else st.get('target_epochs', 10**9)
    return st.get('reason') == 'early_stopped' or st.get('last_epoch', 0) >= goal

FAIR_CROSS_ARCH_COMPARISON = (_model_converged('DenseNet121', full_target=CFG['epochs'])
                              and _model_converged('ConvNeXt-Tiny'))
if not FAIR_CROSS_ARCH_COMPARISON:
    print(f'\nWARNING (fairness-of-comparison check): model convergence status = {_convergence_status}. '
          f'At least one model did not reach genuine convergence (reason != "early_stopped") before this '
          f'comparison. A model stopped at a deliberate phase boundary or time-budget cutoff is undertrained '
          f'relative to one that actually plateaued -- any accuracy-vs-trust gap below is CONFOUNDED with '
          f'training completeness, not a clean architecture comparison. Re-run after both models reach '
          f'"early_stopped" before treating this as a validated finding.')

cam_convnext = GradCAMPlusPlus(model=model_convnext, target_layers=model_convnext.get_cam_target_layers())
model_convnext.eval()
N_CONVNEXT_EVAL = min(300, len(test_df))
eval_sample = test_df.sample(N_CONVNEXT_EVAL, random_state=SEED)

def _convnext_tixai_row(row):
    img_bgr = cv2.imread(row['image_path'])
    if img_bgr is None:
        return None
    img_bgr = dull_razor_hair_removal(img_bgr)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_norm = cv2.resize(img_rgb, (224,224)).astype(np.float32)/255.0
    img_t = torch.from_numpy(((img_norm-_mean_n)/_std_n).transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model_convnext(img_t) / convnext_T
        probs = torch.softmax(logits,1).cpu().numpy()[0]
        pred_idx = int(probs.argmax())

    true_idx = int(row['label_idx'])
    if pred_idx != true_idx:
        return None

    try:
        gcam = cam_convnext(input_tensor=img_t, targets=[ClassifierOutputTarget(pred_idx)])[0]
    except Exception:
        return None

    img_id = str(row.get('image_id',''))
    if not USE_SYNTHETIC_MASKS and img_id in masks_dict:
        score = compute_tixai(gcam, masks_dict[img_id]); ms = 'GT'
    else:
        score = compute_tixai(gcam, make_synthetic_mask()); ms = 'synthetic'

    return {'image_id': img_id, 'true_class': CLASSES[true_idx], 'tixai': score, 'mask_source': ms}


print('\nComputing TIxAI for ConvNeXt-Tiny baseline (resumable)...')
tixai_cn_df = run_resumable_loop(
    eval_sample, CFG['out_dir'] / 'tixai_convnext_results.csv', _convnext_tixai_row,
    id_col='image_id', desc='ConvNeXt TIxAI'
)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#0D1117')

ax1 = axes[0]; ax1.set_facecolor('#161B22')
x = np.arange(len(CLASSES)); width = 0.35
dense_medians = [np.median(correct_df[correct_df['true_class']==c]['tixai'].values)
                    if len(correct_df[correct_df['true_class']==c]) > 0 else 0 for c in CLASSES]
convnext_medians = [np.median(tixai_cn_df[tixai_cn_df['true_class']==c]['tixai'].values)
                    if len(tixai_cn_df[tixai_cn_df['true_class']==c]) > 0 else 0 for c in CLASSES]

ax1.bar(x - width/2, dense_medians, width, label='DenseNet121 (Primary)', color='#2196F3', alpha=0.85, edgecolor='white', linewidth=0.5)
ax1.bar(x + width/2, convnext_medians, width, label='ConvNeXt-Tiny (Baseline)', color='#FF5722', alpha=0.85, edgecolor='white', linewidth=0.5)
ax1.set_xticks(x); ax1.set_xticklabels(CLASSES, rotation=30, color='white', fontsize=10)
ax1.set_ylabel('Median TIxAI Score', color='white', fontsize=12)
ax1.set_title('Per-Class Explanation Trustworthiness\nDenseNet121 vs ConvNeXt-Tiny', color='white', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10, facecolor='#161B22', labelcolor='white')
ax1.axhline(0.45, color='yellow', linestyle='--', alpha=0.7, label='Low trust threshold')
ax1.tick_params(colors='white'); ax1.spines[:].set_color('#30363D'); ax1.set_ylim([0, 1])

ax2 = axes[1]; ax2.set_facecolor('#161B22')
shared_imgs = set(correct_df['image_id'].astype(str)).intersection(set(tixai_cn_df['image_id'].astype(str)))
if len(shared_imgs) > 10:
    d_scores = correct_df[correct_df['image_id'].astype(str).isin(shared_imgs)]['tixai'].values
    c_scores = tixai_cn_df[tixai_cn_df['image_id'].astype(str).isin(shared_imgs)]['tixai'].values
    if len(d_scores) == len(c_scores):
        stat, p_val = wilcoxon(d_scores, c_scores)
        print(f"\nWilcoxon paired signed-rank test (DenseNet vs ConvNeXt TIxAI): p={p_val:.4e}")

acc_change = []
for i in range(len(CLASSES)):
    dn_mask = (dense_results['labels'] == i)
    cn_mask = (convnext_results['labels'] == i)
    dn_acc = (dense_results['preds'][dn_mask] == i).float().mean().item() if dn_mask.sum() > 0 else 0
    cn_acc = (convnext_results['preds'][cn_mask] == i).float().mean().item() if cn_mask.sum() > 0 else 0
    acc_change.append(cn_acc - dn_acc)
tixai_change = [cn - dn for cn, dn in zip(convnext_medians, dense_medians)]

for i, cls in enumerate(CLASSES):
    color = CLASS_COLORS[cls]
    ax2.scatter(acc_change[i], tixai_change[i], s=200, c=color, edgecolors='white', linewidths=1.5, zorder=5)
    ax2.annotate(cls, (acc_change[i], tixai_change[i]), textcoords='offset points', xytext=(8,4),
                 fontsize=11, color='white', fontweight='bold')

ax2.axhline(0, color='white', linestyle='--', alpha=0.5)
ax2.axvline(0, color='white', linestyle='--', alpha=0.5)
ax2.fill_between([-1, 0], [-1,-1], [0,0], alpha=0.1, color='#F44336', label='Worse accuracy + Worse trust')
ax2.fill_between([0, 1], [0,0], [1,1], alpha=0.1, color='#4CAF50', label='Better accuracy + Better trust')
ax2.fill_between([0, 1], [-1,-1], [0,0], alpha=0.15, color='#FF9800', label='Better accuracy + WORSE trust (dissociation)')
ax2.set_xlabel('Accuracy Change (ConvNeXt - DenseNet)', color='white', fontsize=11)
ax2.set_ylabel('TIxAI Change (ConvNeXt - DenseNet)', color='white', fontsize=11)
ax2.set_title('Accuracy vs Explanation Trust DISSOCIATION\nClasses in orange zone: ConvNeXt more accurate BUT less trustworthy',
              color='white', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9, facecolor='#161B22', labelcolor='white', loc='upper left')
ax2.tick_params(colors='white'); ax2.spines[:].set_color('#30363D')

_suptitle = ('HEADLINE FINDING: Higher Accuracy ≠ Higher Explanation Trustworthiness' if FAIR_CROSS_ARCH_COMPARISON
             else 'PRELIMINARY (unfair comparison -- see warning above): Accuracy vs Explanation Trustworthiness')
plt.suptitle(_suptitle, fontsize=14, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'Figure7_accuracy_trust_dissociation.png', dpi=300, bbox_inches='tight', facecolor='#0D1117')
plt.show()
plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise
if FAIR_CROSS_ARCH_COMPARISON:
    print('\nFigure 7 (Accuracy vs Trust Dissociation) saved. This is your headline figure!')
else:
    print('\nFigure 7 (Accuracy vs Trust Dissociation) saved -- PRELIMINARY ONLY, see fairness-of-comparison '
          'warning above. Do not present as the headline figure until both models show "early_stopped".')



## CELL 16b — Explanation-Performance Decoupling Score (EPDS) (NEW — Novelty Add: quadrant analysis of accuracy vs explanation trust)


In [ ]:
# ============================================================
# CELL 16b — Explanation-Performance Decoupling Score (EPDS)
# TIxAI (Cell 10) is only computed for CORRECT predictions -- by design,
# since it measures explanation quality conditional on the model getting
# the diagnosis right. EPDS needs a localization score for WRONG
# predictions too, to find the two dangerous quadrants:
#   Q2 correct + untrustworthy explanation -> right for the wrong reason
#   Q4 wrong   + trustworthy explanation   -> confidently, convincingly wrong
# The correct-subset scores are reused from Cell 10's tixai_df, not
# recomputed -- this cell only runs GradCAM on the misclassified subset.
# ============================================================
assert (not TIXAI_USES_SYNTHETIC_MASKS) or CFG.get('allow_synthetic_masks_in_results', False), (
    'Refusing to compute EPDS on synthetic (non-GT) lesion masks. Attach the ISIC2018 '
    "Task 1 dataset, or set CFG['allow_synthetic_masks_in_results']=True to proceed anyway for a demo run."
)

wrong_df = tixai_df[tixai_df['correct'] == False].copy()
print(f'\nComputing localization scores for {len(wrong_df)} MISCLASSIFIED images '
      f'(needed for EPDS Q2/Q4 -- Cell 10 only scored correct predictions)...')

cam_epds = GradCAMPlusPlus(model=model, target_layers=model.get_cam_target_layers())
model.eval()


def _epds_wrong_row(row):
    img_id = str(row['image_id'])
    src_match = test_df[test_df['image_id'].astype(str) == img_id]
    if len(src_match) == 0:
        return None
    src_row = src_match.iloc[0]
    img_bgr = cv2.imread(src_row['image_path'])
    if img_bgr is None:
        return None
    img_bgr = dull_razor_hair_removal(img_bgr)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_norm = cv2.resize(img_rgb, (CFG['img_size'], CFG['img_size'])).astype(np.float32) / 255.0
    img_t = torch.from_numpy(((img_norm - _mean_n) / _std_n).transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

    pred_idx = CLASS_TO_IDX[row['pred_class']]
    try:
        gradcam_map = cam_epds(input_tensor=img_t, targets=[ClassifierOutputTarget(pred_idx)])[0]
    except Exception:
        return None

    if not USE_SYNTHETIC_MASKS and img_id in masks_dict:
        score = compute_tixai(gradcam_map, masks_dict[img_id])
    else:
        score = compute_tixai(gradcam_map, make_synthetic_mask(CFG['img_size']))

    return {'image_id': img_id, 'tixai_all': score}


wrong_scores_df = run_resumable_loop(
    wrong_df, CFG['out_dir'] / 'epds_wrong_scores.csv', _epds_wrong_row,
    id_col='image_id', desc='EPDS (wrong preds)'
)

epds_df = tixai_df.copy()
epds_df['image_id'] = epds_df['image_id'].astype(str)
epds_df['tixai_all'] = epds_df['tixai']
wrong_scores_df = wrong_scores_df.assign(image_id=lambda d: d['image_id'].astype(str)) \
                                  .rename(columns={'tixai_all': 'tixai_all_wrong'})
epds_df = epds_df.merge(wrong_scores_df[['image_id', 'tixai_all_wrong']], on='image_id', how='left')
epds_df['tixai_all'] = epds_df['tixai_all'].fillna(epds_df['tixai_all_wrong'])
epds_df = epds_df.dropna(subset=['tixai_all']).drop(columns=['tixai_all_wrong'])
epds_df['tixai_all'] = epds_df['tixai_all'].astype(float)

TIXAI_TRUST_THRESHOLD = 0.45  # matches the 'Low' reliability_tier cutoff used in Cell 10
epds_df['trustworthy'] = epds_df['tixai_all'] >= TIXAI_TRUST_THRESHOLD


def _quadrant(r):
    if r['correct'] and r['trustworthy']:
        return 'Q1_trustworthy'
    if r['correct'] and not r['trustworthy']:
        return 'Q2_lucky'
    if not r['correct'] and not r['trustworthy']:
        return 'Q3_safe_fail'
    return 'Q4_misleading'


epds_df['quadrant'] = epds_df.apply(_quadrant, axis=1)
quad_counts = epds_df['quadrant'].value_counts().reindex(
    ['Q1_trustworthy', 'Q2_lucky', 'Q3_safe_fail', 'Q4_misleading'], fill_value=0)
EPDS = (quad_counts['Q2_lucky'] + quad_counts['Q4_misleading']) / len(epds_df)

print(f'\nEPDS quadrant breakdown (n={len(epds_df)}, trust threshold={TIXAI_TRUST_THRESHOLD}):')
for q, c in quad_counts.items():
    print(f'  {q}: {c} ({100*c/len(epds_df):.1f}%)')
print(f'\nEPDS (fraction decoupled: Q2 + Q4) = {EPDS:.4f}')
print('EPDS > 0.30 -> accuracy and explanation trust are meaningfully decoupled; '
      'do not treat accuracy as a proxy for explanation trustworthiness for this model.'
      if EPDS > 0.30 else
      'EPDS <= 0.30 -> explanation trust tracks correctness reasonably well for this model.')

q4_df = epds_df[epds_df['quadrant'] == 'Q4_misleading']
if len(q4_df) > 0:
    print(f'\n{len(q4_df)} Q4 (misleading) cases -- wrong diagnosis, high explanation trust. '
          f'Most common true/predicted class pairs:')
    print(q4_df.groupby(['true_class', 'pred_class']).size().sort_values(ascending=False).head(10))

fig, ax = plt.subplots(figsize=(9, 8))
colors = {'Q1_trustworthy': '#4CAF50', 'Q2_lucky': '#FF9800',
          'Q3_safe_fail': '#9E9E9E', 'Q4_misleading': '#F44336'}
rng = np.random.RandomState(SEED)
for q in ['Q1_trustworthy', 'Q2_lucky', 'Q3_safe_fail', 'Q4_misleading']:
    sub = epds_df[epds_df['quadrant'] == q]
    jitter = rng.uniform(-0.08, 0.08, len(sub))
    ax.scatter(sub['correct'].astype(int) + jitter, sub['tixai_all'],
               s=25, alpha=0.5, c=colors[q], label=f'{q} (n={len(sub)})')
ax.axhline(TIXAI_TRUST_THRESHOLD, color='black', linestyle='--', alpha=0.6)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Incorrect', 'Correct'])
ax.set_ylabel('Explanation localization score (TIxAI-style)')
ax.set_title(f'Explanation-Performance Decoupling (EPDS={EPDS:.3f})\n'
             'Q2 (lucky) + Q4 (misleading) = decoupled cases', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'Figure_EPDS_quadrants.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise

epds_df.to_csv(CFG['out_dir'] / 'Table_EPDS_per_image.csv', index=False)
epds_summary_rows = [{'quadrant': q, 'n': int(quad_counts[q]), 'pct': 100 * quad_counts[q] / len(epds_df)}
                      for q in quad_counts.index]
epds_summary_rows.append({'quadrant': 'EPDS_score', 'n': None, 'pct': EPDS})
pd.DataFrame(epds_summary_rows).to_csv(CFG['out_dir'] / 'Table_EPDS_summary.csv', index=False)
print('\nEPDS results saved: Table_EPDS_per_image.csv, Table_EPDS_summary.csv, Figure_EPDS_quadrants.png')



## CELL 17 — GAP-5: Saliency Geometry Analysis (MODIFIED: resumable)


In [ ]:
# ============================================================
# CELL 17 — GAP #5: Saliency Geometry Analysis (MASK-FREE)
# FIX: resumable. NOTE: this gap was NOT independently verified against
# the literature in the original novelty audit -- run a targeted search
# ("saliency geometry compactness eccentricity XAI quality medical
# imaging") before presenting it as a headline contribution.
# ============================================================
def extract_saliency_geometry(gradcam_map, threshold_pct=50):
    H, W = gradcam_map.shape
    center = np.array([H/2, W/2])
    thresh = np.percentile(gradcam_map, threshold_pct)
    binary = (gradcam_map >= thresh).astype(np.uint8)
    labeled = measure.label(binary)
    regions = regionprops(labeled)
    if not regions:
        return {'compactness':0, 'eccentricity':1, 'centroid_dist':1, 'area_ratio':0, 'symmetry':0, 'n_regions':0}
    main_region = max(regions, key=lambda r: r.area)
    area = main_region.area
    perimeter = max(main_region.perimeter, 1e-6)
    compactness = (4 * np.pi * area) / (perimeter ** 2)
    eccentricity = main_region.eccentricity
    centroid = np.array(main_region.centroid)
    cent_dist = np.linalg.norm(centroid - center) / np.sqrt(H**2 + W**2)
    area_ratio = area / (H * W)
    left = binary[:, :W//2]
    right = np.fliplr(binary[:, W//2:])
    if left.shape != right.shape:
        right = right[:, :left.shape[1]]
    symmetry = (left & right).sum() / (left | right).sum() if (left | right).sum() > 0 else 0
    return {'compactness': float(compactness), 'eccentricity': float(eccentricity),
            'centroid_dist': float(cent_dist), 'area_ratio': float(area_ratio),
            'symmetry': float(symmetry), 'n_regions': len(regions)}

cam_dense = GradCAMPlusPlus(model=model, target_layers=model.get_cam_target_layers())
model.eval()
N_GEOM = min(500, len(test_df))
geom_sample = test_df.sample(N_GEOM, random_state=SEED)

def _geom_process_row(row):
    img_bgr = cv2.imread(row['image_path'])
    if img_bgr is None:
        return None
    img_bgr = dull_razor_hair_removal(img_bgr)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_norm = cv2.resize(img_rgb, (224,224)).astype(np.float32)/255.0
    img_t = torch.from_numpy(((img_norm-_mean_n)/_std_n).transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(img_t) / T_dense
        probs = torch.softmax(logits,1).cpu().numpy()[0]
        pred_idx = int(probs.argmax())
        confidence = float(probs.max())
    correct = pred_idx == int(row['label_idx'])
    try:
        gcam = cam_dense(input_tensor=img_t, targets=[ClassifierOutputTarget(pred_idx)])[0]
        geom = extract_saliency_geometry(gcam)
    except Exception:
        return None
    geom.update({'image_id': str(row.get('image_id', row.name)), 'true_class': row['label'],
                 'pred_class': CLASSES[pred_idx], 'correct': correct, 'confidence': confidence})
    return geom

print(f'\nComputing Saliency Geometry for up to {N_GEOM} samples (GAP #5, resumable)...')
geom_df = run_resumable_loop(
    geom_sample, CFG['out_dir'] / 'saliency_geometry.csv', _geom_process_row,
    id_col='image_id', desc='Geometry'
)

correct_geom = geom_df[geom_df['correct'] == True]
incorrect_geom = geom_df[geom_df['correct'] == False]

print('\nSaliency Geometry: Correct vs Incorrect Predictions')
for feat in ['compactness','centroid_dist','area_ratio','symmetry']:
    c_vals = correct_geom[feat].dropna().values
    i_vals = incorrect_geom[feat].dropna().values
    if len(c_vals) > 5 and len(i_vals) > 5:
        t_stat, p = stats.ttest_ind(c_vals, i_vals)
        print(f'{feat:20s}: correct={np.mean(c_vals):.3f}, incorrect={np.mean(i_vals):.3f}, t={t_stat:.2f}, p={p:.4f}')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor('#0D1117')
features_to_plot = [
    ('compactness', 'Compactness (1=circle, <1=irregular)', 'Higher = more focused saliency'),
    ('centroid_dist', 'Centroid Distance to Image Center', 'Lower = more centered saliency'),
    ('area_ratio', 'Saliency Area Ratio', 'Appropriate area = lesion-sized saliency'),
    ('symmetry', 'Saliency Symmetry', 'Higher = more symmetric saliency'),
]
for ax, (feat, xlabel, subtitle) in zip(axes.flatten(), features_to_plot):
    ax.set_facecolor('#161B22')
    for cls, group in geom_df.groupby('true_class'):
        vals = group[feat].dropna().values
        if len(vals) > 3:
            ax.scatter([cls]*len(vals), vals, alpha=0.4, c=CLASS_COLORS.get(cls, 'gray'), s=15)
    medians = geom_df.groupby('true_class')[feat].median().reindex(CLASSES)
    ax.plot(range(len(CLASSES)), medians.values, 'w-o', linewidth=2, markersize=8, zorder=5)
    ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=30, color='white')
    ax.set_ylabel(xlabel, color='white', fontsize=10)
    ax.set_title(f'{xlabel}\n{subtitle}', color='white', fontsize=10, fontweight='bold')
    ax.tick_params(colors='white'); ax.spines[:].set_color('#30363D')

plt.suptitle('GAP #5: Saliency Geometry Analysis (Mask-Free XAI Quality Metric) -- NOVELTY UNVERIFIED, see note above',
             fontsize=13, fontweight='bold', color='white')
plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'Figure8_saliency_geometry.png', dpi=250, bbox_inches='tight', facecolor='#0D1117')
plt.show()
plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise
print('\nFigure 8 (Saliency Geometry) saved.')



## CELL 17b — Cross-Architecture Trust Audit: DenseNet121 vs ConvNeXt-Tiny (NEW — Novelty Add: does higher accuracy buy higher explanation trust?)


In [ ]:
# ============================================================
# CELL 17b — Cross-Architecture Trust Audit (DenseNet121 vs ConvNeXt-Tiny)
# Every trust metric so far (TIxAI, EPDS, STS) was computed on
# DenseNet121, the primary model. This computes a right-sized version of
# the same metrics on ConvNeXt-Tiny, on an independent random sample, so
# accuracy/AUC can be compared against explanation-trust metrics ACROSS
# architectures, not just within one -- the point being that picking a
# model by accuracy alone can silently pick the LESS trustworthy model.
#
# FIX (fairness-of-comparison guard, audit 2026-07-05): reuses CELL 16's
# FAIR_CROSS_ARCH_COMPARISON check -- this cell's "KEY FINDING" language
# is only valid once both models have actually reached genuine convergence
# (early_stopped), not a deliberate phase boundary or time-budget cutoff.
# ============================================================
CROSS_ARCH_N = min(80, len(test_df))
CROSS_ARCH_STS_T = 8
cross_arch_sample = test_df.sample(CROSS_ARCH_N, random_state=SEED)

if 'FAIR_CROSS_ARCH_COMPARISON' not in dir():
    FAIR_CROSS_ARCH_COMPARISON = all(
        r == 'early_stopped' for r in _model_convergence_reasons().values()
    )

model_convnext_sts = copy.deepcopy(model_convnext)
for _m in model_convnext_sts.modules():
    if isinstance(_m, nn.Dropout):
        _m.train()
cam_convnext_x = GradCAMPlusPlus(model=model_convnext, target_layers=model_convnext.get_cam_target_layers())
cam_convnext_sts = GradCAMPlusPlus(model=model_convnext_sts, target_layers=model_convnext_sts.get_cam_target_layers())

_n_dropout_convnext = sum(1 for _m in model_convnext_sts.modules() if isinstance(_m, nn.Dropout))
COMPUTE_CONVNEXT_STS = _n_dropout_convnext > 0
if not COMPUTE_CONVNEXT_STS:
    print('\nWARNING: ConvNeXt-Tiny has no nn.Dropout modules active (timm\'s convnext_tiny uses stochastic '
          'depth, not dropout, and it defaults to disabled here) -- there is no source of stochasticity to '
          'measure STS against for this architecture. Reporting STS_median=NaN for ConvNeXt-Tiny rather than '
          'a misleading perfect-correlation score.')


def _convnext_cross_row(row):
    img_id = str(row.get('image_id', row.name))
    img_bgr = cv2.imread(row['image_path'])
    if img_bgr is None:
        return None
    img_bgr = dull_razor_hair_removal(img_bgr)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_norm = cv2.resize(img_rgb, (CFG['img_size'], CFG['img_size'])).astype(np.float32) / 255.0
    img_t = torch.from_numpy(((img_norm - _mean_n) / _std_n).transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred_idx = int(torch.softmax(model_convnext(img_t), dim=1).argmax(1).item())
    true_idx = int(row['label_idx'])

    try:
        gcam = cam_convnext_x(input_tensor=img_t, targets=[ClassifierOutputTarget(pred_idx)])[0]
    except Exception:
        return None
    if not USE_SYNTHETIC_MASKS and img_id in masks_dict:
        tixai_score = compute_tixai(gcam, masks_dict[img_id])
    else:
        tixai_score = compute_tixai(gcam, make_synthetic_mask(CFG['img_size']))

    sts_score = None
    if COMPUTE_CONVNEXT_STS:
        sts_cams = []
        for _ in range(CROSS_ARCH_STS_T):
            for _m in model_convnext_sts.modules():
                if isinstance(_m, nn.Dropout):
                    _m.train()
            try:
                g = cam_convnext_sts(input_tensor=img_t, targets=[ClassifierOutputTarget(pred_idx)])[0]
            except Exception:
                continue
            sts_cams.append(g.flatten())
        if len(sts_cams) >= 2:
            cm = np.corrcoef(np.stack(sts_cams))
            if np.isfinite(cm).all():
                sts_score = float(np.mean(cm[np.triu_indices(len(sts_cams), k=1)]))

    return {'image_id': img_id, 'true_class': CLASSES[true_idx], 'pred_class': CLASSES[pred_idx],
            'correct': pred_idx == true_idx, 'tixai_all': tixai_score, 'sts': sts_score}


print('\n[Cross-Arch] Computing ConvNeXt-Tiny TIxAI' + (' + STS' if COMPUTE_CONVNEXT_STS else '') +
      ' on an independent sample...')
convnext_cross_df = run_resumable_loop(
    cross_arch_sample, CFG['out_dir'] / 'cross_arch_convnext.csv', _convnext_cross_row,
    id_col='image_id', desc='ConvNeXt cross-arch'
)

convnext_cross_df['trustworthy'] = convnext_cross_df['tixai_all'] >= TIXAI_TRUST_THRESHOLD
convnext_cross_df['quadrant'] = convnext_cross_df.apply(_quadrant, axis=1)
convnext_quad_counts = convnext_cross_df['quadrant'].value_counts()
convnext_EPDS = ((convnext_quad_counts.get('Q2_lucky', 0) + convnext_quad_counts.get('Q4_misleading', 0))
                  / len(convnext_cross_df))
convnext_sts_valid = convnext_cross_df.dropna(subset=['sts'])

cross_arch_table = pd.DataFrame([
    {'Model': 'DenseNet-121 (Primary)',
     'Accuracy': dense_results['accuracy'], 'AUC': dense_results['macro_auc'],
     'TIxAI_median': correct_df['tixai'].median(),
     'XDI_median': xdi_df['xdi'].median(),
     'EPDS': EPDS, 'STS_median': sts_df['sts'].median()},
    {'Model': 'ConvNeXt-Tiny (Baseline)',
     'Accuracy': convnext_results['accuracy'], 'AUC': convnext_results['macro_auc'],
     'TIxAI_median': convnext_cross_df['tixai_all'].median(),
     'XDI_median': float('nan'),  # SHAP-based XDI (CELL 12) was only computed for the primary model
     'EPDS': convnext_EPDS,
     'STS_median': convnext_sts_valid['sts'].median() if len(convnext_sts_valid) else float('nan')},
])
print('\nCross-Architecture Trust Audit (Table for paper):')
print(cross_arch_table.to_string(index=False))

if not FAIR_CROSS_ARCH_COMPARISON:
    print(f'\nWARNING (fairness-of-comparison check): model convergence status = '
          f'{_model_convergence_reasons()}. At least one model did not reach genuine convergence before this '
          f'audit -- treat everything below as PRELIMINARY, not a validated cross-architecture finding.')

acc_gap = cross_arch_table.iloc[1]['Accuracy'] - cross_arch_table.iloc[0]['Accuracy']
epds_gap = cross_arch_table.iloc[1]['EPDS'] - cross_arch_table.iloc[0]['EPDS']
_finding_prefix = 'KEY FINDING' if FAIR_CROSS_ARCH_COMPARISON else 'PRELIMINARY (unfair comparison -- see warning above)'
if acc_gap > 0 and epds_gap > 0:
    print(f'\n{_finding_prefix}: ConvNeXt-Tiny is more accurate but MORE decoupled (higher EPDS) than DenseNet121 -- '
          f'selecting the model by accuracy alone would have picked the less trustworthy explanation pipeline.')
elif acc_gap > 0 and epds_gap <= 0:
    print(f'\n{_finding_prefix}: Accuracy and EPDS both favor ConvNeXt-Tiny here -- no dissociation detected at the '
          f'model-selection level on this sample; re-check with a larger CROSS_ARCH_N before treating this as a '
          f'stable result.')
else:
    print(f'\n{_finding_prefix}: DenseNet121 is at least as accurate as ConvNeXt-Tiny here -- compare the EPDS/STS '
          f'columns directly for the trust trade-off between the two.')

cross_arch_table.to_csv(CFG['out_dir'] / 'Table_cross_architecture_trust_audit.csv', index=False)
convnext_cross_df.to_csv(CFG['out_dir'] / 'Table_cross_arch_convnext_per_image.csv', index=False)
print('\nCross-architecture trust audit saved: Table_cross_architecture_trust_audit.csv, '
      'Table_cross_arch_convnext_per_image.csv')


## CELL 17c — GAP-9: Synthetic Dark-Skin Diffusion Augmentation (NEW — Novelty Add #9)

In [ ]:
# ============================================================
# CELL 17c — GAP-9: Synthetic Dark-Skin Diffusion Augmentation
# (NEW — Novelty Add #9, per GAP9_Synthetic_Dark_Skin_Data_Report.md)
#
# Research question: does augmenting training data with structure-
# preserving synthetic dark-skin dermoscopy images improve EXPLANATION
# reliability/fairness (per-FST TIxAI from CELL 10b), not just raw
# accuracy? HAM10000 is >95% light-skin (Fitzpatrick I-II) with zero
# FST labels -- CELL 10b can only ESTIMATE skin tone via ITA, and any
# per-FST audit on real data is starved of dark-skin samples to audit
# in the first place. This cell generates candidate dark-skin variants
# of existing light-skin lesions using a ControlNet-conditioned
# diffusion model, which locks the lesion's structural edge map so
# only skin tone/texture is allowed to change (GAP9 Section 5:
# CycleGAN/StyleGAN alternatives were rejected specifically because
# they distort lesion boundaries).
#
# HONEST SCOPE NOTE: this cell VALIDATES and STAGES synthetic images;
# it does NOT retrain the model by itself. Actually using them requires
# setting CFG['use_synthetic_dark_skin']=True and re-running CELL 5
# (loads the augmented manifest produced below) and CELL 7 with
# CFG['force_retrain']=True. That is deliberate, not an oversight:
# GAP-9's own research question is a BEFORE/AFTER comparison (Section 9,
# delta_TIxAI / delta_EFG) -- collapsing the two conditions into one
# training run would make that comparison impossible to report.
#
# REQUIRES INTERNET (Kaggle notebook setting) to download Stable
# Diffusion v1.5 + ControlNet-HED weights (~5GB) on first run. If
# internet is off or the download fails, this cell prints a clear
# skip message and leaves the rest of the notebook unaffected -- it
# does not raise and does not silently fabricate synthetic images.
# ============================================================

SYN_DIR = CFG['out_dir'] / 'synthetic_dark_skin'
SYN_DIR.mkdir(parents=True, exist_ok=True)
SYN_MANIFEST_PATH = CFG['out_dir'] / 'train_df_augmented.csv'

CLASS_FULLNAME = {
    'NV': 'melanocytic nevus', 'MEL': 'melanoma', 'BCC': 'basal cell carcinoma',
    'AKIEC': 'actinic keratosis', 'BKL': 'benign keratosis',
    'DF': 'dermatofibroma', 'VASC': 'vascular lesion',
}

_gap9_ready = False
try:
    # FIX (audit 2026-07-05): the previous pip_install() here left `transformers`
    # unpinned, so it resolved to whatever version Kaggle's base image already
    # had. diffusers==0.31.0's controlnet pipeline import chain touches
    # transformers.utils.FLAX_WEIGHTS_NAME, a Flax-support constant that newer
    # transformers releases have removed -- this failed with
    # "cannot import name 'FLAX_WEIGHTS_NAME' from 'transformers.utils'" on a
    # session where internet WAS available (i.e. not the download-failure case
    # the old except message assumed). Pinning transformers alongside diffusers
    # (same joint-resolve principle as CELL 1) fixes the version skew for the
    # combination this cell was tested against. If a future Kaggle base image
    # ships a transformers/diffusers pair where this specific pin still
    # conflicts, bump both together rather than dropping the pin.
    pip_install('diffusers==0.31.0', 'transformers==4.46.3', 'controlnet-aux==0.0.9', 'accelerate>=0.26.0')
    from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
    from controlnet_aux import HEDdetector

    hed = HEDdetector.from_pretrained('lllyasviel/Annotators')
    controlnet = ControlNetModel.from_pretrained(
        'lllyasviel/sd-controlnet-hed', torch_dtype=torch.float16
    )
    sd_pipe = StableDiffusionControlNetPipeline.from_pretrained(
        'runwayml/stable-diffusion-v1-5', controlnet=controlnet,
        torch_dtype=torch.float16, safety_checker=None
    ).to(DEVICE)
    sd_pipe.scheduler = UniPCMultistepScheduler.from_config(sd_pipe.scheduler.config)
    _gap9_ready = True
    print('ControlNet-HED + Stable Diffusion v1.5 loaded -- GAP-9 generation enabled.')
except Exception as e:
    # FIX (audit 2026-07-05): the old message always blamed "internet disabled /
    # download interrupted" -- but the actual failure observed in this project's
    # last run was an ImportError (wrapped in RuntimeError by diffusers' lazy
    # module loader) from a transformers/diffusers version mismatch, which
    # happens with internet fully enabled and has nothing to do with model
    # downloads. Distinguish the two so the printed remediation is accurate.
    _msg = str(e)
    if isinstance(e, (ImportError, RuntimeError)) and ('cannot import name' in _msg or 'Failed to import' in _msg):
        print(f'GAP-9 SKIPPED: could not load the diffusion pipeline ({type(e).__name__}: {e}). '
              f'This looks like a transformers/diffusers version mismatch (an import inside '
              f'diffusers is failing), NOT a network/download problem -- toggling internet '
              f'on/off will not fix it. Try pinning a different transformers version alongside '
              f'diffusers==0.31.0 above (this run used transformers==4.46.3) and re-running this '
              f'cell. The rest of the notebook is unaffected.')
    else:
        print(f'GAP-9 SKIPPED: could not load the diffusion pipeline ({type(e).__name__}: {e}). '
              f'Likely cause: internet is disabled for this Kaggle session, or the model '
              f'download was interrupted. Enable internet (Settings > Internet > On) and '
              f're-run this cell. The rest of the notebook is unaffected.')

if _gap9_ready:
    from skimage.metrics import structural_similarity as ssim_metric

    def lesion_structure_preservation_index(img_real_rgb, img_syn_rgb):
        # LSPI = SSIM(Otsu(real), Otsu(syn)) -- per GAP9 Section 8.1.3.
        def otsu_mask(img_rgb):
            gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
            _, m = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
            return m.astype(np.float32) / 255.0
        m_real, m_syn = otsu_mask(img_real_rgb), otsu_mask(img_syn_rgb)
        return float(ssim_metric(m_real, m_syn, data_range=1.0))

    LSPI_THRESHOLD = 0.90  # GAP9 Section 8.1.3 acceptance threshold
    N_PER_CLASS = 15  # SD+ControlNet inference is ~5-10s/image on a T4/P100 -- keep modest

    candidate_rows = []
    for cls in CLASSES:
        cls_df = train_df[train_df['label'] == cls]
        if len(cls_df) == 0:
            continue
        sample_n = min(N_PER_CLASS, len(cls_df))
        candidate_rows.append(cls_df.sample(sample_n, random_state=SEED))
    gap9_source_df = pd.concat(candidate_rows, ignore_index=True) if candidate_rows else pd.DataFrame()
    print(f'GAP-9: {len(gap9_source_df)} candidate source images selected for dark-skin '
          f'synthesis ({N_PER_CLASS}/class max).')

    def _gap9_process_row(row):
        img_id = str(row.get('image_id', row.name))
        cls = row['label']
        img_bgr = cv2.imread(row['image_path'])
        if img_bgr is None:
            return None
        img_bgr = dull_razor_hair_removal(img_bgr)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rgb = cv2.resize(img_rgb, (512, 512))  # SD1.5 native resolution

        edge_map = hed(Image.fromarray(img_rgb))
        prompt = (f'dermoscopic image of {CLASS_FULLNAME.get(cls, cls.lower())}, '
                  f'dark brown skin, Fitzpatrick skin type V-VI, high detail, clinical photo')
        neg_prompt = 'light skin, pale skin, cartoon, illustration, blurry, low quality'

        result = sd_pipe(
            prompt, image=edge_map, negative_prompt=neg_prompt,
            num_inference_steps=25, controlnet_conditioning_scale=1.0,
            generator=torch.Generator(device=DEVICE).manual_seed(SEED)
        )
        syn_rgb = np.array(result.images[0].resize((CFG['img_size'], CFG['img_size'])))
        img_rgb_224 = cv2.resize(img_rgb, (CFG['img_size'], CFG['img_size']))

        lspi = lesion_structure_preservation_index(img_rgb_224, syn_rgb)

        # Sanity check that conditioning actually shifted skin tone darker (not a
        # formal fairness metric by itself -- re-uses the CELL 10b ITA estimator,
        # since a GT mask generally will not exist for a brand-new synthetic image).
        synth_mask_bin = (make_synthetic_mask(CFG['img_size']) > 127).astype(np.uint8)
        ita_before = estimate_ita(img_rgb_224, synth_mask_bin)
        ita_after = estimate_ita(syn_rgb, synth_mask_bin)

        accepted = (lspi >= LSPI_THRESHOLD and ita_before is not None
                    and ita_after is not None and ita_after < ita_before)
        out_path = None
        if accepted:
            out_path = SYN_DIR / f'{img_id}_dark.jpg'
            cv2.imwrite(str(out_path), cv2.cvtColor(syn_rgb, cv2.COLOR_RGB2BGR))

        return {
            'source_image_id': img_id, 'label': cls, 'label_idx': int(row['label_idx']),
            'lspi': lspi, 'ita_before': ita_before, 'ita_after': ita_after,
            'accepted': accepted, 'image_path': str(out_path) if out_path else None,
        }

    gap9_results = run_resumable_loop(
        gap9_source_df, CFG['out_dir'] / 'gap9_synthetic_validation.csv',
        _gap9_process_row, id_col='source_image_id', desc='GAP-9 dark-skin synthesis'
    )

    if len(gap9_results):
        n_accepted = int(gap9_results['accepted'].sum())
        print(f'\nGAP-9 validation gate: {n_accepted}/{len(gap9_results)} synthetic images '
              f'passed (LSPI >= {LSPI_THRESHOLD} AND ITA shifted darker).')
        print(f'Mean LSPI: {gap9_results["lspi"].mean():.3f} | '
              f'Rejected for LSPI: {int((gap9_results["lspi"] < LSPI_THRESHOLD).sum())}')

        accepted_df = gap9_results[gap9_results['accepted']].copy()
        accepted_df['image_id'] = accepted_df['source_image_id'] + '_dark'
        accepted_df['is_synthetic'] = True
        train_df_plain = train_df.copy()
        train_df_plain['is_synthetic'] = False
        augmented_df = pd.concat([
            train_df_plain,
            accepted_df[['image_id', 'label', 'label_idx', 'image_path', 'is_synthetic']]
        ], ignore_index=True)
        augmented_df.to_csv(SYN_MANIFEST_PATH, index=False)
        print(f'\nAugmented training manifest written: {SYN_MANIFEST_PATH.name} '
              f'({len(train_df_plain)} real + {len(accepted_df)} accepted synthetic = '
              f'{len(augmented_df)} total).')
        print('To actually train on this: set CFG["use_synthetic_dark_skin"]=True, then '
              're-run CELL 5 and CELL 7 with CFG["force_retrain"]=True. Afterward, re-run '
              'CELL 10b to compare per-FST TIxAI before vs. after -- THAT comparison is the '
              'actual GAP-9 finding, not this generation step.')
        print('\nNOTE: FID/KID against a real dark-skin reference distribution (e.g. the '
              'MSKCC Skin Tone Dataset) were NOT computed -- no such dataset is attached in '
              'this session, and reporting a fabricated number would be worse than reporting '
              'nothing. LSPI + ITA-shift above are necessary but not sufficient validation; '
              'attach a real dark-skin dermoscopy dataset before treating these images as '
              'publication-grade evidence.')
    else:
        print('GAP-9: no results produced -- check the failure warnings printed above.')
else:
    print(f'GAP-9: skipped (diffusion pipeline unavailable this session). '
          f'CFG["use_synthetic_dark_skin"] remains {CFG["use_synthetic_dark_skin"]}.')


## CELL 17d — GAP-10: Explanation Contrastive Analysis (NEW — Novelty Add #10)

In [ ]:
# ============================================================
# CELL 17d — GAP-10: Explanation Contrastive Analysis
# (NEW — Novelty Add #10, per GAP10_Explanation_Contrastive_Analysis_Report.md)
#
# Research question: do visually/semantically similar lesions get
# CONSISTENT explanations? If two near-identical Nevi get Grad-CAM++
# maps pointing at completely different regions, that is evidence of
# unstable, shortcut-sensitive reasoning that a per-image TIxAI score
# alone (CELL 10) cannot detect -- TIxAI only asks "is this one
# explanation lesion-focused", not "is the model's explanation for
# this class stable across similar presentations."
#
# SCOPE SIMPLIFICATIONS vs. the design doc (stated explicitly, not
# hidden):
#  1. Visual-similarity retrieval uses DINOv2 (via timm) zero-shot --
#     the design doc's proposed clinical-metadata contrastive
#     fine-tuning step is not done here. Treat retrieved neighbours as
#     "visually similar by a general-purpose vision encoder", not a
#     clinically-validated similarity metric, until a reader study
#     confirms it (the design doc's own Section 12.1 anticipates this
#     exact caveat).
#  2. SIFT-based spatial registration between the two images (design
#     doc Section 6.1 Step 3) is SKIPPED here: that step exists for the
#     longitudinal case (GAP-6 report) where both images are the SAME
#     physical lesion re-photographed, so SIFT keypoints genuinely
#     correspond. Here the two images are DIFFERENT lesions from
#     different patients -- there is no true correspondence for SIFT to
#     recover, so registering them would just be registering noise.
#     Saliency maps are compared directly in the shared 224x224
#     preprocessing frame instead.
#  3. Earth Mover's Distance is approximated by centroid shift (a
#     lighter-weight spatial-displacement proxy) rather than a full 2D
#     transport-distance solve, matching this notebook's existing
#     preference for compute-light proxies (see CELL 15c).
# ============================================================

DINOV2_MODEL_NAME = 'vit_small_patch14_dinov2.lvd142m'
try:
    dinov2_model = timm.create_model(DINOV2_MODEL_NAME, pretrained=True, num_classes=0).to(DEVICE).eval()
    dinov2_data_cfg = timm.data.resolve_model_data_config(dinov2_model)
    dinov2_transform = timm.data.create_transform(**dinov2_data_cfg, is_training=False)
    _gap10_ready = True
    print(f'DINOv2 ({DINOV2_MODEL_NAME}) loaded for GAP-10 embedding retrieval.')
except Exception as e:
    _gap10_ready = False
    print(f'GAP-10 SKIPPED: could not load {DINOV2_MODEL_NAME} ({type(e).__name__}: {e}). '
          f'Likely cause: internet disabled or weights not cached. Enable internet and re-run.')

if _gap10_ready:
    @torch.no_grad()
    def _dinov2_embed(row):
        img_bgr = cv2.imread(row['image_path'])
        if img_bgr is None:
            return None
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        x = dinov2_transform(Image.fromarray(img_rgb)).unsqueeze(0).to(DEVICE)
        emb = dinov2_model(x).squeeze(0).cpu().numpy()
        return emb / (np.linalg.norm(emb) + 1e-8)

    cached = load_ckpt('gap10_dinov2_embeddings')
    if cached is not None:
        gap10_ids, gap10_embs, gap10_meta = cached
    else:
        gap10_ids, gap10_embs, meta_rows = [], [], []
        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='GAP-10 DINOv2 embed'):
            emb = _dinov2_embed(row)
            if emb is None:
                continue
            gap10_ids.append(str(row.get('image_id', row.name)))
            gap10_embs.append(emb)
            meta_rows.append(row)
        gap10_embs = np.stack(gap10_embs) if gap10_embs else np.zeros((0, 384))
        gap10_meta = pd.DataFrame(meta_rows).reset_index(drop=True)
        save_ckpt('gap10_dinov2_embeddings', (gap10_ids, gap10_embs, gap10_meta))

    print(f'GAP-10: {len(gap10_ids)} test-set embeddings ready.')

    @torch.no_grad()
    def _predict_row(row):
        img_bgr = cv2.imread(row['image_path'])
        img_bgr = dull_razor_hair_removal(img_bgr)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rs = cv2.resize(img_rgb, (CFG['img_size'], CFG['img_size']))
        img_norm = img_rs.astype(np.float32) / 255.0
        img_t = torch.from_numpy(((img_norm - _mean_n) / _std_n).transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)
        logits = model(img_t) / T_dense
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        return int(probs.argmax())

    pred_cached = load_ckpt('gap10_predictions')
    if pred_cached is not None:
        gap10_preds = pred_cached
    else:
        gap10_preds = [_predict_row(gap10_meta.iloc[i])
                        for i in tqdm(range(len(gap10_meta)), desc='GAP-10 predictions')]
        save_ckpt('gap10_predictions', gap10_preds)

    gap10_meta = gap10_meta.copy()
    gap10_meta['pred_idx'] = gap10_preds
    gap10_meta['patient_id'] = gap10_meta['lesion_id'] if 'lesion_id' in gap10_meta.columns \
        else gap10_meta['image_id']

    sim_matrix = gap10_embs @ gap10_embs.T  # rows are already L2-normalized -> cosine similarity

    CAT1_2_3_THRESHOLD = 0.85  # GAP10 Section 7.1
    CAT4_THRESHOLD = 0.50

    def _find_pairs():
        pairs = []
        n = len(gap10_ids)
        for i in range(n):
            row_i = gap10_meta.iloc[i]
            sims = sim_matrix[i].copy()
            sims[i] = -1  # exclude self
            same_patient = (gap10_meta['patient_id'] == row_i['patient_id']).values
            sims[same_patient] = -1  # leakage guard: never pair within the same lesion/patient

            j_similar = int(np.argmax(sims))
            if sims[j_similar] >= CAT1_2_3_THRESHOLD:
                row_j = gap10_meta.iloc[j_similar]
                i_correct = row_i['pred_idx'] == row_i['label_idx']
                j_correct = row_j['pred_idx'] == row_j['label_idx']
                same_pred = row_i['pred_idx'] == row_j['pred_idx']
                if i_correct and j_correct and same_pred:
                    category = 'Cat1_ConsistentSuccess'
                elif (not i_correct) and (not j_correct) and same_pred:
                    category = 'Cat2_ConsistentError'
                elif not same_pred:
                    category = 'Cat3_ContradictoryPrediction'
                else:
                    category = None
                if category:
                    pairs.append((i, j_similar, float(sims[j_similar]), category))

            dissimilar_mask = (sims < CAT4_THRESHOLD) & (sims >= 0) & \
                (gap10_meta['pred_idx'].values == row_i['pred_idx'])
            if dissimilar_mask.any():
                j_dissimilar = int(np.where(dissimilar_mask)[0][0])
                pairs.append((i, j_dissimilar, float(sim_matrix[i, j_dissimilar]),
                              'Cat4_RepresentationDiversity'))
        return pairs

    gap10_pairs = _find_pairs()
    print(f'GAP-10: {len(gap10_pairs)} contrastive pairs found across 4 categories:')
    if gap10_pairs:
        print(pd.Series([p[3] for p in gap10_pairs]).value_counts().to_string())

    def _gradcam_for_idx(i):
        row = gap10_meta.iloc[i]
        img_bgr = cv2.imread(row['image_path'])
        img_bgr = dull_razor_hair_removal(img_bgr)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rs = cv2.resize(img_rgb, (CFG['img_size'], CFG['img_size']))
        img_norm = img_rs.astype(np.float32) / 255.0
        img_t = torch.from_numpy(((img_norm - _mean_n) / _std_n).transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)
        targets = [ClassifierOutputTarget(int(row['pred_idx']))]
        return cam_dense(input_tensor=img_t, targets=targets)[0]

    def _saliency_similarity(cam_a, cam_b):
        cos_sim = float((cam_a.flatten() @ cam_b.flatten()) /
                         (np.linalg.norm(cam_a) * np.linalg.norm(cam_b) + 1e-8))
        mask_a, mask_b = (cam_a > 0.5).astype(np.float32), (cam_b > 0.5).astype(np.float32)
        dice = float(2 * (mask_a * mask_b).sum() / (mask_a.sum() + mask_b.sum() + 1e-8))
        ys, xs = np.mgrid[0:cam_a.shape[0], 0:cam_a.shape[1]]

        def centroid(m):
            total = m.sum() + 1e-8
            return (m * xs).sum() / total, (m * ys).sum() / total

        cx_a, cy_a = centroid(cam_a)
        cx_b, cy_b = centroid(cam_b)
        centroid_shift = float(np.hypot(cx_a - cx_b, cy_a - cy_b))
        return cos_sim, dice, centroid_shift

    gap10_results = []
    for i, j, embed_sim, category in tqdm(gap10_pairs, desc='GAP-10 saliency comparison'):
        try:
            cam_i, cam_j = _gradcam_for_idx(i), _gradcam_for_idx(j)
        except Exception:
            continue
        cos_sim, dice, centroid_shift = _saliency_similarity(cam_i, cam_j)
        gap10_results.append({
            'image_id_a': gap10_ids[i], 'image_id_b': gap10_ids[j], 'category': category,
            'embedding_cossim': embed_sim, 'saliency_cossim': cos_sim,
            'saliency_dice': dice, 'centroid_shift_px': centroid_shift,
        })

    gap10_df = pd.DataFrame(gap10_results)
    gap10_df.to_csv(CFG['out_dir'] / 'gap10_contrastive_results.csv', index=False)

    if len(gap10_df):
        cat1 = gap10_df[gap10_df['category'] == 'Cat1_ConsistentSuccess']
        if len(cat1) >= 5:
            rho, p_spear = spearmanr(cat1['embedding_cossim'], cat1['saliency_dice'])
            print(f'\nSpearman(embedding_cossim, saliency_dice) on Cat-1 pairs: '
                  f'rho={rho:.3f}, p={p_spear:.4f} (n={len(cat1)})')

        groups = [g['saliency_dice'].values for _, g in gap10_df.groupby('category') if len(g) >= 3]
        if len(groups) >= 2:
            H, p_kw = kruskal(*groups)
            print(f'Kruskal-Wallis across categories (saliency_dice): H={H:.3f}, p={p_kw:.4f}')

        fig, ax = plt.subplots(figsize=(8, 5))
        gap10_df.boxplot(column='saliency_dice', by='category', ax=ax)
        ax.set_title('GAP-10: Saliency Similarity (Dice) by Contrastive Category')
        ax.set_ylabel('Grad-CAM++ Dice overlap between paired images')
        plt.suptitle('')
        plt.tight_layout()
        plt.savefig(CFG['out_dir'] / 'Figure_GAP10_contrastive_categories.png', dpi=200, bbox_inches='tight')
        plt.show()
        plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise
        print('GAP-10 figure saved.')
    else:
        print('GAP-10: no valid pairs produced usable Grad-CAM++ maps.')
else:
    print('GAP-10: skipped (DINOv2 unavailable this session).')



## CELL 18 — Executive Summary Table (MODIFIED: includes new novelty results)


In [ ]:
# ============================================================
# CELL 18 — Executive Summary of All Findings
# FIX (audit 2026-07-05): 'Correct vs incorrect XDI p-value' and 'ERCC
# Spearman rho' used to read the bare globals `p` and `rho`, which are
# reused by several unrelated later cells (CELL 17's saliency-geometry
# loop overwrites `p`; GAP-10 overwrites `rho`) and so held the wrong
# cell's leftover value by the time this cell ran. Now reads the
# uniquely-named `p_xdi` (set in CELL 12) and `rho_ercc` (set in CELL 11c).
# ============================================================
summary = {
    'Model Performance': {
        'DenseNet121 Balanced Accuracy': f"{dense_results['balanced_acc']:.4f}",
        'DenseNet121 Macro F1': f"{dense_results['macro_f1']:.4f}",
        'DenseNet121 ECE (calibration)': f"{dense_results['ece']:.4f}",
        'ConvNeXt-Tiny Balanced Accuracy': f"{convnext_results['balanced_acc']:.4f}",
    },
    'GAP-1: Per-Class TIxAI': {
        'Kruskal-Wallis p-value': f"{p_kruskal:.2e}",
        'Uses real ISIC2018 masks': f"{not TIXAI_USES_SYNTHETIC_MASKS}",
    },
    'NEW: Per-FST Explanation Fairness': {
        'Kruskal-Wallis p (FST buckets)': f"{p_fst:.2e}" if 'p_fst' in dir() else 'n/a',
    },
    'GAP-2: Failure Prediction': {'Meta-model AUC-ROC': f"{auc_meta:.4f}" if not np.isnan(auc_meta) else 'n/a'},
    'GAP-4: XDI': {
        'Correct vs incorrect XDI p-value': (f"{p_xdi:.4f}" if ('p_xdi' in dir() and p_xdi == p_xdi)
                                             else 'n/a (XDI test had too few samples)'),
        'XDI verdict': 'NULL result -- excluded from CCRS deferral, kept as diagnostic tool only',
        'XDI beats uncertainty baselines (AUC-selective)': f"{'YES'if auc_xdi > max(auc_ent, auc_conf) else 'NO'}" if 'auc_xdi' in dir() else 'n/a',
        'XDI robust across k=25/50/75%': (f"{all(r['significant_p<0.01'] for r in threshold_robustness)}"
                                           if 'threshold_robustness' in dir() and len(threshold_robustness) == 3 else 'n/a'),
    },
    'NEW: ERCC (confidence-TIxAI calibration)': {
        'Spearman rho': f"{rho_ercc:.3f}" if 'rho_ercc' in dir() else 'n/a',
    },
    'NEW: Uncertainty-Gated Suppression Policy': {
        'Suppression rate (test set)': (f"{100 * decision_counts.get('SUPPRESS', 0) / len(suppress_df):.1f}%"
                                         if 'suppress_df' in dir() and len(suppress_df) else 'n/a'),
    },
    'NEW: Per-FST MC-Dropout Entropy': {
        'Kruskal-Wallis p (entropy across FST)': f"{p_ent:.4f}" if 'p_ent' in dir() else 'n/a',
    },
    'GAP-8: CCRS + Deferral': {
        'Deferral rate': f"{100*n_deferred/len(ccrs_df):.1f}%",
        'AI accuracy (non-deferred)': f"{ai_accuracy:.4f}",
    },
    'External Validation (PAD-UFES-20)': {
        'Balanced accuracy on PAD': f"{pad_results['balanced_acc']:.4f}" if pad_results else 'not run (dataset not attached)',
    },
    'NEW: GAP-9 Synthetic Dark-Skin Augmentation': {
        'Synthetic images accepted': (f"{int(gap9_results['accepted'].sum())}/{len(gap9_results)}"
                                       if 'gap9_results' in dir() and len(gap9_results) else 'not run'),
        'Trained with synthetic data': f"{CFG.get('use_synthetic_dark_skin', False)}",
    },
    'NEW: GAP-10 Explanation Contrastive Analysis': {
        'Contrastive pairs analyzed': f"{len(gap10_df)}" if 'gap10_df' in dir() and len(gap10_df) else 'not run',
        'Kruskal-Wallis p (saliency across categories)': f"{p_kw:.4f}" if 'p_kw' in dir() else 'n/a',
    },
}

print('='*70)
print('TRUSTXAI-DERM: EXECUTIVE SUMMARY')
print('='*70)
for section, items in summary.items():
    print(f'\n> {section}')
    for k, v in items.items():
        print(f'{k}: {v}')

with open(CFG['out_dir'] / 'executive_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSummary saved to executive_summary.json')



## CELL 19 — Qualitative Grad-CAM Gallery (unchanged)


In [ ]:
# ============================================================
# CELL 19 — Qualitative Grad-CAM++ Overlay Gallery
# ============================================================
cam_dense = GradCAMPlusPlus(model=model, target_layers=model.get_cam_target_layers())
model.eval()

n_examples = 3
fig, axes = plt.subplots(len(CLASSES), n_examples, figsize=(4*n_examples, 4*len(CLASSES)))
fig.patch.set_facecolor('#0D1117')

gallery_rows = []
for ri, cls in enumerate(CLASSES):
    cls_df = test_df[test_df['label'] == cls]
    samples = cls_df.sample(min(n_examples, len(cls_df)), random_state=SEED)
    for ci, (_, row) in enumerate(samples.iterrows()):
        ax = axes[ri, ci] if len(CLASSES) > 1 else axes[ci]
        ax.set_facecolor('#161B22')
        img_bgr = cv2.imread(row['image_path'])
        img_bgr = dull_razor_hair_removal(img_bgr)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rs = cv2.resize(img_rgb, (224, 224))
        img_norm = img_rs.astype(np.float32) / 255.0
        img_t = torch.from_numpy(((img_norm-_mean_n)/_std_n).transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = model(img_t) / T_dense
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            pred_idx = int(probs.argmax()); confidence = float(probs.max())
        gcam = cam_dense(input_tensor=img_t, targets=[ClassifierOutputTarget(pred_idx)])[0]
        overlay = show_cam_on_image(img_rs.astype(np.float32)/255.0, gcam, use_rgb=True)
        ax.imshow(overlay)
        correct_mark = 'OK' if pred_idx == int(row['label_idx']) else 'X'
        color = '#4CAF50' if correct_mark == 'OK' else '#F44336'
        ax.set_title(f'{correct_mark} True:{cls} Pred:{CLASSES[pred_idx]}\nConf:{confidence:.2f}',
                    color=color, fontsize=10, fontweight='bold')
        ax.axis('off')
        gallery_rows.append({'true_class': cls, 'pred_class': CLASSES[pred_idx],
                             'confidence': confidence, 'image_path': f'reader_study/{row.get("image_id","")}_overlay.png'})
        overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
        (CFG['out_dir']/'reader_study').mkdir(exist_ok=True)
        cv2.imwrite(str(CFG['out_dir']/'reader_study'/f'{row.get("image_id","img"+str(ri)+str(ci))}_overlay.png'), overlay_bgr)

plt.suptitle('Grad-CAM++ Overlay Gallery: Qualitative Explanation Quality Across Classes',
            fontsize=14, fontweight='bold', color='white', y=1.005)
plt.tight_layout()
plt.savefig(CFG['out_dir'] / 'Figure9_gradcam_gallery.png', dpi=250, bbox_inches='tight', facecolor='#0D1117')
plt.show()
plt.close('all')  # NEW: free figure memory -- matplotlib keeps every shown figure resident for the rest of the session otherwise
print('\nFigure 9 (Grad-CAM Gallery) saved. Overlay images also cached to reader_study/ for CELL 19b.')



## CELL 19b — Reader-Study Export (NEW — Novelty Add #5)


In [ ]:
# ============================================================
# CELL 19b — Export dermatologist reader-study rating sheet
#
# NEW: Your audit says even a small reader study "materially raises
# acceptance odds."This exports a rating sheet + a broader set of
# overlay images (beyond the 21 in CELL 19's gallery) for 2+ raters to
# score offline. Once scores are back, load them into `dermatologist_scores`
# below and this cell will compute the TIxAI/XDI correlation for Table 7.
#
# MODIFIED: also surfaces STS (CELL 10c), the EPDS quadrant label
# (CELL 16b) and the conformal set size (CELL 15) per image, plus a
# plain-language risk caption, so raters see the full trust picture next
# to each overlay instead of just the TIxAI number.
# ============================================================
N_READER = 40
sample_for_reading = test_df.sample(min(N_READER, len(test_df)), random_state=SEED)

reader_dir = CFG['out_dir'] / 'reader_study'
reader_dir.mkdir(exist_ok=True)

sts_lookup = dict(zip(sts_df['image_id'].astype(str), sts_df['sts']))
epds_quadrant_lookup = dict(zip(epds_df['image_id'].astype(str), epds_df['quadrant']))
conf_set_lookup = dict(zip(ccrs_df['image_id'].astype(str), ccrs_df['conf_set_size']))


def _risk_caption(tixai_score, sts_score, conf_set_size):
    parts = []
    if conf_set_size is not None and not (isinstance(conf_set_size, float) and np.isnan(conf_set_size)):
        parts.append('confident (small conformal set)' if conf_set_size <= 1 else
                      f'uncertain (conformal set size={int(conf_set_size)})')
    if sts_score is not None and not (isinstance(sts_score, float) and np.isnan(sts_score)):
        parts.append('stable explanation' if sts_score >= 0.5 else f'unstable explanation (STS={sts_score:.2f})')
    if tixai_score is not None and not (isinstance(tixai_score, float) and np.isnan(tixai_score)):
        parts.append('well-localized' if tixai_score >= 0.45 else f'poorly localized (TIxAI={tixai_score:.2f})')
    if not parts:
        return 'Insufficient trust-signal coverage for this image.'
    flags = sum(1 for p in parts if 'uncertain' in p or 'unstable' in p or 'poorly' in p)
    prefix = 'HIGH RISK: ' if flags >= 2 else ('CAUTION: ' if flags == 1 else 'LOW RISK: ')
    return prefix + 'Model is ' + ', '.join(parts) + '.'


reader_rows = []
for _, row in tqdm(sample_for_reading.iterrows(), total=len(sample_for_reading), desc='Reader-study overlays'):
    img_id = str(row.get('image_id', ''))
    out_path = reader_dir / f'{img_id}_overlay.png'
    if not out_path.exists():
        img_bgr = cv2.imread(row['image_path'])
        if img_bgr is None:
            continue
        img_bgr = dull_razor_hair_removal(img_bgr)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_rs = cv2.resize(img_rgb, (224, 224))
        img_norm = img_rs.astype(np.float32) / 255.0
        img_t = torch.from_numpy(((img_norm-_mean_n)/_std_n).transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = model(img_t) / T_dense
            pred_idx = int(torch.softmax(logits, dim=1).argmax(1).item())
        gcam = cam_dense(input_tensor=img_t, targets=[ClassifierOutputTarget(pred_idx)])[0]
        overlay = show_cam_on_image(img_rs.astype(np.float32)/255.0, gcam, use_rgb=True)
        cv2.imwrite(str(out_path), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    tixai_val = tixai_lookup.get(img_id, None)
    sts_val = sts_lookup.get(img_id, None)
    conf_set_val = conf_set_lookup.get(img_id, None)

    reader_rows.append({
        'image_id': img_id, 'true_label': row['label'],
        'overlay_path': str(out_path.name),
        'tixai_score': tixai_val,
        'xdi_score': xdi_lookup.get(img_id, None),
        'sts_score': sts_val,
        'epds_quadrant': epds_quadrant_lookup.get(img_id, None),
        'conformal_set_size': conf_set_val,
        'risk_caption': _risk_caption(tixai_val, sts_val, conf_set_val),
        'dermatologist_score': np.nan, # 1-5 clinical plausibility, filled in offline by raters
        'rater_id': np.nan,
    })

reader_sheet = pd.DataFrame(reader_rows)
reader_sheet.to_csv(CFG['out_dir'] / 'reader_study_sheet.csv', index=False)
print(f'\nReader-study sheet exported: {len(reader_sheet)} images -> reader_study_sheet.csv')
print(f'Overlay images saved to: {reader_dir}')
n_high_risk = reader_sheet['risk_caption'].str.startswith('HIGH RISK').sum()
print(f'{n_high_risk}/{len(reader_sheet)} images flagged HIGH RISK (uncertain/unstable/poorly-localized on 2+ signals).')
print('\nNEXT STEPS (offline, outside this notebook):')
print('1. Download reader_study/ folder + reader_study_sheet.csv.')
print('2. Have 2+ dermatologists independently rate each overlay 1-5 for clinical plausibility.')
print('   (risk_caption gives each rater a plain-language summary of the trust signals for that image.)')
print('3. Re-upload the filled sheet as a Kaggle dataset, load it below, and run the correlation:')
print('filled = pd.read_csv(".../reader_study_sheet_FILLED.csv")')
print('pearsonr(filled["tixai_score"].dropna(), filled["dermatologist_score"].dropna())')
print('-> This is your Table 7 (TIxAI vs clinical plausibility validation).')



## CELL 20 — Persist Checkpoints as a Kaggle Dataset (NEW — enables true multi-session resume)


In [ ]:
# ============================================================
# CELL 20 — Package checkpoints + outputs for the NEXT session
#
# Run this at the END of every session (even a killed/stopped one will
# have partial checkpoints on disk that are worth saving). This zips
# /kaggle/working/checkpoints and the *_results.csv / *_best.pth files
# into one archive. Manually create a new Kaggle Dataset from this
# output, then set CFG['ckpt_input_dir'] in CELL 2 of your NEXT commit
# to point at it -- that's what makes a true multi-session run possible,
# since /kaggle/working itself does not persist across separate commits.
# ============================================================
import shutil

# FIX (audit 2026-07-05): this manifest previously omitted
# '{name}_status.json', even though CELL 2b's own restore logic and the
# printed instructions below both assume it's included. Without it, a
# Phase-2 session that attaches this export finds best.pth + history.csv
# but no status.json, falls into train_model()'s "pre-fix checkpoint"
# branch, and skips training for that entire commit instead of continuing.
PERSIST_MANIFEST = [
    'checkpoints', 'train_split.csv', 'val_split.csv', 'test_split.csv',
    'DenseNet121_best.pth', 'DenseNet121_history.csv', 'DenseNet121_status.json',
    'ConvNeXt-Tiny_best.pth', 'ConvNeXt-Tiny_history.csv', 'ConvNeXt-Tiny_status.json',
    'tixai_results_dense.csv', 'fst_tixai_results.csv', 'xdi_results.csv',
    'csm_index.csv', 'meta_features_raw.csv', 'ccrs_results.csv',
    'cell15_tauval.csv', 'cell15b_cal_fst.csv', 'cell15b_tauval_fst.csv',
    'tixai_convnext_results.csv', 'saliency_geometry.csv',
    'executive_summary.json',
]

persist_dir = CFG['out_dir'] / 'session_checkpoint_export'
persist_dir.mkdir(exist_ok=True)
n_persisted = 0
for item in PERSIST_MANIFEST:
    src = CFG['out_dir'] / item
    if src.exists():
        dst = persist_dir / item
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy(src, dst)
        n_persisted += 1

print(f'Persisted {n_persisted}/{len(PERSIST_MANIFEST)} checkpoint items to: {persist_dir}')
print('\nTO RESUME IN YOUR NEXT SESSION (e.g. Phase 2 of the 2-phase DenseNet121 run):')
print('1. Commit this notebook now (Save & Run All -> Save Version).')
print('2. Go to the committed version''s Output tab, click "New Dataset" on session_checkpoint_export/.')
print('3. In your NEXT session, "Add Data" -> attach that new dataset.')
print('4. In CELL 2, set: CFG["ckpt_input_dir"] = Path("/kaggle/input/<your-dataset-name>")')
print('   -- point this at the DATASET ROOT, not a nested checkpoints/ folder. CELL 2b now')
print('   restores BOTH the top-level files (DenseNet121_best.pth, DenseNet121_history.csv,')
print('   DenseNet121_status.json, train/val/test_split.csv, result CSVs) AND the nested')
print('   checkpoints/ subfolder from there -- pointing at checkpoints/ directly would miss')
print('   the model weights and silently retrain from scratch.')
print('5. Also set CFG["training_phase"] = 2 (for the 2-phase DenseNet121 schedule) if that')
print('   is what you are doing.')
print('6. Re-run the whole notebook top to bottom -- every resumable cell will skip')
print('already-completed work and continue exactly where the last session stopped.')


## CELL 21 - External-Dataset Validation (NEW)

Checks whether the trained DenseNet121 generalizes to a **different** dermoscopy dataset (PAD-UFES-20, ISIC-2019, BCN20000, or your own images) using the exact training-time preprocessing. Edit the 4 config values at the top of the code cell. If the dataset is not attached, the cell prints a skip message and does not break the run.

In [ ]:
# ============================================================
# CELL 21 — EXTERNAL-DATASET VALIDATION
#
# Purpose: check whether the trained DenseNet121 generalizes to a DIFFERENT
# dermoscopy dataset (e.g. PAD-UFES-20, ISIC-2019, BCN20000, or your own
# images) -- pure classification metrics, no masks needed.
#
# It reuses the EXACT training-time preprocessing (DullRazor hair removal +
# resize 224 + ImageNet normalization + temperature scaling T_dense) so the
# comparison is fair. Any external label that is not one of the 7 HAM10000
# classes is skipped and counted, so a partial-overlap dataset still works.
# If the dataset is not attached, this cell prints a skip message and does
# NOT break the notebook run.
# ============================================================
import os, cv2, torch, numpy as np, pandas as pd
import torch.nn.functional as F
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix, roc_auc_score)

# ---------- 1. EDIT THESE FOUR THINGS FOR YOUR DATASET ----------
EXTERNAL_IMAGES_DIR = '/kaggle/input/pad-ufes-20/images'        # folder of image files
EXTERNAL_LABELS_CSV = '/kaggle/input/pad-ufes-20/metadata.csv'  # CSV with an id col + a label col
EXT_ID_COL    = 'img_id'      # column holding the image filename (with or without extension)
EXT_LABEL_COL = 'diagnostic'  # column holding the ground-truth label

# Map the external label strings -> one of the 7 HAM10000 classes.
# Anything not in this map (or mapped to None) is skipped. Edit for YOUR dataset.
LABEL_MAP = {
    'MEL': 'MEL', 'melanoma': 'MEL',
    'NEV': 'NV',  'nevus': 'NV', 'NV': 'NV',
    'BCC': 'BCC', 'basal cell carcinoma': 'BCC',
    'ACK': 'AKIEC', 'AKIEC': 'AKIEC', 'actinic keratosis': 'AKIEC', 'SCC': 'AKIEC',
    'SEK': 'BKL', 'BKL': 'BKL', 'seborrheic keratosis': 'BKL',
    'DF': 'DF', 'dermatofibroma': 'DF',
    'VASC': 'VASC', 'vascular': 'VASC',
}
# ----------------------------------------------------------------

# ---------- 2. reuse notebook objects, with safe fallbacks ----------
_CLASSES = CLASSES if 'CLASSES' in dir() else ['AKIEC','BCC','BKL','DF','MEL','NV','VASC']
_cls_to_idx = {c: i for i, c in enumerate(_CLASSES)}
_dev = DEVICE if 'DEVICE' in dir() else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_T = float(T_dense) if ('T_dense' in dir() and T_dense is not None) else 1.0
_hair = dull_razor_hair_removal if 'dull_razor_hair_removal' in dir() else (lambda x: x)
_mn = _mean_n if '_mean_n' in dir() else np.array([0.485, 0.456, 0.406], dtype=np.float32)
_sd = _std_n if '_std_n' in dir() else np.array([0.229, 0.224, 0.225], dtype=np.float32)

if 'model' not in dir():
    raise RuntimeError("`model` is not in scope -- run the notebook's model + CELL 9 first, "
                       "or load DenseNet121_best.pth into a `model` variable before this cell.")
model.eval()

# ---------- 3. guard: bail out cleanly if data not attached ----------
if not (os.path.isdir(EXTERNAL_IMAGES_DIR) and os.path.isfile(EXTERNAL_LABELS_CSV)):
    print('EXTERNAL VALIDATION SKIPPED -- dataset not found.')
    print(f'  images dir exists: {os.path.isdir(EXTERNAL_IMAGES_DIR)} ({EXTERNAL_IMAGES_DIR})')
    print(f'  labels csv exists: {os.path.isfile(EXTERNAL_LABELS_CSV)} ({EXTERNAL_LABELS_CSV})')
    print('  -> attach the dataset and edit the 4 config values at the top of this cell.')
else:
    meta = pd.read_csv(EXTERNAL_LABELS_CSV)
    assert EXT_ID_COL in meta.columns and EXT_LABEL_COL in meta.columns, \
        f'CSV must contain columns {EXT_ID_COL!r} and {EXT_LABEL_COL!r}; found {list(meta.columns)}'

    _files = {}
    for fn in os.listdir(EXTERNAL_IMAGES_DIR):
        _files[fn] = fn
        _files[os.path.splitext(fn)[0]] = fn

    def _preprocess(path):
        bgr = cv2.imread(path)
        if bgr is None:
            return None
        bgr = _hair(bgr)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        norm = cv2.resize(rgb, (224, 224)).astype(np.float32) / 255.0
        t = torch.from_numpy(((norm - _mn) / _sd).transpose(2, 0, 1)).float().unsqueeze(0)
        return t.to(_dev)

    y_true, y_pred, prob_rows = [], [], []
    n_skip_label, n_skip_missing, n_skip_read = 0, 0, 0

    with torch.no_grad():
        for _, row in meta.iterrows():
            raw_label = str(row[EXT_LABEL_COL]).strip()
            mapped = LABEL_MAP.get(raw_label, LABEL_MAP.get(raw_label.upper(), LABEL_MAP.get(raw_label.lower())))
            if mapped is None or mapped not in _cls_to_idx:
                n_skip_label += 1
                continue
            fid = str(row[EXT_ID_COL]).strip()
            fn = _files.get(fid) or _files.get(os.path.splitext(fid)[0])
            if fn is None:
                n_skip_missing += 1
                continue
            t = _preprocess(os.path.join(EXTERNAL_IMAGES_DIR, fn))
            if t is None:
                n_skip_read += 1
                continue
            probs = F.softmax(model(t) / _T, dim=1).cpu().numpy()[0]
            y_true.append(_cls_to_idx[mapped])
            y_pred.append(int(probs.argmax()))
            prob_rows.append(probs)

    if len(y_true) < 5:
        print(f'EXTERNAL VALIDATION: only {len(y_true)} usable images '
              f'(skipped {n_skip_label} unmapped-label, {n_skip_missing} missing-file, '
              f'{n_skip_read} unreadable) -- too few to report. Check LABEL_MAP and paths.')
    else:
        y_true = np.array(y_true); y_pred = np.array(y_pred); P = np.array(prob_rows)
        present = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
        present_names = [_CLASSES[i] for i in present]

        acc  = accuracy_score(y_true, y_pred)
        bacc = balanced_accuracy_score(y_true, y_pred)
        mf1  = f1_score(y_true, y_pred, average='macro', labels=present, zero_division=0)

        auc = float('nan')
        try:
            cols = [c for c in present if 0 < (y_true == c).sum() < len(y_true)]
            if len(cols) >= 2:
                yb = np.stack([(y_true == c).astype(int) for c in cols], axis=1)
                auc = roc_auc_score(yb, P[:, cols], average='macro', multi_class='ovr')
        except Exception as e:
            print(f'(AUC not computed: {e})')

        print('=' * 64)
        print('EXTERNAL-DATASET VALIDATION  (DenseNet121, temperature-scaled)')
        print('=' * 64)
        print(f'Source: {EXTERNAL_IMAGES_DIR}')
        print(f'Usable images: {len(y_true)}   '
              f'(skipped: {n_skip_label} unmapped label, {n_skip_missing} missing file, {n_skip_read} unreadable)')
        print(f'Classes present in this external set: {present_names}')
        print(f'  Accuracy          : {acc:.4f}')
        print(f'  Balanced accuracy : {bacc:.4f}')
        print(f'  Macro F1          : {mf1:.4f}')
        print(f'  Macro AUC (OvR)   : {auc:.4f}' if auc == auc else '  Macro AUC (OvR)   : n/a')
        print('\nPer-class report:')
        print(classification_report(y_true, y_pred, labels=present,
                                    target_names=present_names, zero_division=0))
        print('Confusion matrix (rows=true, cols=pred):')
        cm = confusion_matrix(y_true, y_pred, labels=present)
        print(pd.DataFrame(cm, index=present_names, columns=present_names).to_string())

        out = pd.DataFrame(P, columns=_CLASSES)
        out.insert(0, 'true_class', [_CLASSES[i] for i in y_true])
        out.insert(1, 'pred_class', [_CLASSES[i] for i in y_pred])
        out.insert(2, 'correct', y_true == y_pred)
        _out_dir = CFG['out_dir'] if 'CFG' in dir() else '.'
        out.to_csv(f'{_out_dir}/external_validation_predictions.csv', index=False)
        print('\nSaved per-image predictions: external_validation_predictions.csv')
        print('\nHONEST READING: external accuracy will usually be LOWER than the HAM10000 test '
              'accuracy -- different scanners, skin-tone distribution and label taxonomy cause '
              'domain shift. A large drop is a generalization finding to report, not a bug. Compare '
              'against your in-distribution HAM10000 test accuracy, not against 100%.')
